## 1. Setup and GNN Runtime Audit

Cette section vérifie l'environnement d'exécution, les ressources CPU/GPU, la disponibilité des principales APIs PyTorch Geometric et le contrat minimal des artefacts produits par le notebook de construction du graphe.

Aucun modèle GNN, graphe PyG global, voisinage, tenseur GPU ou pipeline d'entraînement n'est construit à ce stade. L'objectif est uniquement de confirmer que l'environnement et les artefacts sont prêts pour la mise en place d'un baseline GNN causal et inductif.

In [1]:
import os
import sys
import json
import random
import platform
import importlib
from importlib.util import find_spec
from importlib.metadata import version, PackageNotFoundError
from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
import torch
import torch_geometric

RANDOM_STATE = 42

random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)
torch.manual_seed(RANDOM_STATE)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RANDOM_STATE)

print("Python :", sys.version.split()[0])
print("Plateforme :", platform.platform())
print("NumPy :", np.__version__)
print("Pandas :", pd.__version__)
print("PyArrow :", pa.__version__)
print("PyTorch :", torch.__version__)
print("PyTorch Geometric :", torch_geometric.__version__)

print()
print("CPU logique :", os.cpu_count())

try:
    import psutil

    cpu_physical = psutil.cpu_count(logical=False)
    cpu_logical = psutil.cpu_count(logical=True)
    memory = psutil.virtual_memory()

    print("CPU physique :", cpu_physical)
    print("CPU logique psutil :", cpu_logical)
    print(f"RAM totale : {memory.total / 1024**3:.2f} GB")
    print(f"RAM disponible : {memory.available / 1024**3:.2f} GB")
except ImportError:
    print("psutil : non disponible")

print()
cuda_available = torch.cuda.is_available()

print("CUDA disponible :", cuda_available)
print("CUDA PyTorch :", torch.version.cuda)

if cuda_available:
    device = torch.device("cuda")

    print("GPU :", torch.cuda.get_device_name(0))

    try:
        free_vram, total_vram = torch.cuda.mem_get_info(0)

        print(f"VRAM totale : {total_vram / 1024**3:.2f} GB")
        print(f"VRAM disponible : {free_vram / 1024**3:.2f} GB")
    except Exception as exc:
        properties = torch.cuda.get_device_properties(0)

        print(f"VRAM totale : {properties.total_memory / 1024**3:.2f} GB")
        print("VRAM disponible : non accessible")
        print("Détail VRAM :", str(exc))
else:
    device = torch.device("cpu")

print("Device recommandé :", device)

c:\Users\bouza\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Python : 3.13.7
Plateforme : Windows-11-10.0.22631-SP0
NumPy : 2.2.6
Pandas : 3.0.3
PyArrow : 24.0.0
PyTorch : 2.12.0+cpu
PyTorch Geometric : 2.7.0

CPU logique : 4
CPU physique : 2
CPU logique psutil : 4
RAM totale : 15.84 GB
RAM disponible : 2.82 GB

CUDA disponible : False
CUDA PyTorch : None
Device recommandé : cpu


In [2]:
pyg_status = {}

try:
    from torch_geometric.loader import NeighborLoader

    pyg_status["NeighborLoader"] = True
except Exception as exc:
    pyg_status["NeighborLoader"] = False
    print("Erreur NeighborLoader :", repr(exc))

try:
    from torch_geometric.loader import LinkNeighborLoader

    pyg_status["LinkNeighborLoader"] = True
except Exception as exc:
    pyg_status["LinkNeighborLoader"] = False
    print("Erreur LinkNeighborLoader :", repr(exc))


def check_optional_package(module_name, distribution_name=None):
    distribution_name = distribution_name or module_name

    result = {
        "installed": find_spec(module_name) is not None,
        "importable": False,
        "version": None,
        "error": None,
    }

    if not result["installed"]:
        return result

    try:
        importlib.import_module(module_name)
        result["importable"] = True
    except Exception as exc:
        result["error"] = repr(exc)

    try:
        result["version"] = version(distribution_name)
    except PackageNotFoundError:
        pass

    return result


optional_packages = {
    "torch_scatter": check_optional_package(
        "torch_scatter",
        "torch-scatter",
    ),
    "torch_sparse": check_optional_package(
        "torch_sparse",
        "torch-sparse",
    ),
    "pyg_lib": check_optional_package(
        "pyg_lib",
        "pyg-lib",
    ),
}

print("torch_geometric importable :", True)
print("NeighborLoader disponible :", pyg_status["NeighborLoader"])
print("LinkNeighborLoader disponible :", pyg_status["LinkNeighborLoader"])

print()

for package_name, status in optional_packages.items():
    print(
        f"{package_name} : "
        f"installed={status['installed']}, "
        f"importable={status['importable']}, "
        f"version={status['version']}"
    )

    if status["error"] is not None:
        print("  erreur :", status["error"])

print()

if pyg_status["LinkNeighborLoader"]:
    print(
        "API candidate edge classification : "
        "LinkNeighborLoader, sous contrôle strict de G_<t."
    )
elif pyg_status["NeighborLoader"]:
    print(
        "API candidate : NeighborLoader avec gestion explicite "
        "des query edges."
    )
else:
    print(
        "Aucun NeighborLoader PyG disponible directement. "
        "Une stratégie alternative devra être étudiée."
    )

print("Negative sampling prévu : False")

torch_geometric importable : True
NeighborLoader disponible : True
LinkNeighborLoader disponible : True

torch_scatter : installed=False, importable=False, version=None
torch_sparse : installed=False, importable=False, version=None
pyg_lib : installed=False, importable=False, version=None

API candidate edge classification : LinkNeighborLoader, sous contrôle strict de G_<t.
Negative sampling prévu : False


In [3]:
GRAPH_DIR = Path("../data/graph")
PROCESSED_DIR = Path("../data/processed")

artifact_paths = {
    "nodes": GRAPH_DIR / "paysim_nodes.parquet",
    "src": GRAPH_DIR / "paysim_src.npy",
    "dst": GRAPH_DIR / "paysim_dst.npy",
    "step": GRAPH_DIR / "paysim_step.npy",
    "label": GRAPH_DIR / "paysim_label.npy",
    "offsets": GRAPH_DIR / "paysim_step_offsets.npy",
    "graph_metadata": GRAPH_DIR / "graph_construction_metadata.json",
    "graph_ready": PROCESSED_DIR / "paysim_graph_ready.parquet",
}

print("Répertoire courant :", Path.cwd())
print()

all_exist = True

for name, path in artifact_paths.items():
    exists = path.exists()
    all_exist &= exists

    print(f"{name:16s} : {exists} | {path}")

print()
print("Tous les artefacts existent :", all_exist)

if not all_exist:
    missing_files = [
        str(path)
        for path in artifact_paths.values()
        if not path.exists()
    ]

    raise FileNotFoundError(
        "Artefacts manquants :\n" + "\n".join(missing_files)
    )

Répertoire courant : c:\Users\bouza\OneDrive\Bureau\MobileMoney-Fraud-GNN\notebooks

nodes            : True | ..\data\graph\paysim_nodes.parquet
src              : True | ..\data\graph\paysim_src.npy
dst              : True | ..\data\graph\paysim_dst.npy
step             : True | ..\data\graph\paysim_step.npy
label            : True | ..\data\graph\paysim_label.npy
offsets          : True | ..\data\graph\paysim_step_offsets.npy
graph_metadata   : True | ..\data\graph\graph_construction_metadata.json
graph_ready      : True | ..\data\processed\paysim_graph_ready.parquet

Tous les artefacts existent : True


In [4]:
# Charger les métadonnées

with open(
    artifact_paths["graph_metadata"],
    "r",
    encoding="utf-8",
) as file:
    graph_metadata = json.load(file)


# Charger les arrays en mmap

src = np.load(
    artifact_paths["src"],
    mmap_mode="r",
)

dst = np.load(
    artifact_paths["dst"],
    mmap_mode="r",
)

step = np.load(
    artifact_paths["step"],
    mmap_mode="r",
)

label = np.load(
    artifact_paths["label"],
    mmap_mode="r",
)

step_offsets = np.load(
    artifact_paths["offsets"],
    mmap_mode="r",
)


# Lire les métadonnées Parquet

nodes_parquet = pq.ParquetFile(
    artifact_paths["nodes"]
)

graph_ready_parquet = pq.ParquetFile(
    artifact_paths["graph_ready"]
)

graph_ready_schema = graph_ready_parquet.schema_arrow


# Résumé léger

num_nodes = nodes_parquet.metadata.num_rows
num_events = src.shape[0]
num_frauds = int(label.sum(dtype=np.int64))

print("Nœuds :", f"{num_nodes:,}")
print("Événements :", f"{num_events:,}")
print("Fraudes :", f"{num_frauds:,}")

print()
print("src :", src.shape, src.dtype)
print("dst :", dst.shape, dst.dtype)
print("step :", step.shape, step.dtype)
print("label :", label.shape, label.dtype)
print("offsets :", step_offsets.shape, step_offsets.dtype)

print()
print("Nombre d'offsets :", len(step_offsets))
print("Premier offset :", int(step_offsets[0]))
print("Dernier offset :", int(step_offsets[-1]))

print()
print(
    "Lignes Graph-ready :",
    f"{graph_ready_parquet.metadata.num_rows:,}",
)
print(
    "Colonnes Graph-ready :",
    len(graph_ready_schema.names),
)

Nœuds : 9,073,900
Événements : 6,362,620
Fraudes : 8,213

src : (6362620,) int32
dst : (6362620,) int32
step : (6362620,) int16
label : (6362620,) int8
offsets : (744,) int64

Nombre d'offsets : 744
Premier offset : 0
Dernier offset : 6362620

Lignes Graph-ready : 6,362,620
Colonnes Graph-ready : 29


In [5]:
def normalize_key(value):
    return "".join(
        char.lower()
        for char in str(value)
        if char.isalnum()
    )


def find_metadata_value(data, aliases):
    targets = {
        normalize_key(alias)
        for alias in aliases
    }

    stack = [("", data)]

    while stack:
        current_path, current_value = stack.pop()

        if not isinstance(current_value, dict):
            continue

        for key, value in current_value.items():
            path = (
                f"{current_path}.{key}"
                if current_path
                else key
            )

            if normalize_key(key) in targets:
                return value, path

            if isinstance(value, dict):
                stack.append((path, value))

    return None, None


def extract_feature_list(value):
    if isinstance(value, list):
        return value

    if isinstance(value, tuple):
        return list(value)

    if isinstance(value, dict):
        for key in ["features", "columns", "names", "feature_names"]:
            candidate = value.get(key)

            if isinstance(candidate, list):
                return candidate

    return None


core_value, core_path = find_metadata_value(
    graph_metadata,
    [
        "graph_core",
        "graph_core_features",
        "graph_core_feature_names",
    ],
)

core_edge_value, core_edge_path = find_metadata_value(
    graph_metadata,
    [
        "graph_core_edge",
        "graph_core_edge_features",
        "graph_core_edge_feature_names",
    ],
)

node_history_value, node_history_path = find_metadata_value(
    graph_metadata,
    [
        "graph_node_history",
        "graph_node_history_features",
        "node_history_features",
    ],
)

full_value, full_path = find_metadata_value(
    graph_metadata,
    [
        "graph_full",
        "graph_full_features",
        "graph_full_feature_names",
    ],
)

sensitive_value, sensitive_path = find_metadata_value(
    graph_metadata,
    [
        "graph_sensitive",
        "graph_sensitive_features",
        "graph_paysim_sensitive",
        "graph_paysim_sensitive_features",
    ],
)


core_features = extract_feature_list(core_value)
core_edge_features = extract_feature_list(core_edge_value)
node_history_features = extract_feature_list(node_history_value)
full_features = extract_feature_list(full_value)
sensitive_features = extract_feature_list(sensitive_value)


# Déduire uniquement depuis les métadonnées

if (
    core_edge_features is None
    and core_features is not None
    and node_history_features is not None
):
    node_history_set = set(node_history_features)

    core_edge_features = [
        feature
        for feature in core_features
        if feature not in node_history_set
    ]


if (
    sensitive_features is None
    and full_features is not None
    and core_features is not None
):
    core_set = set(core_features)

    sensitive_features = [
        feature
        for feature in full_features
        if feature not in core_set
    ]


print(
    "Graph Core :",
    None if core_features is None else len(core_features),
    "| path :",
    core_path,
)

print(
    "Graph Core Edge :",
    None if core_edge_features is None else len(core_edge_features),
    "| path :",
    core_edge_path,
)

print(
    "Graph Node History :",
    None if node_history_features is None else len(node_history_features),
    "| path :",
    node_history_path,
)

print(
    "Graph Full :",
    None if full_features is None else len(full_features),
    "| path :",
    full_path,
)

print(
    "Graph Sensitive :",
    None if sensitive_features is None else len(sensitive_features),
    "| path :",
    sensitive_path,
)

Graph Core : 15 | path : feature_sets.graph_core
Graph Core Edge : 9 | path : feature_sets.graph_core_edge
Graph Node History : 6 | path : feature_sets.graph_node_history
Graph Full : 24 | path : feature_sets.graph_full
Graph Sensitive : 9 | path : feature_sets.graph_sensitive


In [6]:
representation, representation_path = find_metadata_value(
    graph_metadata,
    [
        "representation",
        "graph_representation",
        "temporal_representation",
    ],
)

task, task_path = find_metadata_value(
    graph_metadata,
    [
        "task",
        "graph_task",
        "prediction_task",
    ],
)

history_rule, history_rule_path = find_metadata_value(
    graph_metadata,
    [
        "history_rule",
        "causal_history_rule",
        "history_context_rule",
    ],
)

event_based, event_based_path = find_metadata_value(
    graph_metadata,
    ["event_based"],
)

edge_classification, edge_classification_path = find_metadata_value(
    graph_metadata,
    ["edge_classification"],
)

causal_history, causal_history_path = find_metadata_value(
    graph_metadata,
    ["causal_history"],
)

same_step_context, same_step_path = find_metadata_value(
    graph_metadata,
    [
        "same_step_context",
        "allow_same_step_context",
    ],
)

no_same_step_context, no_same_step_path = find_metadata_value(
    graph_metadata,
    ["no_same_step_context"],
)

identity_embedding_default, identity_path = find_metadata_value(
    graph_metadata,
    ["identity_embedding_default"],
)

no_identity_embedding, no_identity_path = find_metadata_value(
    graph_metadata,
    ["no_identity_embedding"],
)


# Résoudre les fallbacks

if representation is None and event_based is True:
    representation = "event-based"

if task is None and edge_classification is True:
    task = "edge classification"

if history_rule is None and causal_history is True:
    history_rule = "step < t"

if same_step_context is None and no_same_step_context is not None:
    same_step_context = not bool(no_same_step_context)

if (
    identity_embedding_default is None
    and no_identity_embedding is not None
):
    identity_embedding_default = not bool(
        no_identity_embedding
    )


contract = {
    "nodes_9_073_900": num_nodes == 9_073_900,
    "events_6_362_620": num_events == 6_362_620,
    "frauds_8_213": num_frauds == 8_213,
    "src_int32": src.dtype == np.int32,
    "dst_int32": dst.dtype == np.int32,
    "step_int16": step.dtype == np.int16,
    "label_int8": label.dtype == np.int8,
    "offsets_int64": step_offsets.dtype == np.int64,
    "offsets_744": len(step_offsets) == 744,
    "offset_start_0": int(step_offsets[0]) == 0,
    "offset_end_events": int(step_offsets[-1]) == num_events,
    "graph_ready_rows": (
        graph_ready_parquet.metadata.num_rows
        == num_events
    ),
    "graph_core_15": (
        core_features is not None
        and len(core_features) == 15
    ),
    "graph_core_edge_9": (
        core_edge_features is not None
        and len(core_edge_features) == 9
    ),
    "graph_full_24": (
        full_features is not None
        and len(full_features) == 24
    ),
    "graph_sensitive_9": (
        sensitive_features is not None
        and len(sensitive_features) == 9
    ),
}


print("Représentation :", representation)
print("Task :", task)
print("History rule :", history_rule)
print("Same-step context :", same_step_context)
print(
    "Identity embedding default :",
    identity_embedding_default,
)

print()

for name, valid in contract.items():
    print(f"{name:26s} : {valid}")

print()

contract_ok = all(contract.values())

print("Contrat numérique valide :", contract_ok)

causal_contract_ok = (
    str(representation).lower().replace("_", "-")
    in {"event-based", "eventbased"}
    and "edge" in str(task).lower()
    and "classification" in str(task).lower()
    and same_step_context is False
    and identity_embedding_default is False
)

print("Contrat causal minimal valide :", causal_contract_ok)

print()

if (
    contract_ok
    and causal_contract_ok
    and (
        pyg_status["LinkNeighborLoader"]
        or pyg_status["NeighborLoader"]
    )
):
    print("SECTION 1 : environnement prêt pour poursuivre l'étude.")
else:
    print(
        "SECTION 1 : au moins un point doit être analysé "
        "avant de construire le pipeline GNN."
    )

Représentation : {'node': 'PaySim account', 'edge': 'individual transaction', 'source_column': 'nameOrig', 'destination_column': 'nameDest', 'directed': True, 'multigraph_model': True, 'observed_repeated_pairs': 0, 'observed_reciprocal_relations': 0, 'task': 'edge classification', 'canonical_temporal_form': 'event-based'}
Task : edge classification
History rule : None
Same-step context : None
Identity embedding default : False

nodes_9_073_900            : True
events_6_362_620           : True
frauds_8_213               : True
src_int32                  : True
dst_int32                  : True
step_int16                 : True
label_int8                 : True
offsets_int64              : True
offsets_744                : True
offset_start_0             : True
offset_end_events          : True
graph_ready_rows           : True
graph_core_15              : True
graph_core_edge_9          : True
graph_full_24              : True
graph_sensitive_9          : True

Contrat numérique valid

In [7]:
def walk_metadata(value, path=""):
    if isinstance(value, dict):
        for key, child in value.items():
            child_path = f"{path}.{key}" if path else key
            yield child_path, child
            yield from walk_metadata(child, child_path)

    elif isinstance(value, list):
        for index, child in enumerate(value):
            child_path = f"{path}[{index}]"
            yield child_path, child
            yield from walk_metadata(child, child_path)


keywords = [
    "event",
    "causal",
    "history",
    "same",
    "identity",
    "representation",
    "task",
]

relevant_metadata = []

for path, value in walk_metadata(graph_metadata):
    path_lower = path.lower()

    if any(keyword in path_lower for keyword in keywords):
        if not isinstance(value, (dict, list)):
            relevant_metadata.append((path, value))


for path, value in relevant_metadata:
    print(f"{path} : {value}")

graph_representation.node : PaySim account
graph_representation.edge : individual transaction
graph_representation.source_column : nameOrig
graph_representation.destination_column : nameDest
graph_representation.directed : True
graph_representation.multigraph_model : True
graph_representation.observed_repeated_pairs : 0
graph_representation.observed_reciprocal_relations : 0
graph_representation.task : edge classification
graph_representation.canonical_temporal_form : event-based
event_storage.rows : 6362620
event_storage.canonical_order : chronological graph-ready row order
event_storage.arrays.src.file : paysim_src.npy
event_storage.arrays.src.dtype : int32
event_storage.arrays.dst.file : paysim_dst.npy
event_storage.arrays.dst.dtype : int32
event_storage.arrays.step.file : paysim_step.npy
event_storage.arrays.step.dtype : int16
event_storage.arrays.label.file : paysim_label.npy
event_storage.arrays.label.dtype : int8
event_storage.arrays.step_offsets.file : paysim_step_offsets.npy
ev

In [8]:
# Résoudre la représentation temporelle

if isinstance(representation, dict):
    temporal_representation = representation.get(
        "canonical_temporal_form"
    )
    graph_task = representation.get(
        "task",
        task,
    )
else:
    temporal_representation = representation
    graph_task = task


# Chercher les validations booléennes

metadata_leaves = {
    normalize_key(path.split(".")[-1]): value
    for path, value in walk_metadata(graph_metadata)
    if not isinstance(value, (dict, list))
}


event_based_flag = metadata_leaves.get(
    normalize_key("event_based")
)

causal_history_flag = metadata_leaves.get(
    normalize_key("causal_history")
)

no_same_step_flag = metadata_leaves.get(
    normalize_key("no_same_step_context")
)

no_identity_flag = metadata_leaves.get(
    normalize_key("no_identity_embedding")
)


# Résoudre les règles

if history_rule is None and causal_history_flag is True:
    resolved_history_rule = "step < t"
else:
    resolved_history_rule = history_rule


if same_step_context is None and no_same_step_flag is not None:
    resolved_same_step_context = not bool(
        no_same_step_flag
    )
else:
    resolved_same_step_context = same_step_context


if (
    identity_embedding_default is None
    and no_identity_flag is not None
):
    resolved_identity_embedding = not bool(
        no_identity_flag
    )
else:
    resolved_identity_embedding = identity_embedding_default


# Vérifier le contrat causal

representation_ok = (
    str(temporal_representation)
    .lower()
    .replace("_", "-")
    == "event-based"
)

task_ok = (
    "edge" in str(graph_task).lower()
    and "classification" in str(graph_task).lower()
)

history_ok = (
    resolved_history_rule == "step < t"
    or causal_history_flag is True
)

same_step_ok = resolved_same_step_context is False

identity_ok = resolved_identity_embedding is False


causal_contract_ok = all([
    representation_ok,
    task_ok,
    history_ok,
    same_step_ok,
    identity_ok,
])


print("Représentation :", temporal_representation)
print("Task :", graph_task)
print("History rule :", resolved_history_rule)
print(
    "Causal history flag :",
    causal_history_flag,
)
print(
    "Same-step context :",
    resolved_same_step_context,
)
print(
    "Identity embedding default :",
    resolved_identity_embedding,
)

print()
print("representation_ok :", representation_ok)
print("task_ok :", task_ok)
print("history_ok :", history_ok)
print("same_step_ok :", same_step_ok)
print("identity_ok :", identity_ok)

print()
print(
    "Contrat numérique valide :",
    contract_ok,
)
print(
    "Contrat causal minimal valide :",
    causal_contract_ok,
)

print()

if contract_ok and causal_contract_ok:
    print("SECTION 1 : contrat des artefacts validé.")
else:
    print(
        "SECTION 1 : contrat à examiner avant de poursuivre."
    )

Représentation : event-based
Task : edge classification
History rule : None
Causal history flag : None
Same-step context : None
Identity embedding default : False

representation_ok : True
task_ok : True
history_ok : False
same_step_ok : False
identity_ok : True

Contrat numérique valide : True
Contrat causal minimal valide : False

SECTION 1 : contrat à examiner avant de poursuivre.


In [9]:
# Lire directement le protocole temporel

temporal_protocol = graph_metadata.get(
    "temporal_protocol",
    {},
)

cold_start = graph_metadata.get(
    "cold_start",
    {},
)

graph_representation = graph_metadata.get(
    "graph_representation",
    {},
)


temporal_representation = graph_representation.get(
    "canonical_temporal_form"
)

graph_task = graph_representation.get(
    "task"
)

resolved_history_rule = temporal_protocol.get(
    "history_for_step_t"
)

resolved_same_step_context = temporal_protocol.get(
    "same_step_edges_in_context"
)

resolved_identity_embedding = cold_start.get(
    "identity_embedding_default"
)


representation_ok = (
    temporal_representation == "event-based"
)

task_ok = (
    graph_task == "edge classification"
)

history_ok = (
    resolved_history_rule == "step < t"
)

same_step_ok = (
    resolved_same_step_context is False
)

identity_ok = (
    resolved_identity_embedding is False
)


causal_contract_ok = all([
    representation_ok,
    task_ok,
    history_ok,
    same_step_ok,
    identity_ok,
])


print("Représentation :", temporal_representation)
print("Task :", graph_task)
print("History rule :", resolved_history_rule)
print(
    "Same-step context :",
    resolved_same_step_context,
)
print(
    "Identity embedding default :",
    resolved_identity_embedding,
)

print()
print("representation_ok :", representation_ok)
print("task_ok :", task_ok)
print("history_ok :", history_ok)
print("same_step_ok :", same_step_ok)
print("identity_ok :", identity_ok)

print()
print(
    "Contrat numérique valide :",
    contract_ok,
)
print(
    "Contrat causal minimal valide :",
    causal_contract_ok,
)

print()

if contract_ok and causal_contract_ok:
    print("SECTION 1 : environnement et contrat validés.")
else:
    print("SECTION 1 : validation incomplète.")

Représentation : event-based
Task : edge classification
History rule : step < t
Same-step context : False
Identity embedding default : False

representation_ok : True
task_ok : True
history_ok : True
same_step_ok : True
identity_ok : True

Contrat numérique valide : True
Contrat causal minimal valide : True

SECTION 1 : environnement et contrat validés.


### Runtime audit and artifact contract

L'environnement d'exécution repose sur Python 3.13.7, PyTorch 2.12.0 et PyTorch Geometric 2.7.0. Aucun GPU CUDA n'est disponible ; les expérimentations GNN devront donc être réalisées sur CPU. La machine dispose de 2 cœurs physiques, 4 threads logiques et 15,84 GB de RAM, avec une mémoire disponible observée d'environ 2,82 GB au moment de l'audit.

Les artefacts produits par le notebook de construction du graphe sont tous accessibles et cohérents avec le contrat attendu : 9 073 900 nœuds, 6 362 620 événements, 8 213 fraudes et 744 offsets temporels. Les arrays NumPy présentent les dtypes attendus et sont chargés en memory mapping afin de limiter l'utilisation mémoire. Le dataset Graph-ready contient exactement le même nombre de transactions que les arrays événementiels.

Les feature sets officiels sont également cohérents : 9 features Graph Core Edge, 6 features Graph Node History, 15 features Graph Core, 24 features Graph Full et 9 features Graph Sensitive.

La représentation canonique est event-based et la tâche est une classification d'arêtes. Le protocole temporel impose explicitement que, pour une transaction observée au step t, seul le contexte historique step < t soit disponible. Les arêtes du même step sont exclues du contexte de prédiction et l'utilisation d'un embedding appris par identité de nœud est désactivée par défaut.

NeighborLoader et LinkNeighborLoader sont importables depuis PyTorch Geometric. En revanche, les extensions optionnelles pyg-lib, torch-scatter et torch-sparse ne sont pas installées. Leur absence devra être prise en compte avant de retenir une stratégie de neighbor sampling, car l'importabilité des loaders ne garantit pas encore que leur backend de sampling soit opérationnel dans cet environnement.

Compte tenu de l'absence de GPU, du nombre élevé de nœuds et de la faible RAM disponible au moment de l'audit, la suite du notebook devra privilégier une stratégie strictement memory-efficient, avec chargement progressif des données, batching temporel et absence de représentation globale dense du graphe.

## 2. Causal Temporal Access and Protocol Validation

Cette section met en place l'accès temporel aux événements à partir des offsets validés dans le notebook de construction du graphe.

Pour chaque step `t`, le contexte historique autorisé est strictement constitué des événements dont le step est inférieur à `t`, tandis que les événements du step courant constituent les arêtes à prédire.

Le protocole appliqué est donc :

- contexte : `G_<t = events[:step_offsets[t-1]]` ;
- queries : `E_t = events[step_offsets[t-1]:step_offsets[t]]`.

Aucune arête du step courant ou d'un step futur ne doit apparaître dans le contexte utilisé pour sa prédiction.

Cette section vérifie également les frontières temporelles officielles Train, Validation et Test, sans construire encore de graphe PyTorch Geometric.

In [10]:
# Définir les splits temporels

TRAIN_START = 1
TRAIN_END = 334

VAL_START = 335
VAL_END = 392

TEST_START = 393
TEST_END = 743


SPLITS = {
    "train": (TRAIN_START, TRAIN_END),
    "validation": (VAL_START, VAL_END),
    "test": (TEST_START, TEST_END),
}


for split_name, bounds in SPLITS.items():
    print(
        f"{split_name:10s} : "
        f"steps {bounds[0]} à {bounds[1]}"
    )

train      : steps 1 à 334
validation : steps 335 à 392
test       : steps 393 à 743


In [11]:
# Définir les bornes d'un step

def get_step_bounds(t, offsets=step_offsets):
    if t < 1 or t >= len(offsets):
        raise ValueError(
            f"Step invalide : {t}. "
            f"Valeurs autorisées : 1 à {len(offsets) - 1}."
        )

    query_start = int(offsets[t - 1])
    query_end = int(offsets[t])

    return query_start, query_end


# Définir le contexte historique

def get_history_bounds(t, offsets=step_offsets):
    query_start, _ = get_step_bounds(
        t,
        offsets,
    )

    return 0, query_start


# Résumer un step

def describe_step(t):
    query_start, query_end = get_step_bounds(t)
    history_start, history_end = get_history_bounds(t)

    return {
        "step": t,
        "history_start": history_start,
        "history_end": history_end,
        "history_events": history_end - history_start,
        "query_start": query_start,
        "query_end": query_end,
        "query_events": query_end - query_start,
    }

In [12]:
# Vérifier les steps frontières

boundary_steps = [
    1,
    TRAIN_END,
    VAL_START,
    VAL_END,
    TEST_START,
    TEST_END,
]


for t in boundary_steps:
    info = describe_step(t)

    print(
        f"Step {t:3d} | "
        f"historique={info['history_events']:,} | "
        f"queries={info['query_events']:,} | "
        f"indices query="
        f"[{info['query_start']:,}, {info['query_end']:,})"
    )

Step   1 | historique=0 | queries=2,708 | indices query=[0, 2,708)
Step 334 | historique=4,759,371 | queries=12,016 | indices query=[4,759,371, 4,771,387)
Step 335 | historique=4,771,387 | queries=10,456 | indices query=[4,771,387, 4,781,843)
Step 392 | historique=5,563,783 | queries=10 | indices query=[5,563,783, 5,563,793)
Step 393 | historique=5,563,793 | queries=19,269 | indices query=[5,563,793, 5,583,062)
Step 743 | historique=6,362,612 | queries=8 | indices query=[6,362,612, 6,362,620)


In [13]:
# Vérifier la causalité aux frontières

causal_checks = {}

for t in boundary_steps:
    query_start, query_end = get_step_bounds(t)

    query_steps = step[
        query_start:query_end
    ]

    query_ok = (
        len(query_steps) > 0
        and np.all(query_steps == t)
    )

    if query_start == 0:
        history_ok = True
        last_history_step = None
    else:
        last_history_step = int(
            step[query_start - 1]
        )

        history_ok = (
            last_history_step < t
        )

    causal_checks[t] = (
        query_ok
        and history_ok
    )

    print(
        f"Step {t:3d} | "
        f"query uniquement step t : {query_ok} | "
        f"dernier step historique : {last_history_step} | "
        f"historique < t : {history_ok}"
    )


print()
print(
    "Toutes les frontières sont causales :",
    all(causal_checks.values()),
)

Step   1 | query uniquement step t : True | dernier step historique : None | historique < t : True
Step 334 | query uniquement step t : True | dernier step historique : 333 | historique < t : True
Step 335 | query uniquement step t : True | dernier step historique : 334 | historique < t : True
Step 392 | query uniquement step t : True | dernier step historique : 391 | historique < t : True
Step 393 | query uniquement step t : True | dernier step historique : 392 | historique < t : True
Step 743 | query uniquement step t : True | dernier step historique : 742 | historique < t : True

Toutes les frontières sont causales : True


In [14]:
# Calculer les tailles depuis les offsets

def split_event_count(start_step, end_step):
    start_index = int(
        step_offsets[start_step - 1]
    )

    end_index = int(
        step_offsets[end_step]
    )

    return end_index - start_index


split_counts = {
    split_name: split_event_count(
        start_step,
        end_step,
    )
    for split_name, (
        start_step,
        end_step,
    ) in SPLITS.items()
}


expected_counts = {
    "train": 4_771_387,
    "validation": 792_406,
    "test": 798_827,
}


for split_name in SPLITS:
    observed = split_counts[
        split_name
    ]

    expected = expected_counts[
        split_name
    ]

    print(
        f"{split_name:10s} : "
        f"{observed:,} | "
        f"attendu={expected:,} | "
        f"valide={observed == expected}"
    )


print()
print(
    "Total :",
    f"{sum(split_counts.values()):,}",
)

print(
    "Total valide :",
    sum(split_counts.values())
    == len(src),
)

train      : 4,771,387 | attendu=4,771,387 | valide=True
validation : 792,406 | attendu=792,406 | valide=True
test       : 798,827 | attendu=798,827 | valide=True

Total : 6,362,620
Total valide : True


In [15]:
# Calculer les fraudes par split

expected_frauds = {
    "train": 3_745,
    "validation": 638,
    "test": 3_830,
}


split_frauds = {}

for split_name, (
    start_step,
    end_step,
) in SPLITS.items():

    start_index = int(
        step_offsets[start_step - 1]
    )

    end_index = int(
        step_offsets[end_step]
    )

    fraud_count = int(
        label[start_index:end_index].sum(
            dtype=np.int64
        )
    )

    split_frauds[split_name] = fraud_count

    print(
        f"{split_name:10s} : "
        f"{fraud_count:,} fraudes | "
        f"attendu={expected_frauds[split_name]:,} | "
        f"valide="
        f"{fraud_count == expected_frauds[split_name]}"
    )


print()
print(
    "Fraudes totales :",
    sum(split_frauds.values()),
)

train      : 3,745 fraudes | attendu=3,745 | valide=True
validation : 638 fraudes | attendu=638 | valide=True
test       : 3,830 fraudes | attendu=3,830 | valide=True

Fraudes totales : 8213


In [16]:
# Vérifier le premier step Validation

t = VAL_START

query_start, query_end = get_step_bounds(t)

history_events = query_start
query_events = query_end - query_start

train_events = split_counts["train"]


print("Step :", t)

print(
    "Événements historiques :",
    f"{history_events:,}",
)

print(
    "Événements Train :",
    f"{train_events:,}",
)

print(
    "Historique = Train complet :",
    history_events == train_events,
)

print(
    "Transactions du step courant :",
    f"{query_events:,}",
)

print(
    "Premier step query :",
    int(step[query_start]),
)

print(
    "Dernier step query :",
    int(step[query_end - 1]),
)

print(
    "Dernier step du contexte :",
    int(step[query_start - 1]),
)

Step : 335
Événements historiques : 4,771,387
Événements Train : 4,771,387
Historique = Train complet : True
Transactions du step courant : 10,456
Premier step query : 335
Dernier step query : 335
Dernier step du contexte : 334


In [17]:
# Valider la section

temporal_protocol_valid = (
    all(causal_checks.values())
    and split_counts == expected_counts
    and split_frauds == expected_frauds
    and sum(split_counts.values()) == len(src)
    and sum(split_frauds.values()) == 8_213
)


print(
    "Accès temporel causal valide :",
    temporal_protocol_valid,
)

print(
    "Même-step dans le contexte :",
    False,
)

print(
    "Information future utilisée :",
    False,
)

print(
    "Arrays toujours memory-mapped :",
    isinstance(src, np.memmap)
    and isinstance(dst, np.memmap)
    and isinstance(step, np.memmap)
    and isinstance(label, np.memmap)
    and isinstance(step_offsets, np.memmap),
)


if temporal_protocol_valid:
    print(
        "SECTION 2 : protocole temporel validé."
    )
else:
    print(
        "SECTION 2 : anomalie à analyser."
    )

Accès temporel causal valide : True
Même-step dans le contexte : False
Information future utilisée : False
Arrays toujours memory-mapped : True
SECTION 2 : protocole temporel validé.


### Causal temporal protocol validation

L'accès temporel aux événements a été reconstruit directement à partir des offsets validés du graphe, sans recalcul ni chargement complet du dataset.

Pour chaque step `t`, les transactions courantes sont définies par `E_t = events[step_offsets[t-1]:step_offsets[t]]`, tandis que le contexte historique autorisé correspond strictement à `G_<t = events[:step_offsets[t-1]]`.

Les vérifications effectuées aux principales frontières temporelles confirment que les queries contiennent uniquement le step courant et que le dernier événement du contexte appartient toujours à un step strictement antérieur. Aucun contexte intra-step ni aucune information future ne sont donc utilisés.

Les splits temporels reconstruits correspondent exactement au protocole officiel :

- Train : 4 771 387 transactions et 3 745 fraudes ;
- Validation : 792 406 transactions et 638 fraudes ;
- Test : 798 827 transactions et 3 830 fraudes.

Au premier step de Validation (`t = 335`), le contexte historique contient exactement les 4 771 387 transactions du Train et s'arrête au step 334. Le step 335 reste entièrement réservé aux queries à prédire.

Les arrays événementiels restent chargés en memory mapping, ce qui permet de conserver un accès causal aux données sans matérialiser inutilement l'ensemble du graphe en mémoire.

## 3. GNN Input Preprocessing - Graph Core Edge

Cette section prépare les features de transaction utilisées par le premier baseline GNN.

L'expérience `GNN Core Edge` utilise uniquement les features transactionnelles courantes du jeu officiel `Graph Core Edge`. Les variables numériques continues sont standardisées à partir de statistiques calculées exclusivement sur le Train, les variables binaires restent non standardisées et la variable catégorielle `type` est encodée en one-hot.

Le preprocessing est ajusté en streaming afin d'éviter de charger l'ensemble du dataset Graph-ready en mémoire. Les transformations finales produisent des entrées `float32` adaptées à PyTorch.

Aucune information de Validation ou de Test n'est utilisée pour ajuster les paramètres de preprocessing.

In [18]:
# Charger le feature set officiel

core_edge_features = graph_metadata[
    "feature_sets"
]["graph_core_edge"]

print(
    "Graph Core Edge :",
    len(core_edge_features),
)

print(core_edge_features)


# Vérifier le schéma Parquet

missing_core_edge = [
    feature
    for feature in core_edge_features
    if feature not in graph_ready_schema.names
]

print()
print(
    "Features manquantes :",
    missing_core_edge,
)

print()
print("Dtypes Parquet :")

for feature in core_edge_features:
    field = graph_ready_schema.field(feature)

    print(
        f"{feature:30s} : {field.type}"
    )

Graph Core Edge : 9
['amount', 'log_amount', 'type', 'origin_balance_change', 'destination_balance_change', 'origin_zero_before', 'origin_zero_after', 'destination_zero_before', 'destination_zero_after']

Features manquantes : []

Dtypes Parquet :
amount                         : double
log_amount                     : float
type                           : dictionary<values=string, indices=int8, ordered=0>
origin_balance_change          : float
destination_balance_change     : float
origin_zero_before             : int8
origin_zero_after              : int8
destination_zero_before        : int8
destination_zero_after         : int8


In [19]:
# Identifier les groupes de variables

categorical_features = [
    feature
    for feature in core_edge_features
    if feature == "type"
]

binary_features = [
    feature
    for feature in core_edge_features
    if (
        "zero_before" in feature
        or "zero_after" in feature
    )
]

continuous_features = [
    feature
    for feature in core_edge_features
    if (
        feature not in categorical_features
        and feature not in binary_features
    )
]


print(
    "Variables continues :",
    continuous_features,
)

print(
    "Variables binaires :",
    binary_features,
)

print(
    "Variables catégorielles :",
    categorical_features,
)

print()

partition_ok = (
    set(
        continuous_features
        + binary_features
        + categorical_features
    )
    == set(core_edge_features)
)

no_overlap = (
    len(
        continuous_features
        + binary_features
        + categorical_features
    )
    == len(core_edge_features)
)

print(
    "Partition complète :",
    partition_ok,
)

print(
    "Aucun chevauchement :",
    no_overlap,
)

Variables continues : ['amount', 'log_amount', 'origin_balance_change', 'destination_balance_change']
Variables binaires : ['origin_zero_before', 'origin_zero_after', 'destination_zero_before', 'destination_zero_after']
Variables catégorielles : ['type']

Partition complète : True
Aucun chevauchement : True


In [20]:
import pyarrow.compute as pc


# Paramètres du streaming

PREPROCESS_BATCH_SIZE = 100_000
TRAIN_ROWS = split_counts["train"]


# Initialiser les statistiques

stats = {
    feature: {
        "count": 0,
        "sum": 0.0,
        "sum_sq": 0.0,
        "min": np.inf,
        "max": -np.inf,
    }
    for feature in continuous_features
}

train_type_categories = set()

rows_seen = 0


# Parcourir uniquement le Train

batch_iterator = graph_ready_parquet.iter_batches(
    columns=core_edge_features,
    batch_size=PREPROCESS_BATCH_SIZE,
    use_threads=False,
)


for batch in batch_iterator:

    remaining = TRAIN_ROWS - rows_seen

    if remaining <= 0:
        break

    if batch.num_rows > remaining:
        batch = batch.slice(
            0,
            remaining,
        )

    # Statistiques numériques
    for feature in continuous_features:

        column = batch.column(
            batch.schema.get_field_index(feature)
        )

        values = column.to_numpy(
            zero_copy_only=False
        ).astype(
            np.float64,
            copy=False,
        )

        feature_stats = stats[feature]

        feature_stats["count"] += values.size
        feature_stats["sum"] += float(
            values.sum(dtype=np.float64)
        )
        feature_stats["sum_sq"] += float(
            np.square(
                values,
                dtype=np.float64,
            ).sum(dtype=np.float64)
        )

        feature_stats["min"] = min(
            feature_stats["min"],
            float(values.min()),
        )

        feature_stats["max"] = max(
            feature_stats["max"],
            float(values.max()),
        )

    # Catégories de type
    type_column = batch.column(
        batch.schema.get_field_index("type")
    )

    unique_types = pc.unique(
        type_column
    ).to_pylist()

    train_type_categories.update(
        value
        for value in unique_types
        if value is not None
    )

    rows_seen += batch.num_rows


print(
    "Lignes Train parcourues :",
    f"{rows_seen:,}",
)

print(
    "Train complet utilisé :",
    rows_seen == TRAIN_ROWS,
)

Lignes Train parcourues : 4,771,387
Train complet utilisé : True


In [21]:
# Calculer moyenne et écart-type

train_scaler = {}


for feature in continuous_features:

    feature_stats = stats[feature]

    count = feature_stats["count"]

    mean = (
        feature_stats["sum"]
        / count
    )

    variance = (
        feature_stats["sum_sq"]
        / count
        - mean ** 2
    )

    variance = max(
        variance,
        0.0,
    )

    std = np.sqrt(
        variance
    )

    train_scaler[feature] = {
        "mean": mean,
        "std": std,
        "min": feature_stats["min"],
        "max": feature_stats["max"],
    }


for feature, values in train_scaler.items():

    print(feature)

    print(
        f"  mean : {values['mean']:.6f}"
    )

    print(
        f"  std  : {values['std']:.6f}"
    )

    print(
        f"  min  : {values['min']:.6f}"
    )

    print(
        f"  max  : {values['max']:.6f}"
    )


print()

scaler_valid = all(
    np.isfinite(values["mean"])
    and np.isfinite(values["std"])
    and values["std"] > 0
    for values in train_scaler.values()
)

print(
    "Scaler Train valide :",
    scaler_valid,
)

amount
  mean : 179748.531939
  std  : 648694.425604
  min  : 0.000000
  max  : 92445516.640000
log_amount
  mean : 10.833058
  std  : 1.820169
  min  : 0.000000
  max  : 18.342131
origin_balance_change
  mean : -21731.548861
  std  : 128793.638313
  min  : -1915267.875000
  max  : 10000000.000000
destination_balance_change
  mean : 130154.927112
  std  : 900249.394962
  min  : -13060826.000000
  max  : 105687840.000000

Scaler Train valide : True


In [22]:
# Construire le vocabulaire catégoriel

type_categories = sorted(
    train_type_categories
)

type_to_index = {
    category: index
    for index, category
    in enumerate(type_categories)
}


print(
    "Catégories Train :",
    type_categories,
)

print(
    "Nombre de catégories :",
    len(type_categories),
)

print(
    "Mapping :",
    type_to_index,
)


core_edge_input_dim = (
    len(continuous_features)
    + len(binary_features)
    + len(type_categories)
)


print()
print(
    "Dimension continue :",
    len(continuous_features),
)

print(
    "Dimension binaire :",
    len(binary_features),
)

print(
    "Dimension one-hot type :",
    len(type_categories),
)

print(
    "Dimension finale Core Edge :",
    core_edge_input_dim,
)

Catégories Train : ['CASH_IN', 'CASH_OUT', 'DEBIT', 'PAYMENT', 'TRANSFER']
Nombre de catégories : 5
Mapping : {'CASH_IN': 0, 'CASH_OUT': 1, 'DEBIT': 2, 'PAYMENT': 3, 'TRANSFER': 4}

Dimension continue : 4
Dimension binaire : 4
Dimension one-hot type : 5
Dimension finale Core Edge : 13


In [23]:
# Transformer un RecordBatch

def transform_core_edge_batch(batch):

    n_rows = batch.num_rows

    numeric_parts = []


    # Standardiser les variables continues

    for feature in continuous_features:

        column = batch.column(
            batch.schema.get_field_index(feature)
        )

        values = column.to_numpy(
            zero_copy_only=False
        ).astype(
            np.float32,
            copy=False,
        )

        mean = train_scaler[
            feature
        ]["mean"]

        std = train_scaler[
            feature
        ]["std"]

        scaled = (
            values - mean
        ) / std

        numeric_parts.append(
            scaled.reshape(-1, 1)
        )


    # Ajouter les variables binaires

    for feature in binary_features:

        column = batch.column(
            batch.schema.get_field_index(feature)
        )

        values = column.to_numpy(
            zero_copy_only=False
        ).astype(
            np.float32,
            copy=False,
        )

        numeric_parts.append(
            values.reshape(-1, 1)
        )


    # Encoder type en one-hot

    type_column = batch.column(
        batch.schema.get_field_index("type")
    )

    type_values = type_column.to_pylist()

    type_one_hot = np.zeros(
        (
            n_rows,
            len(type_categories),
        ),
        dtype=np.float32,
    )

    unknown_type_count = 0

    for row_index, category in enumerate(
        type_values
    ):

        category_index = type_to_index.get(
            category
        )

        if category_index is None:
            unknown_type_count += 1
            continue

        type_one_hot[
            row_index,
            category_index,
        ] = 1.0


    numeric_parts.append(
        type_one_hot
    )


    features = np.concatenate(
        numeric_parts,
        axis=1,
    ).astype(
        np.float32,
        copy=False,
    )


    return features, unknown_type_count

In [24]:
# Lire un petit batch Train

sample_iterator = graph_ready_parquet.iter_batches(
    columns=core_edge_features,
    batch_size=2_048,
    use_threads=False,
)

sample_batch = next(
    sample_iterator
)


# Transformer le batch

sample_features, sample_unknown_types = (
    transform_core_edge_batch(
        sample_batch
    )
)


print(
    "Shape entrée brute :",
    (
        sample_batch.num_rows,
        len(core_edge_features),
    ),
)

print(
    "Shape après preprocessing :",
    sample_features.shape,
)

print(
    "Dtype final :",
    sample_features.dtype,
)

print(
    "NaN présents :",
    bool(
        np.isnan(
            sample_features
        ).any()
    ),
)

print(
    "Inf présents :",
    bool(
        np.isinf(
            sample_features
        ).any()
    ),
)

print(
    "Types inconnus :",
    sample_unknown_types,
)


print()

print(
    "Dimension correcte :",
    sample_features.shape[1]
    == core_edge_input_dim,
)

print(
    "float32 valide :",
    sample_features.dtype
    == np.float32,
)

Shape entrée brute : (2048, 9)
Shape après preprocessing : (2048, 13)
Dtype final : float32
NaN présents : False
Inf présents : False
Types inconnus : 0

Dimension correcte : True
float32 valide : True


In [25]:
# Vérifier la mémoire après preprocessing

if "psutil" in sys.modules:
    memory = psutil.virtual_memory()

    print(
        f"RAM totale : "
        f"{memory.total / 1024**3:.2f} GB"
    )

    print(
        f"RAM disponible : "
        f"{memory.available / 1024**3:.2f} GB"
    )

    print(
        f"RAM utilisée : "
        f"{memory.percent:.1f} %"
    )


print()

preprocessing_valid = (
    missing_core_edge == []
    and partition_ok
    and no_overlap
    and rows_seen == TRAIN_ROWS
    and scaler_valid
    and sample_features.shape[1]
        == core_edge_input_dim
    and sample_features.dtype
        == np.float32
    and not np.isnan(
        sample_features
    ).any()
    and not np.isinf(
        sample_features
    ).any()
)


print(
    "Preprocessing Core Edge valide :",
    preprocessing_valid,
)


if preprocessing_valid:
    print(
        "SECTION 3 : preprocessing Core Edge validé."
    )
else:
    print(
        "SECTION 3 : point à analyser avant de poursuivre."
    )

RAM totale : 15.84 GB
RAM disponible : 1.83 GB
RAM utilisée : 88.5 %

Preprocessing Core Edge valide : True
SECTION 3 : preprocessing Core Edge validé.


### Graph Core Edge preprocessing validation

Le premier baseline GNN utilise les 9 features officielles du jeu `Graph Core Edge`. Le schéma du Parquet confirme leur présence et leur compatibilité avec le pipeline de preprocessing.

Les variables ont été réparties en quatre features numériques continues, quatre features binaires et une variable catégorielle `type`. Les cinq catégories transactionnelles de PaySim sont toutes présentes dans le Train.

Les paramètres de standardisation des variables continues ont été calculés exclusivement sur les 4 771 387 transactions du Train, par lecture progressive du Parquet. Aucune observation de Validation ou de Test n'a été utilisée pour ajuster le preprocessing.

La variable `type` est représentée par un one-hot encoding déterministe à cinq dimensions, tandis que les variables binaires sont conservées sans standardisation. La représentation finale d'une transaction `Graph Core Edge` possède ainsi 13 dimensions.

Un micro-test sur 2 048 transactions confirme que le pipeline produit correctement une matrice `float32` de dimension `(2048, 13)`, sans valeur manquante, valeur infinie ou catégorie inconnue.

La mémoire disponible après cette étape est néanmoins limitée à environ 1,83 GB. La construction du futur contexte graphique devra donc éviter toute duplication globale des arrays, notamment les conversions simultanées de plusieurs millions d'identifiants de nœuds vers `int64`. Les prochaines étapes privilégieront des micro-tests et une matérialisation strictement contrôlée du contexte historique.

In [27]:
# Libérer les objets temporaires

import gc

for name in [
    "batch_iterator",
    "batch",
    "sample_iterator",
    "sample_batch",
    "sample_features",
]:
    if name in globals():
        del globals()[name]

gc.collect()

if "psutil" in sys.modules:
    memory = psutil.virtual_memory()

    print(
        f"RAM disponible : "
        f"{memory.available / 1024**3:.2f} GB"
    )

    print(
        f"RAM utilisée : "
        f"{memory.percent:.1f} %"
    )

RAM disponible : 1.56 GB
RAM utilisée : 90.2 %


In [28]:
# Auditer la mémoire du kernel

import os
import gc
import psutil

gc.collect()

process = psutil.Process(os.getpid())
process_memory = process.memory_info()

system_memory = psutil.virtual_memory()

print(
    f"RAM kernel Python : "
    f"{process_memory.rss / 1024**3:.2f} GB"
)

print(
    f"RAM système disponible : "
    f"{system_memory.available / 1024**3:.2f} GB"
)

print(
    f"RAM système utilisée : "
    f"{system_memory.percent:.1f} %"
)

print()

print(
    "src memmap :",
    isinstance(src, np.memmap),
)

print(
    "dst memmap :",
    isinstance(dst, np.memmap),
)

print(
    "step memmap :",
    isinstance(step, np.memmap),
)

print(
    "label memmap :",
    isinstance(label, np.memmap),
)

print(
    "offsets memmap :",
    isinstance(step_offsets, np.memmap),
)

RAM kernel Python : 0.33 GB
RAM système disponible : 2.01 GB
RAM système utilisée : 87.3 %

src memmap : True
dst memmap : True
step memmap : True
label memmap : True
offsets memmap : True


## 4. PyG Sampling Feasibility and Memory Micro-Test

Cette section vérifie si les APIs de neighbor sampling de PyTorch Geometric sont réellement opérationnelles dans l'environnement courant.

Le test utilise un très petit contexte causal : pour les transactions du step 2, seules les transactions du step 1 sont présentes dans le graphe historique. Les identifiants globaux des comptes sont remappés localement afin d'éviter toute structure indexée sur les 9 millions de nœuds du dataset.

L'objectif est uniquement de tester le backend de sampling, la séparation entre contexte historique et query edges, ainsi que l'impact mémoire. Aucun modèle GNN n'est construit ou entraîné.

In [29]:
# Vérifier les capacités PyG

import torch_geometric.typing as pyg_typing

print(
    "WITH_PYG_LIB :",
    getattr(pyg_typing, "WITH_PYG_LIB", None),
)

print(
    "WITH_TORCH_SPARSE :",
    getattr(pyg_typing, "WITH_TORCH_SPARSE", None),
)

print(
    "NeighborLoader importable :",
    pyg_status["NeighborLoader"],
)

print(
    "LinkNeighborLoader importable :",
    pyg_status["LinkNeighborLoader"],
)

WITH_PYG_LIB : False
WITH_TORCH_SPARSE : False
NeighborLoader importable : True
LinkNeighborLoader importable : True


In [30]:
# Définir le micro-test causal

MICRO_STEP = 2
MICRO_QUERY_LIMIT = 512

query_start, query_end = get_step_bounds(
    MICRO_STEP
)

history_start, history_end = get_history_bounds(
    MICRO_STEP
)


# Lire le contexte historique

context_src_global = np.asarray(
    src[history_start:history_end],
    dtype=np.int32,
)

context_dst_global = np.asarray(
    dst[history_start:history_end],
    dtype=np.int32,
)


# Limiter les queries du micro-test

micro_query_end = min(
    query_start + MICRO_QUERY_LIMIT,
    query_end,
)

query_src_global = np.asarray(
    src[query_start:micro_query_end],
    dtype=np.int32,
)

query_dst_global = np.asarray(
    dst[query_start:micro_query_end],
    dtype=np.int32,
)

query_label_micro = np.asarray(
    label[query_start:micro_query_end],
    dtype=np.int8,
)


print("Step query :", MICRO_STEP)
print(
    "Arêtes contexte :",
    len(context_src_global),
)

print(
    "Arêtes query disponibles :",
    query_end - query_start,
)

print(
    "Arêtes query utilisées :",
    len(query_src_global),
)

print(
    "Dernier step contexte :",
    int(step[history_end - 1]),
)

print(
    "Premier step query :",
    int(step[query_start]),
)

print(
    "Contexte strictement passé :",
    int(step[history_end - 1]) < MICRO_STEP,
)

Step query : 2
Arêtes contexte : 2708
Arêtes query disponibles : 1014
Arêtes query utilisées : 512
Dernier step contexte : 1
Premier step query : 2
Contexte strictement passé : True


In [31]:
# Remapper les nœuds localement

all_nodes_global = np.concatenate([
    context_src_global,
    context_dst_global,
    query_src_global,
    query_dst_global,
])

local_nodes, inverse = np.unique(
    all_nodes_global,
    return_inverse=True,
)


n_context = len(context_src_global)
n_query = len(query_src_global)


context_src_local = inverse[
    :n_context
]

context_dst_local = inverse[
    n_context:2 * n_context
]

query_src_local = inverse[
    2 * n_context:
    2 * n_context + n_query
]

query_dst_local = inverse[
    2 * n_context + n_query:
]


print(
    "Nœuds locaux :",
    len(local_nodes),
)

print(
    "ID local min :",
    int(inverse.min()),
)

print(
    "ID local max :",
    int(inverse.max()),
)

print(
    "Mapping local valide :",
    int(inverse.max())
    < len(local_nodes),
)

Nœuds locaux : 5151
ID local min : 0
ID local max : 5150
Mapping local valide : True


In [32]:
# Construire les petits tenseurs PyG

from torch_geometric.data import Data


context_edge_index = torch.from_numpy(
    np.stack([
        context_src_local,
        context_dst_local,
    ])
).long()


query_edge_index = torch.from_numpy(
    np.stack([
        query_src_local,
        query_dst_local,
    ])
).long()


query_edge_labels = torch.from_numpy(
    query_label_micro.astype(
        np.float32,
        copy=False,
    )
)


micro_data = Data(
    edge_index=context_edge_index,
    num_nodes=len(local_nodes),
)


print(
    "Context edge_index :",
    tuple(context_edge_index.shape),
)

print(
    "Query edge_index :",
    tuple(query_edge_index.shape),
)

print(
    "Query labels :",
    tuple(query_edge_labels.shape),
)

print(
    "Nombre de nœuds PyG :",
    micro_data.num_nodes,
)

print(
    "Context dtype :",
    context_edge_index.dtype,
)

print(
    "Query dtype :",
    query_edge_index.dtype,
)

Context edge_index : (2, 2708)
Query edge_index : (2, 512)
Query labels : (512,)
Nombre de nœuds PyG : 5151
Context dtype : torch.int64
Query dtype : torch.int64


In [33]:
# Vérifier la séparation contexte / queries

context_pairs = set(
    zip(
        context_src_global.tolist(),
        context_dst_global.tolist(),
    )
)

query_pairs = set(
    zip(
        query_src_global.tolist(),
        query_dst_global.tolist(),
    )
)

pair_overlap = (
    context_pairs
    & query_pairs
)


print(
    "Couples contexte :",
    len(context_pairs),
)

print(
    "Couples query :",
    len(query_pairs),
)

print(
    "Overlap contexte / query :",
    len(pair_overlap),
)

print(
    "Query-edge leakage :",
    len(pair_overlap) > 0,
)

Couples contexte : 2708
Couples query : 512
Overlap contexte / query : 0
Query-edge leakage : False


In [34]:
# Tester LinkNeighborLoader

link_loader_created = False
link_sampling_works = False
link_sampling_error = None
sampled_batch = None


try:
    link_loader = LinkNeighborLoader(
        micro_data,
        num_neighbors=[5],
        edge_label_index=query_edge_index,
        edge_label=query_edge_labels,
        batch_size=128,
        shuffle=False,
        neg_sampling=None,
        num_workers=0,
    )

    link_loader_created = True

    print(
        "LinkNeighborLoader construit :",
        True,
    )

    sampled_batch = next(
        iter(link_loader)
    )

    link_sampling_works = True

    print(
        "Sampling exécuté :",
        True,
    )

except Exception as exc:
    link_sampling_error = exc

    print(
        "LinkNeighborLoader construit :",
        link_loader_created,
    )

    print(
        "Sampling exécuté :",
        False,
    )

    print(
        "Type erreur :",
        type(exc).__name__,
    )

    print(
        "Erreur :",
        str(exc),
    )

LinkNeighborLoader construit : True
LinkNeighborLoader construit : True
Sampling exécuté : False
Type erreur : ImportError
Erreur : 'NeighborSampler' requires either 'pyg-lib' or 'torch-sparse'


In [35]:
# Inspecter le batch échantillonné

if link_sampling_works:

    print(
        "Batch num_nodes :",
        sampled_batch.num_nodes,
    )

    print(
        "Batch num_edges :",
        sampled_batch.num_edges,
    )

    print(
        "edge_index shape :",
        tuple(
            sampled_batch.edge_index.shape
        ),
    )

    if hasattr(
        sampled_batch,
        "edge_label_index",
    ):
        print(
            "edge_label_index shape :",
            tuple(
                sampled_batch.edge_label_index.shape
            ),
        )

    if hasattr(
        sampled_batch,
        "edge_label",
    ):
        print(
            "edge_label shape :",
            tuple(
                sampled_batch.edge_label.shape
            ),
        )

else:
    print(
        "Inspection batch ignorée : "
        "sampling indisponible."
    )

Inspection batch ignorée : sampling indisponible.


In [36]:
# Vérifier la mémoire après le test

gc.collect()

process = psutil.Process(
    os.getpid()
)

process_memory = process.memory_info()
system_memory = psutil.virtual_memory()


print(
    f"RAM kernel Python : "
    f"{process_memory.rss / 1024**3:.2f} GB"
)

print(
    f"RAM système disponible : "
    f"{system_memory.available / 1024**3:.2f} GB"
)

print(
    f"RAM système utilisée : "
    f"{system_memory.percent:.1f} %"
)

print()

print(
    "LinkNeighborLoader importable :",
    pyg_status["LinkNeighborLoader"],
)

print(
    "LinkNeighborLoader construit :",
    link_loader_created,
)

print(
    "Neighbor sampling opérationnel :",
    link_sampling_works,
)

RAM kernel Python : 0.33 GB
RAM système disponible : 2.88 GB
RAM système utilisée : 81.8 %

LinkNeighborLoader importable : True
LinkNeighborLoader construit : True
Neighbor sampling opérationnel : False


### PyG sampling feasibility

Un micro-test causal a été réalisé sur le step 2 afin de vérifier la faisabilité du neighbor sampling dans l'environnement PyTorch Geometric disponible.

Le contexte contient uniquement les 2 708 transactions du step 1, tandis qu'un sous-ensemble de 512 transactions du step 2 est utilisé comme queries. Les identifiants globaux des comptes ont été remappés localement sur 5 151 nœuds afin d'éviter toute structure dimensionnée sur les 9 millions de nœuds du graphe complet.

La séparation causale est correctement respectée : aucune paire source-destination utilisée comme query n'apparaît dans le contexte historique du micro-test.

`NeighborLoader` et `LinkNeighborLoader` sont importables depuis PyTorch Geometric, mais leur backend de neighbor sampling n'est pas opérationnel dans l'environnement actuel. L'exécution de `LinkNeighborLoader` échoue car ni `pyg-lib` ni `torch-sparse` ne sont disponibles.

Cette limitation concerne le mécanisme de sampling et non les couches de message passing PyTorch Geometric elles-mêmes. La suite devra donc soit utiliser une extraction contrôlée des voisinages suivie d'un message passing PyG sur de petits sous-graphes, soit adopter une autre stratégie sparse compatible avec le protocole causal.

Aucune installation supplémentaire n'est effectuée automatiquement.

## 5. GraphSAGE Structural Signal Sanity Check

Avant de définir définitivement l'architecture du baseline, cette section vérifie qu'un état initial partagé entre les nœuds permet réellement à GraphSAGE d'exploiter la topologie particulière de PaySim.

Dans un graphe principalement constitué d'étoiles convergentes, une agrégation moyenne appliquée à des représentations initiales identiques peut supprimer l'information liée au nombre de voisins historiques.

Un micro-test est donc réalisé sur le contexte causal déjà construit afin de comparer le comportement structurel des agrégations `mean` et `sum`.

Aucun apprentissage supervisé n'est effectué.

In [37]:
# Calculer les degrés entrants locaux

micro_num_nodes = len(local_nodes)

in_degree = torch.bincount(
    context_edge_index[1],
    minlength=micro_num_nodes,
)

positive_degree = in_degree[
    in_degree > 0
]


print(
    "Nœuds locaux :",
    micro_num_nodes,
)

print(
    "Nœuds sans voisin entrant :",
    int((in_degree == 0).sum()),
)

print(
    "Nœuds avec voisin entrant :",
    int((in_degree > 0).sum()),
)

print(
    "Nœuds avec degré entrant > 1 :",
    int((in_degree > 1).sum()),
)

print(
    "Degré entrant maximal :",
    int(in_degree.max()),
)

print(
    "Degrés entrants positifs distincts :",
    torch.unique(
        positive_degree
    ).tolist(),
)

Nœuds locaux : 5151
Nœuds sans voisin entrant : 3518
Nœuds avec voisin entrant : 1633
Nœuds avec degré entrant > 1 : 181
Degré entrant maximal : 32
Degrés entrants positifs distincts : [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 20, 21, 22, 28, 32]


In [38]:
# Créer un état initial partagé

shared_state = torch.ones(
    (
        micro_num_nodes,
        1,
    ),
    dtype=torch.float32,
)


# Construire les messages source

source_messages = shared_state[
    context_edge_index[0]
]


# Agrégation somme

sum_aggregation = torch.zeros(
    (
        micro_num_nodes,
        1,
    ),
    dtype=torch.float32,
)

sum_aggregation.index_add_(
    0,
    context_edge_index[1],
    source_messages,
)


# Agrégation moyenne

mean_aggregation = torch.zeros_like(
    sum_aggregation
)

positive_mask = (
    in_degree > 0
)

mean_aggregation[
    positive_mask
] = (
    sum_aggregation[
        positive_mask
    ]
    / in_degree[
        positive_mask
    ].float().unsqueeze(1)
)


print(
    "Valeurs mean pour degré > 0 :",
    torch.unique(
        mean_aggregation[
            positive_mask
        ]
    ).tolist(),
)

print(
    "Mean distingue les degrés positifs :",
    len(
        torch.unique(
            mean_aggregation[
                positive_mask
            ]
        )
    ) > 1,
)

print()

print(
    "Sum = degré entrant :",
    torch.allclose(
        sum_aggregation.squeeze(),
        in_degree.float(),
    ),
)

print(
    "Valeurs sum distinctes :",
    torch.unique(
        sum_aggregation
    ).tolist(),
)

Valeurs mean pour degré > 0 : [1.0]
Mean distingue les degrés positifs : False

Sum = degré entrant : True
Valeurs sum distinctes : [0.0, 1.0, 2.0, 3.0, 4.0, 5.0, 6.0, 7.0, 8.0, 9.0, 10.0, 11.0, 12.0, 13.0, 14.0, 15.0, 16.0, 17.0, 18.0, 20.0, 21.0, 22.0, 28.0, 32.0]


In [39]:
# Tester SAGEConv directement

from torch_geometric.nn import SAGEConv

torch.manual_seed(
    RANDOM_STATE
)


x_shared = torch.ones(
    (
        micro_num_nodes,
        1,
    ),
    dtype=torch.float32,
)


sage_mean = SAGEConv(
    in_channels=1,
    out_channels=4,
    aggr="mean",
)


sage_sum = SAGEConv(
    in_channels=1,
    out_channels=4,
    aggr="sum",
)


sage_forward_mean_ok = False
sage_forward_sum_ok = False


try:
    with torch.no_grad():
        h_mean = sage_mean(
            x_shared,
            context_edge_index,
        )

    sage_forward_mean_ok = True

except Exception as exc:
    print(
        "Erreur SAGE mean :",
        repr(exc),
    )


try:
    with torch.no_grad():
        h_sum = sage_sum(
            x_shared,
            context_edge_index,
        )

    sage_forward_sum_ok = True

except Exception as exc:
    print(
        "Erreur SAGE sum :",
        repr(exc),
    )


print(
    "SAGEConv mean opérationnel :",
    sage_forward_mean_ok,
)

print(
    "SAGEConv sum opérationnel :",
    sage_forward_sum_ok,
)

print(
    "Flow SAGEConv :",
    sage_mean.flow,
)

SAGEConv mean opérationnel : True
SAGEConv sum opérationnel : True
Flow SAGEConv : source_to_target


In [40]:
# Mesurer la dispersion des embeddings

if sage_forward_mean_ok:

    positive_indices = torch.where(
        positive_mask
    )[0]

    zero_indices = torch.where(
        ~positive_mask
    )[0]


    if len(positive_indices) > 0:

        reference_positive = h_mean[
            positive_indices[0]
        ]

        mean_positive_spread = (
            h_mean[
                positive_indices
            ]
            - reference_positive
        ).abs().max().item()

    else:
        mean_positive_spread = np.nan


    if len(zero_indices) > 0:

        reference_zero = h_mean[
            zero_indices[0]
        ]

        mean_zero_spread = (
            h_mean[
                zero_indices
            ]
            - reference_zero
        ).abs().max().item()

    else:
        mean_zero_spread = np.nan


    print(
        "Dispersion embeddings mean, degré > 0 :",
        mean_positive_spread,
    )

    print(
        "Dispersion embeddings mean, degré = 0 :",
        mean_zero_spread,
    )

    print(
        "Collapse positif <= 1e-6 :",
        mean_positive_spread <= 1e-6,
    )

Dispersion embeddings mean, degré > 0 : 0.0
Dispersion embeddings mean, degré = 0 : 0.0
Collapse positif <= 1e-6 : True


In [41]:
# Tester la sensibilité de sum au degré

if (
    sage_forward_sum_ok
    and len(
        torch.unique(
            positive_degree
        )
    ) > 1
):

    unique_degrees = torch.unique(
        positive_degree
    )

    print(
        "Embedding moyen par degré entrant :"
    )

    degree_embeddings = {}

    for degree in unique_degrees:

        degree_mask = (
            in_degree == degree
        )

        degree_embedding = h_sum[
            degree_mask
        ].mean(
            dim=0
        )

        degree_embeddings[
            int(degree)
        ] = degree_embedding

        print(
            f"degré {int(degree):3d} :",
            degree_embedding.tolist(),
        )


    degree_vectors = torch.stack(
        list(
            degree_embeddings.values()
        )
    )

    sum_degree_sensitive = (
        degree_vectors.max(dim=0).values
        - degree_vectors.min(dim=0).values
    ).abs().max().item() > 1e-6


    print()
    print(
        "SAGE sum sensible au degré :",
        sum_degree_sensitive,
    )

else:
    print(
        "Test sum par degré non conclusif : "
        "pas assez de degrés positifs distincts "
        "ou forward indisponible."
    )

Embedding moyen par degré entrant :
degré   1 : [1.002593994140625, 0.09961140155792236, 1.1074419021606445, 0.756375253200531]
degré   2 : [1.1584447622299194, 0.9075747132301331, 1.216761589050293, 0.44100192189216614]
degré   3 : [1.3142956495285034, 1.7155381441116333, 1.3260810375213623, 0.12562882900238037]
degré   4 : [1.4701462984085083, 2.5235018730163574, 1.435401201248169, -0.18974438309669495]
degré   5 : [1.6259974241256714, 3.331465005874634, 1.5447205305099487, -0.5051174759864807]
degré   6 : [1.781847596168518, 4.13942813873291, 1.6540403366088867, -0.8204907774925232]
degré   7 : [1.937698483467102, 4.947391986846924, 1.7633599042892456, -1.135864019393921]
degré   8 : [2.0935492515563965, 5.755355358123779, 1.8726797103881836, -1.4512370824813843]
degré   9 : [2.2494001388549805, 6.563319206237793, 1.981999397277832, -1.7666102647781372]
degré  10 : [2.4052507877349854, 7.37128210067749, 2.0913190841674805, -2.0819833278656006]
degré  11 : [2.5611016750335693, 8.1792

In [42]:
# Vérifier la mémoire

gc.collect()

process_memory = psutil.Process(
    os.getpid()
).memory_info()

system_memory = psutil.virtual_memory()


print(
    f"RAM kernel Python : "
    f"{process_memory.rss / 1024**3:.2f} GB"
)

print(
    f"RAM système disponible : "
    f"{system_memory.available / 1024**3:.2f} GB"
)

print()

print(
    "GraphSAGE direct disponible :",
    (
        sage_forward_mean_ok
        and sage_forward_sum_ok
    ),
)

print(
    "Neighbor sampling PyG disponible :",
    link_sampling_works,
)

RAM kernel Python : 0.33 GB
RAM système disponible : 2.62 GB

GraphSAGE direct disponible : True
Neighbor sampling PyG disponible : False


### GraphSAGE aggregation sanity check

L'utilisation d'un état initial partagé entre les nœuds impose de vérifier que l'opérateur de message passing conserve effectivement un signal topologique.

Le micro-test causal montre que l'agrégation `mean`, utilisée par défaut par GraphSAGE, produit exactement la même information agrégée pour tous les nœuds possédant au moins un voisin entrant. Dans le contexte étudié, les degrés entrants positifs varient pourtant de 1 à 32, mais leur représentation agrégée reste identique. Les embeddings produits par une couche `SAGEConv(mean)` présentent ainsi une dispersion nulle entre les nœuds de degré positif.

Ce comportement est particulièrement limitant pour PaySim, dont la structure est principalement composée d'étoiles convergentes vers des destinations récurrentes. L'information liée au nombre de transactions historiques reçues constitue donc une part importante du signal structurel disponible.

À l'inverse, l'agrégation `sum` conserve exactement la variation du degré entrant avec l'état initial partagé. Le micro-test `SAGEConv(sum)` confirme que les représentations obtenues varient avec le nombre de voisins historiques.

Le baseline retient par conséquent une architecture GraphSAGE dirigée avec `aggr="sum"` et un flux `source_to_target`. Ce choix ne constitue pas une recherche supplémentaire d'hyperparamètres, mais une correction méthodologique nécessaire afin d'éviter un effondrement du signal topologique dans la représentation particulière de PaySim.

## 6. Memory-Efficient Causal Adjacency State

L'absence de backend PyG pour le neighbor sampling nécessite une méthode alternative d'accès aux voisinages historiques.

Le graphe étant dirigé et le message passing retenu utilisant le flux `source_to_target`, seules les listes de voisins entrants sont nécessaires pour le premier baseline.

Un état d'adjacence causal compact est représenté par deux arrays `int32` :

- `head_event[node]` contient l'événement entrant historique le plus récent du nœud ;
- `prev_in_event[event]` contient l'événement entrant précédent de la même destination.

Les événements sont ajoutés à cette structure uniquement après la prédiction du step courant. Cette représentation permet donc de récupérer les voisinages historiques sans construire d'adjacence dense, d'objet NetworkX ou d'`edge_index` global.

In [43]:
# Estimer la mémoire de l'adjacence

NUM_NODES = num_nodes
NUM_EVENTS = num_events

head_memory_mb = (
    NUM_NODES
    * np.dtype(np.int32).itemsize
    / 1024**2
)

prev_memory_mb = (
    NUM_EVENTS
    * np.dtype(np.int32).itemsize
    / 1024**2
)

total_adjacency_mb = (
    head_memory_mb
    + prev_memory_mb
)


print(
    f"head_event : {head_memory_mb:.2f} MB"
)

print(
    f"prev_in_event : {prev_memory_mb:.2f} MB"
)

print(
    f"Total estimé : {total_adjacency_mb:.2f} MB"
)


memory = psutil.virtual_memory()

print(
    f"RAM disponible : "
    f"{memory.available / 1024**3:.2f} GB"
)

head_event : 34.61 MB
prev_in_event : 24.27 MB
Total estimé : 58.89 MB
RAM disponible : 3.15 GB


In [44]:
# Libérer les objets temporaires

for name in [
    "sage_mean",
    "sage_sum",
    "h_mean",
    "h_sum",
    "shared_state",
    "x_shared",
    "source_messages",
    "sum_aggregation",
    "mean_aggregation",
    "sampled_batch",
    "link_loader",
    "context_pairs",
    "query_pairs",
    "pair_overlap",
]:
    if name in globals():
        del globals()[name]

gc.collect()


memory = psutil.virtual_memory()

print(
    f"RAM disponible après nettoyage : "
    f"{memory.available / 1024**3:.2f} GB"
)

RAM disponible après nettoyage : 3.22 GB


In [45]:
# Initialiser l'adjacence causale

head_event = np.full(
    NUM_NODES,
    -1,
    dtype=np.int32,
)

prev_in_event = np.full(
    NUM_EVENTS,
    -1,
    dtype=np.int32,
)


print(
    "head_event :",
    head_event.shape,
    head_event.dtype,
)

print(
    "prev_in_event :",
    prev_in_event.shape,
    prev_in_event.dtype,
)

print(
    "État initial vide :",
    bool(
        np.all(head_event == -1)
        and np.all(prev_in_event == -1)
    ),
)


memory = psutil.virtual_memory()

print(
    f"RAM disponible : "
    f"{memory.available / 1024**3:.2f} GB"
)

head_event : (9073900,) int32
prev_in_event : (6362620,) int32
État initial vide : True
RAM disponible : 3.20 GB


In [46]:
# Intégrer des événements historiques

def integrate_event_range(
    start_index,
    end_index,
    head,
    previous,
):
    dst_values = np.asarray(
        dst[start_index:end_index],
        dtype=np.int32,
    )

    for offset, destination in enumerate(
        dst_values
    ):
        event_index = start_index + offset

        previous[event_index] = head[
            destination
        ]

        head[destination] = event_index

In [47]:
# Réinitialiser l'état

head_event.fill(-1)
prev_in_event.fill(-1)


step_1_start, step_1_end = get_step_bounds(
    1
)


# Intégrer le step 1

integrate_event_range(
    step_1_start,
    step_1_end,
    head_event,
    prev_in_event,
)


print(
    "Événements intégrés :",
    step_1_end - step_1_start,
)

print(
    "Contexte attendu avant step 2 :",
    query_start,
)

print(
    "Contexte correct :",
    (
        step_1_end
        == query_start
    ),
)

Événements intégrés : 2708
Contexte attendu avant step 2 : 2708
Contexte correct : True


In [48]:
# Extraire les événements entrants

def get_inbound_events(
    node_id,
    head,
    previous,
    max_neighbors=None,
):
    events = []

    event_index = int(
        head[node_id]
    )

    while event_index != -1:

        events.append(
            event_index
        )

        if (
            max_neighbors is not None
            and len(events) >= max_neighbors
        ):
            break

        event_index = int(
            previous[event_index]
        )

    return np.asarray(
        events,
        dtype=np.int32,
    )

In [49]:
# Construire la référence du micro-contexte

unique_destinations, destination_counts = (
    np.unique(
        context_dst_global,
        return_counts=True,
    )
)

reference_counts = dict(
    zip(
        unique_destinations.tolist(),
        destination_counts.tolist(),
    )
)


# Comparer aux listes chaînées

linked_counts = []

causal_neighbors_ok = True
destination_match_ok = True


for destination in query_dst_global:

    inbound_events = get_inbound_events(
        int(destination),
        head_event,
        prev_in_event,
    )

    linked_counts.append(
        len(inbound_events)
    )

    if len(inbound_events) > 0:

        if not np.all(
            inbound_events < query_start
        ):
            causal_neighbors_ok = False

        if not np.all(
            dst[inbound_events]
            == destination
        ):
            destination_match_ok = False


reference_query_counts = np.asarray(
    [
        reference_counts.get(
            int(destination),
            0,
        )
        for destination
        in query_dst_global
    ],
    dtype=np.int32,
)

linked_counts = np.asarray(
    linked_counts,
    dtype=np.int32,
)


print(
    "Comptages exacts :",
    np.array_equal(
        linked_counts,
        reference_query_counts,
    ),
)

print(
    "Tous les voisins sont historiques :",
    causal_neighbors_ok,
)

print(
    "Destination des voisins correcte :",
    destination_match_ok,
)

print()

print(
    "Queries destination sans historique :",
    int(
        (linked_counts == 0).sum()
    ),
)

print(
    "Queries destination avec historique :",
    int(
        (linked_counts > 0).sum()
    ),
)

print(
    "Maximum voisins récupérés :",
    int(
        linked_counts.max()
    ),
)

print(
    "Moyenne voisins récupérés :",
    float(
        linked_counts.mean()
    ),
)

Comptages exacts : True
Tous les voisins sont historiques : True
Destination des voisins correcte : True

Queries destination sans historique : 305
Queries destination avec historique : 207
Maximum voisins récupérés : 32
Moyenne voisins récupérés : 3.748046875


In [50]:
# Afficher quelques voisinages connus

known_positions = np.where(
    linked_counts > 0
)[0][:5]


for position in known_positions:

    destination = int(
        query_dst_global[position]
    )

    inbound_events = get_inbound_events(
        destination,
        head_event,
        prev_in_event,
    )

    print(
        f"Query {int(position):3d} | "
        f"destination={destination} | "
        f"voisins={len(inbound_events)} | "
        f"events={inbound_events[:10].tolist()}"
    )

Query  13 | destination=6353619 | voisins=6 | events=[2509, 2246, 1730, 1306, 1179, 518]
Query  17 | destination=6353577 | voisins=13 | events=[2689, 2632, 2585, 2507, 2460, 2256, 2179, 1898, 1816, 1074]
Query  32 | destination=6353588 | voisins=1 | events=[418]
Query  33 | destination=6353403 | voisins=8 | events=[2694, 2678, 1856, 1071, 453, 422, 384, 107]
Query  34 | destination=6353537 | voisins=14 | events=[2284, 2245, 1833, 1711, 1349, 1312, 1275, 1162, 890, 877]


In [51]:
# Valider l'adjacence causale

gc.collect()

process_memory = psutil.Process(
    os.getpid()
).memory_info()

system_memory = psutil.virtual_memory()


adjacency_valid = (
    np.array_equal(
        linked_counts,
        reference_query_counts,
    )
    and causal_neighbors_ok
    and destination_match_ok
)


print(
    "Adjacence causale valide :",
    adjacency_valid,
)

print(
    f"RAM kernel Python : "
    f"{process_memory.rss / 1024**3:.2f} GB"
)

print(
    f"RAM système disponible : "
    f"{system_memory.available / 1024**3:.2f} GB"
)


if adjacency_valid:
    print(
        "SECTION 6 : accès causal aux voisinages validé."
    )
else:
    print(
        "SECTION 6 : anomalie à analyser."
    )

Adjacence causale valide : True
RAM kernel Python : 0.39 GB
RAM système disponible : 2.99 GB
SECTION 6 : accès causal aux voisinages validé.


### Causal adjacency state validation

L'accès aux voisinages historiques est représenté par une structure compacte de listes chaînées basée sur deux arrays `int32`.

`head_event` conserve, pour chaque nœud, l'identifiant du dernier événement entrant actuellement intégré au graphe causal. `prev_in_event` relie chaque événement à l'événement entrant précédent de la même destination.

Cette représentation nécessite environ 58,9 MB pour l'ensemble des 9 073 900 nœuds et 6 362 620 événements, ce qui reste compatible avec les contraintes mémoire de l'environnement.

Le micro-test sur le contexte `G_<2` confirme l'exactitude de cette structure. Les nombres de voisins historiques retrouvés correspondent exactement aux comptages directs du contexte, tous les événements récupérés sont strictement antérieurs au step à prédire et leurs destinations correspondent aux nœuds interrogés.

L'état d'adjacence permet ainsi une récupération causale et dirigée des voisinages sans dépendre des backends de neighbor sampling optionnels de PyTorch Geometric et sans construire d'objet graphe global.

## 7. Causal Local Subgraph Extraction — 1-Hop vs 2-Hops

Cette section construit des sous-graphes locaux autour d'un petit batch de transactions à prédire.

Les voisinages sont extraits exclusivement depuis l'état causal `G_<t`. Pour une architecture GraphSAGE dirigée avec un flux `source_to_target`, l'expansion suit récursivement les arêtes entrantes des endpoints des transactions query.

Une profondeur de 1-hop et une profondeur de 2-hops sont comparées afin de vérifier la faisabilité technique des deux architectures prévues, sans construire de graphe global ni réaliser d'entraînement supervisé.

In [52]:
# Extraire un sous-graphe causal

def extract_causal_subgraph(
    query_src_nodes,
    query_dst_nodes,
    num_hops,
    max_neighbors=None,
):
    query_src_nodes = np.asarray(
        query_src_nodes,
        dtype=np.int32,
    )

    query_dst_nodes = np.asarray(
        query_dst_nodes,
        dtype=np.int32,
    )

    query_nodes = np.unique(
        np.concatenate([
            query_src_nodes,
            query_dst_nodes,
        ])
    )

    frontier = set(
        query_nodes.tolist()
    )

    expanded_nodes = set()
    context_event_ids = set()


    # Explorer les voisinages entrants

    for _ in range(num_hops):

        next_frontier = set()

        for node_id in frontier:

            if node_id in expanded_nodes:
                continue

            expanded_nodes.add(
                node_id
            )

            inbound_events = get_inbound_events(
                node_id,
                head_event,
                prev_in_event,
                max_neighbors=max_neighbors,
            )

            for event_id in inbound_events:

                event_id = int(
                    event_id
                )

                context_event_ids.add(
                    event_id
                )

                source_node = int(
                    src[event_id]
                )

                next_frontier.add(
                    source_node
                )

        frontier = next_frontier

        if not frontier:
            break


    # Trier les événements historiques

    context_event_ids = np.asarray(
        sorted(context_event_ids),
        dtype=np.int32,
    )


    # Lire les endpoints historiques

    if len(context_event_ids) > 0:

        context_src = np.asarray(
            src[context_event_ids],
            dtype=np.int32,
        )

        context_dst = np.asarray(
            dst[context_event_ids],
            dtype=np.int32,
        )

        local_nodes = np.unique(
            np.concatenate([
                query_src_nodes,
                query_dst_nodes,
                context_src,
                context_dst,
            ])
        )

    else:

        context_src = np.empty(
            0,
            dtype=np.int32,
        )

        context_dst = np.empty(
            0,
            dtype=np.int32,
        )

        local_nodes = query_nodes


    # Remapper localement

    context_src_local = np.searchsorted(
        local_nodes,
        context_src,
    )

    context_dst_local = np.searchsorted(
        local_nodes,
        context_dst,
    )

    query_src_local = np.searchsorted(
        local_nodes,
        query_src_nodes,
    )

    query_dst_local = np.searchsorted(
        local_nodes,
        query_dst_nodes,
    )


    context_edge_index = torch.tensor(
        np.stack([
            context_src_local,
            context_dst_local,
        ]),
        dtype=torch.long,
    )

    query_edge_index = torch.tensor(
        np.stack([
            query_src_local,
            query_dst_local,
        ]),
        dtype=torch.long,
    )


    return {
        "local_nodes": local_nodes,
        "context_event_ids": context_event_ids,
        "context_edge_index": context_edge_index,
        "query_edge_index": query_edge_index,
    }

In [53]:
# Définir le batch du micro-test

LOCAL_QUERY_BATCH = 128

local_query_src = query_src_global[
    :LOCAL_QUERY_BATCH
]

local_query_dst = query_dst_global[
    :LOCAL_QUERY_BATCH
]


# Extraire 1-hop

subgraph_1hop = extract_causal_subgraph(
    local_query_src,
    local_query_dst,
    num_hops=1,
)


# Extraire 2-hops

subgraph_2hop = extract_causal_subgraph(
    local_query_src,
    local_query_dst,
    num_hops=2,
)


print("1-hop")
print(
    "  Nœuds :",
    len(subgraph_1hop["local_nodes"]),
)
print(
    "  Arêtes historiques :",
    len(subgraph_1hop["context_event_ids"]),
)
print(
    "  Queries :",
    LOCAL_QUERY_BATCH,
)

print()

print("2-hops")
print(
    "  Nœuds :",
    len(subgraph_2hop["local_nodes"]),
)
print(
    "  Arêtes historiques :",
    len(subgraph_2hop["context_event_ids"]),
)
print(
    "  Queries :",
    LOCAL_QUERY_BATCH,
)

1-hop
  Nœuds : 678
  Arêtes historiques : 439
  Queries : 128

2-hops
  Nœuds : 678
  Arêtes historiques : 439
  Queries : 128


In [54]:
# Vérifier les sous-graphes

def validate_local_subgraph(
    subgraph,
    query_step,
    query_start_index,
    query_src_nodes,
    query_dst_nodes,
):
    event_ids = subgraph[
        "context_event_ids"
    ]

    if len(event_ids) > 0:

        events_before_query = bool(
            np.all(
                event_ids
                < query_start_index
            )
        )

        steps_before_query = bool(
            np.all(
                step[event_ids]
                < query_step
            )
        )

        context_pairs = set(
            zip(
                src[event_ids].tolist(),
                dst[event_ids].tolist(),
            )
        )

    else:

        events_before_query = True
        steps_before_query = True
        context_pairs = set()


    query_pairs = set(
        zip(
            query_src_nodes.tolist(),
            query_dst_nodes.tolist(),
        )
    )

    no_query_edge_leakage = (
        len(
            context_pairs
            & query_pairs
        )
        == 0
    )


    return {
        "events_before_query": events_before_query,
        "steps_before_query": steps_before_query,
        "no_query_edge_leakage": no_query_edge_leakage,
    }


validation_1hop = validate_local_subgraph(
    subgraph_1hop,
    MICRO_STEP,
    query_start,
    local_query_src,
    local_query_dst,
)

validation_2hop = validate_local_subgraph(
    subgraph_2hop,
    MICRO_STEP,
    query_start,
    local_query_src,
    local_query_dst,
)


print(
    "Validation 1-hop :",
    validation_1hop,
)

print(
    "Validation 2-hops :",
    validation_2hop,
)

print()

print(
    "Arêtes 1-hop incluses dans 2-hops :",
    set(
        subgraph_1hop[
            "context_event_ids"
        ].tolist()
    ).issubset(
        set(
            subgraph_2hop[
                "context_event_ids"
            ].tolist()
        )
    ),
)

Validation 1-hop : {'events_before_query': True, 'steps_before_query': True, 'no_query_edge_leakage': True}
Validation 2-hops : {'events_before_query': True, 'steps_before_query': True, 'no_query_edge_leakage': True}

Arêtes 1-hop incluses dans 2-hops : True


In [55]:
# Vérifier les index locaux

for name, subgraph in [
    ("1-hop", subgraph_1hop),
    ("2-hops", subgraph_2hop),
]:

    num_local_nodes = len(
        subgraph["local_nodes"]
    )

    context_index = subgraph[
        "context_edge_index"
    ]

    query_index = subgraph[
        "query_edge_index"
    ]


    context_valid = (
        context_index.numel() == 0
        or (
            int(context_index.min()) >= 0
            and int(context_index.max())
            < num_local_nodes
        )
    )

    query_valid = (
        int(query_index.min()) >= 0
        and int(query_index.max())
        < num_local_nodes
    )


    print(name)

    print(
        "  context edge_index :",
        tuple(context_index.shape),
    )

    print(
        "  query edge_index :",
        tuple(query_index.shape),
    )

    print(
        "  Mapping contexte valide :",
        context_valid,
    )

    print(
        "  Mapping query valide :",
        query_valid,
    )

1-hop
  context edge_index : (2, 439)
  query edge_index : (2, 128)
  Mapping contexte valide : True
  Mapping query valide : True
2-hops
  context edge_index : (2, 439)
  query edge_index : (2, 128)
  Mapping contexte valide : True
  Mapping query valide : True


In [57]:
# Tester GraphSAGE 1 couche

torch.manual_seed(
    RANDOM_STATE
)


num_nodes_1hop = len(
    subgraph_1hop["local_nodes"]
)


x_1hop = torch.ones(
    (
        num_nodes_1hop,
        1,
    ),
    dtype=torch.float32,
)


sage_1hop = SAGEConv(
    in_channels=1,
    out_channels=8,
    aggr="sum",
    normalize=False,
)


with torch.no_grad():

    h_1hop = torch.relu(
        sage_1hop(
            x_1hop,
            subgraph_1hop[
                "context_edge_index"
            ],
        )
    )


query_index_1hop = subgraph_1hop[
    "query_edge_index"
]


query_repr_1hop = torch.cat(
    [
        h_1hop[
            query_index_1hop[0]
        ],
        h_1hop[
            query_index_1hop[1]
        ],
    ],
    dim=1,
)


print(
    "Node embeddings 1-hop :",
    tuple(h_1hop.shape),
)

print(
    "Query representations 1-hop :",
    tuple(query_repr_1hop.shape),
)

print(
    "NaN :",
    bool(
        torch.isnan(
            query_repr_1hop
        ).any()
    ),
)

Node embeddings 1-hop : (678, 8)
Query representations 1-hop : (128, 16)
NaN : False


In [58]:
# Tester GraphSAGE 2 couches

torch.manual_seed(
    RANDOM_STATE
)


num_nodes_2hop = len(
    subgraph_2hop["local_nodes"]
)


x_2hop = torch.ones(
    (
        num_nodes_2hop,
        1,
    ),
    dtype=torch.float32,
)


sage_2hop_1 = SAGEConv(
    in_channels=1,
    out_channels=8,
    aggr="sum",
    normalize=False,
)

sage_2hop_2 = SAGEConv(
    in_channels=8,
    out_channels=8,
    aggr="sum",
    normalize=False,
)


with torch.no_grad():

    h_2hop_1 = torch.relu(
        sage_2hop_1(
            x_2hop,
            subgraph_2hop[
                "context_edge_index"
            ],
        )
    )

    h_2hop = torch.relu(
        sage_2hop_2(
            h_2hop_1,
            subgraph_2hop[
                "context_edge_index"
            ],
        )
    )


query_index_2hop = subgraph_2hop[
    "query_edge_index"
]


query_repr_2hop = torch.cat(
    [
        h_2hop[
            query_index_2hop[0]
        ],
        h_2hop[
            query_index_2hop[1]
        ],
    ],
    dim=1,
)


print(
    "Node embeddings 2-hops :",
    tuple(h_2hop.shape),
)

print(
    "Query representations 2-hops :",
    tuple(query_repr_2hop.shape),
)

print(
    "NaN :",
    bool(
        torch.isnan(
            query_repr_2hop
        ).any()
    ),
)

Node embeddings 2-hops : (678, 8)
Query representations 2-hops : (128, 16)
NaN : False


In [59]:
# Vérifier la section

gc.collect()

process_memory = psutil.Process(
    os.getpid()
).memory_info()

system_memory = psutil.virtual_memory()


subgraph_1hop_valid = all(
    validation_1hop.values()
)

subgraph_2hop_valid = all(
    validation_2hop.values()
)

sage_local_valid = (
    not torch.isnan(
        query_repr_1hop
    ).any()
    and not torch.isnan(
        query_repr_2hop
    ).any()
)


print(
    "Sous-graphe 1-hop valide :",
    subgraph_1hop_valid,
)

print(
    "Sous-graphe 2-hops valide :",
    subgraph_2hop_valid,
)

print(
    "Forward GraphSAGE local valide :",
    bool(sage_local_valid),
)

print()

print(
    f"RAM kernel Python : "
    f"{process_memory.rss / 1024**3:.2f} GB"
)

print(
    f"RAM système disponible : "
    f"{system_memory.available / 1024**3:.2f} GB"
)


if (
    subgraph_1hop_valid
    and subgraph_2hop_valid
    and sage_local_valid
):
    print(
        "SECTION 7 : sous-graphes locaux causaux validés."
    )
else:
    print(
        "SECTION 7 : anomalie à analyser."
    )

Sous-graphe 1-hop valide : True
Sous-graphe 2-hops valide : True
Forward GraphSAGE local valide : True

RAM kernel Python : 0.38 GB
RAM système disponible : 3.30 GB
SECTION 7 : sous-graphes locaux causaux validés.


### Local causal subgraph validation

L'extraction de sous-graphes causaux locaux a été validée sur un batch de 128 transactions du step 2.

Les sous-graphes 1-hop et 2-hops contiennent tous deux 678 nœuds et 439 arêtes historiques. Toutes les arêtes utilisées appartiennent strictement au contexte antérieur au step courant, et aucune transaction query n'est présente dans le graphe de message passing.

Sur ce micro-batch, l'expansion à 2 hops n'ajoute aucun nœud ni aucune arête par rapport au voisinage 1-hop. Ce comportement est cohérent avec la topologie de PaySim, dans laquelle les sources sont presque toujours des nœuds sans historique entrant et très peu de comptes apparaissent dans les deux rôles.

Les deux profondeurs restent néanmoins techniquement compatibles avec `SAGEConv(sum)`. Les forwards 1 couche et 2 couches produisent correctement des représentations des transactions de dimension 16, sans valeur NaN.

La comparaison 1 couche contre 2 couches sera conservée comme ablation contrôlée, mais un éventuel bénéfice de la deuxième couche devra être distingué d'un véritable gain de portée topologique.

In [1]:
import sys
import platform
import torch

print("Python :", sys.version.split()[0])
print("Exécutable :", sys.executable)
print("Plateforme :", platform.platform())

print()
print("PyTorch :", torch.__version__)
print("CUDA disponible :", torch.cuda.is_available())
print("CUDA PyTorch :", torch.version.cuda)

if torch.cuda.is_available():
    print("GPU :", torch.cuda.get_device_name(0))

    props = torch.cuda.get_device_properties(0)

    print(
        f"VRAM totale : "
        f"{props.total_memory / 1024**3:.2f} GB"
    )

Python : 3.13.15
Exécutable : /usr/bin/python3
Plateforme : Linux-6.6.122+-x86_64-with-glibc2.35

PyTorch : 2.11.0+cpu
CUDA disponible : False
CUDA PyTorch : None


In [2]:
!nvidia-smi

/bin/bash: line 1: nvidia-smi: command not found


In [1]:
# Vérifier l'environnement de reprise

import sys
import platform
import os
import psutil
import numpy as np
import torch

print(
    "Python :",
    sys.version.split()[0],
)

print(
    "Executable :",
    sys.executable,
)

print(
    "Platform :",
    platform.platform(),
)

print()

print(
    "PyTorch :",
    torch.__version__,
)

print(
    "CUDA disponible :",
    torch.cuda.is_available(),
)

print(
    "Device retenu :",
    "cuda"
    if torch.cuda.is_available()
    else "cpu",
)

print()

memory = psutil.virtual_memory()

print(
    f"RAM totale : "
    f"{memory.total / 1024**3:.2f} GB"
)

print(
    f"RAM disponible : "
    f"{memory.available / 1024**3:.2f} GB"
)

print(
    "Répertoire courant :",
    os.getcwd(),
)

Python : 3.13.7
Executable : c:\Users\bouza\AppData\Local\Programs\Python\Python313\python.exe
Platform : Windows-11-10.0.22631-SP0

PyTorch : 2.12.0+cpu
CUDA disponible : False
Device retenu : cpu

RAM totale : 15.84 GB
RAM disponible : 2.60 GB
Répertoire courant : c:\Users\bouza\OneDrive\Bureau\MobileMoney-Fraud-GNN\notebooks


In [2]:
# Vérifier les artefacts locaux

from pathlib import Path

candidate_roots = [
    Path("../data"),
    Path("./data"),
]


for root in candidate_roots:

    graph_dir = root / "graph"
    processed_dir = root / "processed"

    print(
        "\nRoot :",
        root.resolve(),
    )

    print(
        "src :",
        (
            graph_dir
            / "paysim_src.npy"
        ).exists(),
    )

    print(
        "dst :",
        (
            graph_dir
            / "paysim_dst.npy"
        ).exists(),
    )

    print(
        "step :",
        (
            graph_dir
            / "paysim_step.npy"
        ).exists(),
    )

    print(
        "label :",
        (
            graph_dir
            / "paysim_label.npy"
        ).exists(),
    )

    print(
        "offsets :",
        (
            graph_dir
            / "paysim_step_offsets.npy"
        ).exists(),
    )

    print(
        "Graph-ready :",
        (
            processed_dir
            / "paysim_graph_ready.parquet"
        ).exists(),
    )


Root : C:\Users\bouza\OneDrive\Bureau\MobileMoney-Fraud-GNN\data
src : True
dst : True
step : True
label : True
offsets : True
Graph-ready : True

Root : C:\Users\bouza\OneDrive\Bureau\MobileMoney-Fraud-GNN\notebooks\data
src : False
dst : False
step : False
label : False
offsets : False
Graph-ready : False


In [3]:
# Réhydrater la session locale

import gc
import json
import time
from pathlib import Path

import numpy as np
import pyarrow as pa
import pyarrow.parquet as pq
import torch


ROOT = Path("..")

GRAPH_DIR = ROOT / "data" / "graph"
PROCESSED_DIR = ROOT / "data" / "processed"


src = np.load(
    GRAPH_DIR / "paysim_src.npy",
    mmap_mode="r",
)

dst = np.load(
    GRAPH_DIR / "paysim_dst.npy",
    mmap_mode="r",
)

step = np.load(
    GRAPH_DIR / "paysim_step.npy",
    mmap_mode="r",
)

label = np.load(
    GRAPH_DIR / "paysim_label.npy",
    mmap_mode="r",
)

step_offsets = np.load(
    GRAPH_DIR / "paysim_step_offsets.npy",
    mmap_mode="r",
)


with open(
    GRAPH_DIR / "graph_construction_metadata.json",
    "r",
    encoding="utf-8",
) as file:
    graph_metadata = json.load(file)


graph_ready_parquet = pq.ParquetFile(
    PROCESSED_DIR / "paysim_graph_ready.parquet"
)

graph_ready_schema = (
    graph_ready_parquet.schema_arrow
)


num_events = len(src)
num_nodes = 9_073_900

TRAIN_START = 1
TRAIN_END = 334

VAL_START = 335
VAL_END = 392

TEST_START = 393
TEST_END = 743


def get_step_bounds(t):
    return (
        int(step_offsets[t - 1]),
        int(step_offsets[t]),
    )


TRAIN_ROWS = int(
    step_offsets[TRAIN_END]
)


device = torch.device("cpu")


print(
    "Device :",
    device,
)

print(
    "Événements :",
    f"{num_events:,}",
)

print(
    "Train :",
    f"{TRAIN_ROWS:,}",
)

print(
    "Fraudes Train :",
    int(
        label[:TRAIN_ROWS].sum(
            dtype=np.int64
        )
    ),
)

print(
    "Arrays memmap :",
    all(
        isinstance(array, np.memmap)
        for array in [
            src,
            dst,
            step,
            label,
            step_offsets,
        ]
    ),
)

Device : cpu
Événements : 6,362,620
Train : 4,771,387
Fraudes Train : 3745
Arrays memmap : True


In [4]:
# Restaurer le preprocessing

core_edge_features = (
    graph_metadata[
        "feature_sets"
    ][
        "graph_core_edge"
    ]
)


continuous_features = [
    "amount",
    "log_amount",
    "origin_balance_change",
    "destination_balance_change",
]

binary_features = [
    "origin_zero_before",
    "origin_zero_after",
    "destination_zero_before",
    "destination_zero_after",
]

type_categories = [
    "CASH_IN",
    "CASH_OUT",
    "DEBIT",
    "PAYMENT",
    "TRANSFER",
]

type_to_index = {
    category: index
    for index, category
    in enumerate(type_categories)
}


train_scaler = {
    "amount": {
        "mean": 179748.531939,
        "std": 648694.425604,
    },
    "log_amount": {
        "mean": 10.833058,
        "std": 1.820169,
    },
    "origin_balance_change": {
        "mean": -21731.548861,
        "std": 128793.638313,
    },
    "destination_balance_change": {
        "mean": 130154.927112,
        "std": 900249.394962,
    },
}


core_edge_input_dim = (
    len(continuous_features)
    + len(binary_features)
    + len(type_categories)
)


print(
    "Graph Core Edge :",
    len(core_edge_features),
)

print(
    "Dimension :",
    core_edge_input_dim,
)

Graph Core Edge : 9
Dimension : 13


In [5]:
# Préparer les row groups

num_row_groups = (
    graph_ready_parquet.metadata.num_row_groups
)

row_group_offsets = np.zeros(
    num_row_groups + 1,
    dtype=np.int64,
)


for row_group_id in range(
    num_row_groups
):
    row_group_offsets[
        row_group_id + 1
    ] = (
        row_group_offsets[row_group_id]
        + graph_ready_parquet.metadata
        .row_group(row_group_id)
        .num_rows
    )


def read_graph_ready_range(
    start_index,
    end_index,
    columns,
):
    first_group = int(
        np.searchsorted(
            row_group_offsets,
            start_index,
            side="right",
        )
        - 1
    )

    last_group = int(
        np.searchsorted(
            row_group_offsets,
            end_index - 1,
            side="right",
        )
        - 1
    )

    tables = []


    for row_group_id in range(
        first_group,
        last_group + 1,
    ):
        group_start = int(
            row_group_offsets[
                row_group_id
            ]
        )

        group_end = int(
            row_group_offsets[
                row_group_id + 1
            ]
        )

        local_start = (
            max(
                start_index,
                group_start,
            )
            - group_start
        )

        local_end = (
            min(
                end_index,
                group_end,
            )
            - group_start
        )

        table = (
            graph_ready_parquet
            .read_row_group(
                row_group_id,
                columns=columns,
                use_threads=True,
            )
        )

        tables.append(
            table.slice(
                local_start,
                local_end - local_start,
            )
        )


    if len(tables) == 1:
        return tables[0]

    return pa.concat_tables(
        tables
    )

In [6]:
# Transformer les features

def transform_core_edge_batch(
    batch,
):
    parts = []


    for feature in continuous_features:

        values = (
            batch[feature]
            .combine_chunks()
            .to_numpy(
                zero_copy_only=False
            )
            .astype(
                np.float32,
                copy=False,
            )
        )

        mean = train_scaler[
            feature
        ]["mean"]

        std = train_scaler[
            feature
        ]["std"]

        parts.append(
            (
                (values - mean)
                / std
            ).reshape(-1, 1)
        )


    for feature in binary_features:

        values = (
            batch[feature]
            .combine_chunks()
            .to_numpy(
                zero_copy_only=False
            )
            .astype(
                np.float32,
                copy=False,
            )
        )

        parts.append(
            values.reshape(-1, 1)
        )


    type_values = (
        batch["type"]
        .combine_chunks()
        .to_pylist()
    )


    one_hot = np.zeros(
        (
            batch.num_rows,
            len(type_categories),
        ),
        dtype=np.float32,
    )

    unknown_types = 0


    for row_index, category in enumerate(
        type_values
    ):
        category_index = (
            type_to_index.get(
                category
            )
        )

        if category_index is None:
            unknown_types += 1
            continue

        one_hot[
            row_index,
            category_index
        ] = 1.0


    parts.append(one_hot)


    features = np.concatenate(
        parts,
        axis=1,
    ).astype(
        np.float32,
        copy=False,
    )


    return (
        features,
        unknown_types,
    )

In [7]:
# Vérifier la RAM avant benchmark

import psutil

memory = psutil.virtual_memory()

print(
    f"RAM disponible : "
    f"{memory.available / 1024**3:.2f} GB"
)

print(
    f"RAM utilisée : "
    f"{memory.percent:.1f}%"
)

RAM disponible : 2.75 GB
RAM utilisée : 82.6%


In [8]:
# Initialiser l'adjacence causale

head_event = np.full(
    num_nodes,
    -1,
    dtype=np.int32,
)

prev_in_event = np.full(
    num_events,
    -1,
    dtype=np.int32,
)


def integrate_event_range(
    start_index,
    end_index,
):
    dst_values = np.asarray(
        dst[start_index:end_index],
        dtype=np.int32,
    )

    for offset, destination in enumerate(
        dst_values
    ):
        event_index = (
            start_index + offset
        )

        prev_in_event[
            event_index
        ] = head_event[
            destination
        ]

        head_event[
            destination
        ] = event_index


def get_inbound_events(
    node_id,
    max_neighbors=None,
):
    events = []

    event_index = int(
        head_event[node_id]
    )

    while event_index != -1:

        events.append(
            event_index
        )

        if (
            max_neighbors is not None
            and len(events)
            >= max_neighbors
        ):
            break

        event_index = int(
            prev_in_event[
                event_index
            ]
        )

    return np.asarray(
        events,
        dtype=np.int32,
    )


print(
    "head_event :",
    head_event.shape,
    head_event.dtype,
)

print(
    "prev_in_event :",
    prev_in_event.shape,
    prev_in_event.dtype,
)

print(
    f"Mémoire adjacency : "
    f"{(head_event.nbytes + prev_in_event.nbytes) / 1024**2:.2f} MB"
)

head_event : (9073900,) int32
prev_in_event : (6362620,) int32
Mémoire adjacency : 58.89 MB


In [9]:
# Extraire un sous-graphe causal

def extract_causal_subgraph(
    query_src_nodes,
    query_dst_nodes,
    num_hops=1,
    max_neighbors=None,
):
    query_src_nodes = np.asarray(
        query_src_nodes,
        dtype=np.int32,
    )

    query_dst_nodes = np.asarray(
        query_dst_nodes,
        dtype=np.int32,
    )


    query_nodes = np.unique(
        np.concatenate([
            query_src_nodes,
            query_dst_nodes,
        ])
    )

    frontier = set(
        query_nodes.tolist()
    )

    expanded_nodes = set()
    context_event_ids = set()


    for _ in range(num_hops):

        next_frontier = set()

        for node_id in frontier:

            if node_id in expanded_nodes:
                continue

            expanded_nodes.add(
                node_id
            )

            inbound_events = (
                get_inbound_events(
                    node_id,
                    max_neighbors=max_neighbors,
                )
            )

            for event_id in inbound_events:

                event_id = int(
                    event_id
                )

                context_event_ids.add(
                    event_id
                )

                next_frontier.add(
                    int(
                        src[event_id]
                    )
                )

        frontier = next_frontier

        if not frontier:
            break


    context_event_ids = np.asarray(
        sorted(context_event_ids),
        dtype=np.int32,
    )


    if len(context_event_ids) > 0:

        context_src = np.asarray(
            src[context_event_ids],
            dtype=np.int32,
        )

        context_dst = np.asarray(
            dst[context_event_ids],
            dtype=np.int32,
        )

        local_nodes = np.unique(
            np.concatenate([
                query_src_nodes,
                query_dst_nodes,
                context_src,
                context_dst,
            ])
        )

    else:

        context_src = np.empty(
            0,
            dtype=np.int32,
        )

        context_dst = np.empty(
            0,
            dtype=np.int32,
        )

        local_nodes = query_nodes


    context_src_local = np.searchsorted(
        local_nodes,
        context_src,
    )

    context_dst_local = np.searchsorted(
        local_nodes,
        context_dst,
    )

    query_src_local = np.searchsorted(
        local_nodes,
        query_src_nodes,
    )

    query_dst_local = np.searchsorted(
        local_nodes,
        query_dst_nodes,
    )


    return {
        "local_nodes": local_nodes,
        "context_event_ids": context_event_ids,

        "context_edge_index": torch.tensor(
            np.stack([
                context_src_local,
                context_dst_local,
            ]),
            dtype=torch.long,
        ),

        "query_edge_index": torch.tensor(
            np.stack([
                query_src_local,
                query_dst_local,
            ]),
            dtype=torch.long,
        ),
    }

In [10]:
# Définir le modèle

import torch.nn as nn
import torch.nn.functional as F

from torch_geometric.nn import SAGEConv


class GraphSAGEEdgeClassifier(
    nn.Module
):

    def __init__(
        self,
        edge_feature_dim,
        hidden_dim=32,
        num_layers=1,
        dropout=0.10,
    ):
        super().__init__()

        if num_layers not in (1, 2):
            raise ValueError(
                "num_layers doit être 1 ou 2."
            )

        self.num_layers = num_layers
        self.dropout = dropout


        self.conv1 = SAGEConv(
            in_channels=1,
            out_channels=hidden_dim,
            aggr="sum",
            normalize=False,
        )


        if num_layers == 2:

            self.conv2 = SAGEConv(
                in_channels=hidden_dim,
                out_channels=hidden_dim,
                aggr="sum",
                normalize=False,
            )


        classifier_input_dim = (
            2 * hidden_dim
            + edge_feature_dim
        )


        self.classifier = nn.Sequential(
            nn.Linear(
                classifier_input_dim,
                hidden_dim,
            ),
            nn.ReLU(),
            nn.Dropout(
                dropout
            ),
            nn.Linear(
                hidden_dim,
                1,
            ),
        )


    def forward(
        self,
        num_nodes,
        context_edge_index,
        query_edge_index,
        edge_features,
    ):
        x = torch.ones(
            (num_nodes, 1),
            dtype=torch.float32,
            device=context_edge_index.device,
        )


        h = F.relu(
            self.conv1(
                x,
                context_edge_index,
            )
        )


        if self.num_layers == 2:

            h = F.dropout(
                h,
                p=self.dropout,
                training=self.training,
            )

            h = F.relu(
                self.conv2(
                    h,
                    context_edge_index,
                )
            )


        h_src = h[
            query_edge_index[0]
        ]

        h_dst = h[
            query_edge_index[1]
        ]


        edge_repr = torch.cat(
            [
                h_src,
                h_dst,
                edge_features,
            ],
            dim=1,
        )


        return self.classifier(
            edge_repr
        ).squeeze(-1)

c:\Users\bouza\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [11]:
# Vérifier le runtime CPU

print(
    "Threads PyTorch :",
    torch.get_num_threads(),
)

print(
    "Device :",
    device,
)

Threads PyTorch : 2
Device : cpu


In [12]:
# Reconstruire G_<334

PILOT_STEP = 334

pilot_start, pilot_end = (
    get_step_bounds(
        PILOT_STEP
    )
)


head_event.fill(-1)
prev_in_event.fill(-1)

gc.collect()


replay_start_time = (
    time.perf_counter()
)


integrate_event_range(
    0,
    pilot_start,
)


replay_seconds = (
    time.perf_counter()
    - replay_start_time
)


last_history_event = int(
    head_event.max()
)


print(
    "Step pilote :",
    PILOT_STEP,
)

print(
    "Événements rejoués :",
    f"{pilot_start:,}",
)

print(
    "Transactions query :",
    f"{pilot_end - pilot_start:,}",
)

print(
    f"Temps replay CPU : "
    f"{replay_seconds:.2f} s"
)

print(
    "Historique causal valide :",
    last_history_event
    < pilot_start,
)

Step pilote : 334
Événements rejoués : 4,759,371
Transactions query : 12,016
Temps replay CPU : 5.91 s
Historique causal valide : True


In [13]:
# Charger le step pilote

pilot_src = np.asarray(
    src[
        pilot_start:pilot_end
    ],
    dtype=np.int32,
)

pilot_dst = np.asarray(
    dst[
        pilot_start:pilot_end
    ],
    dtype=np.int32,
)

pilot_labels = np.asarray(
    label[
        pilot_start:pilot_end
    ],
    dtype=np.float32,
)


pilot_table = read_graph_ready_range(
    pilot_start,
    pilot_end,
    columns=core_edge_features,
)


pilot_features, pilot_unknown = (
    transform_core_edge_batch(
        pilot_table
    )
)


print(
    "Transactions :",
    len(pilot_src),
)

print(
    "Fraudes :",
    int(
        pilot_labels.sum()
    ),
)

print(
    "Features :",
    pilot_features.shape,
    pilot_features.dtype,
)

print(
    "Types inconnus :",
    pilot_unknown,
)

Transactions : 12016
Fraudes : 4
Features : (12016, 13) float32
Types inconnus : 0


In [15]:
# Initialiser le modèle pilote

RANDOM_STATE = 42

PILOT_BATCH_SIZE = 4_096

PILOT_LR = 1e-3
PILOT_WEIGHT_DECAY = 1e-5


train_positive = int(
    label[:TRAIN_ROWS].sum(
        dtype=np.int64
    )
)

train_negative = (
    TRAIN_ROWS
    - train_positive
)

train_pos_weight = (
    train_negative
    / train_positive
)


torch.manual_seed(
    RANDOM_STATE
)


pilot_model = (
    GraphSAGEEdgeClassifier(
        edge_feature_dim=(
            core_edge_input_dim
        ),
        hidden_dim=32,
        num_layers=1,
        dropout=0.10,
    )
    .to(device)
)


pilot_optimizer = (
    torch.optim.AdamW(
        pilot_model.parameters(),
        lr=PILOT_LR,
        weight_decay=(
            PILOT_WEIGHT_DECAY
        ),
    )
)


pilot_criterion = (
    torch.nn.BCEWithLogitsLoss(
        pos_weight=torch.tensor(
            [train_pos_weight],
            dtype=torch.float32,
            device=device,
        ),
        reduction="sum",
    )
)


print(
    "Paramètres :",
    sum(
        parameter.numel()
        for parameter
        in pilot_model.parameters()
    ),
)

print(
    "Batch size :",
    PILOT_BATCH_SIZE,
)

print(
    f"pos_weight : "
    f"{train_pos_weight:.6f}"
)

Paramètres : 2625
Batch size : 4096
pos_weight : 1273.068625


In [16]:
# Entraîner un step sur CPU

def train_causal_step_cpu(
    model,
    optimizer,
    criterion,
    query_src,
    query_dst,
    query_features,
    query_labels,
    batch_size,
    num_hops=1,
):
    model.train()

    optimizer.zero_grad(
        set_to_none=True
    )


    num_queries = len(
        query_src
    )

    num_batches = int(
        np.ceil(
            num_queries
            / batch_size
        )
    )


    total_loss_sum = 0.0

    local_nodes_counts = []
    context_edge_counts = []


    process = psutil.Process()

    peak_rss_mb = (
        process.memory_info().rss
        / 1024**2
    )


    start_time = (
        time.perf_counter()
    )


    for batch_id in range(
        num_batches
    ):
        batch_start = (
            batch_id
            * batch_size
        )

        batch_end = min(
            batch_start
            + batch_size,
            num_queries,
        )


        subgraph = (
            extract_causal_subgraph(
                query_src[
                    batch_start:batch_end
                ],
                query_dst[
                    batch_start:batch_end
                ],
                num_hops=num_hops,
            )
        )


        local_nodes_counts.append(
            len(
                subgraph[
                    "local_nodes"
                ]
            )
        )

        context_edge_counts.append(
            len(
                subgraph[
                    "context_event_ids"
                ]
            )
        )


        edge_features_cpu = (
            torch.from_numpy(
                query_features[
                    batch_start:batch_end
                ]
            )
        )


        labels_cpu = torch.from_numpy(
            query_labels[
                batch_start:batch_end
            ]
        )


        logits = model(
            num_nodes=len(
                subgraph[
                    "local_nodes"
                ]
            ),
            context_edge_index=(
                subgraph[
                    "context_edge_index"
                ]
            ),
            query_edge_index=(
                subgraph[
                    "query_edge_index"
                ]
            ),
            edge_features=(
                edge_features_cpu
            ),
        )


        batch_loss_sum = criterion(
            logits,
            labels_cpu,
        )


        normalized_loss = (
            batch_loss_sum
            / num_queries
        )


        normalized_loss.backward()


        total_loss_sum += float(
            batch_loss_sum.detach()
        )


        current_rss_mb = (
            process.memory_info().rss
            / 1024**2
        )

        peak_rss_mb = max(
            peak_rss_mb,
            current_rss_mb,
        )


        del (
            subgraph,
            edge_features_cpu,
            labels_cpu,
            logits,
            batch_loss_sum,
            normalized_loss,
        )


    runtime_seconds = (
        time.perf_counter()
        - start_time
    )


    gradients_finite = True
    gradient_squared_sum = 0.0


    for parameter in (
        model.parameters()
    ):
        if parameter.grad is None:
            continue

        if not torch.isfinite(
            parameter.grad
        ).all():

            gradients_finite = False

        gradient_squared_sum += float(
            parameter.grad
            .detach()
            .pow(2)
            .sum()
        )


    gradient_norm = (
        gradient_squared_sum
        ** 0.5
    )


    if gradients_finite:
        optimizer.step()


    return {
        "num_queries": num_queries,
        "num_batches": num_batches,

        "mean_loss": (
            total_loss_sum
            / num_queries
        ),

        "gradients_finite": (
            gradients_finite
        ),

        "gradient_norm": (
            gradient_norm
        ),

        "runtime_seconds": (
            runtime_seconds
        ),

        "mean_local_nodes": float(
            np.mean(
                local_nodes_counts
            )
        ),

        "max_local_nodes": int(
            np.max(
                local_nodes_counts
            )
        ),

        "mean_context_edges": float(
            np.mean(
                context_edge_counts
            )
        ),

        "max_context_edges": int(
            np.max(
                context_edge_counts
            )
        ),

        "peak_process_ram_mb": (
            peak_rss_mb
        ),
    }

In [17]:
# Exécuter le benchmark CPU

pilot_results_cpu = (
    train_causal_step_cpu(
        model=pilot_model,
        optimizer=pilot_optimizer,
        criterion=pilot_criterion,
        query_src=pilot_src,
        query_dst=pilot_dst,
        query_features=pilot_features,
        query_labels=pilot_labels,
        batch_size=PILOT_BATCH_SIZE,
        num_hops=1,
    )
)


for key, value in (
    pilot_results_cpu.items()
):
    print(
        f"{key:24s} : {value}"
    )

num_queries              : 12016
num_batches              : 3
mean_loss                : 1.1965625016254369
gradients_finite         : True
gradient_norm            : 3.125987734412668
runtime_seconds          : 0.9994565999950282
mean_local_nodes         : 25549.666666666668
max_local_nodes          : 28042
mean_context_edges       : 17543.666666666668
max_context_edges        : 19856
peak_process_ram_mb      : 541.2265625


In [18]:
# Valider le benchmark CPU

history_before = int(
    head_event.max()
)


integration_start = (
    time.perf_counter()
)


integrate_event_range(
    pilot_start,
    pilot_end,
)


integration_seconds = (
    time.perf_counter()
    - integration_start
)


history_after = int(
    head_event.max()
)


memory = psutil.virtual_memory()


cpu_pilot_valid = (
    pilot_results_cpu[
        "gradients_finite"
    ]
    and np.isfinite(
        pilot_results_cpu[
            "mean_loss"
        ]
    )
    and history_before
        < pilot_start
    and history_after
        >= pilot_start
    and history_after
        < pilot_end
)


print(
    "Benchmark CPU valide :",
    bool(cpu_pilot_valid),
)

print(
    f"Temps intégration : "
    f"{integration_seconds:.4f} s"
)

print(
    f"RAM disponible finale : "
    f"{memory.available / 1024**3:.2f} GB"
)

print(
    f"RAM utilisée : "
    f"{memory.percent:.1f}%"
)

Benchmark CPU valide : True
Temps intégration : 0.0184 s
RAM disponible finale : 2.86 GB
RAM utilisée : 82.0%


### Local CPU training feasibility

Le pipeline causal a été benchmarké localement sur CPU afin de déterminer si l'utilisation d'un GPU est nécessaire pour l'entraînement du baseline GraphSAGE.

Le test a été réalisé sur le step 334, qui constitue l'un des cas les plus exigeants du Train avec 4 759 371 événements historiques et 12 016 transactions à prédire.

La reconstruction de `G_<334` nécessite environ 5,91 secondes sur la machine locale. Le traitement complet du step, comprenant l'extraction des sous-graphes, le forward GraphSAGE, le calcul de la loss et le backward sur trois batches, nécessite environ 1 seconde.

Les sous-graphes conservent les mêmes dimensions que lors du benchmark GPU, avec environ 25 550 nœuds et 17 544 arêtes historiques par batch en moyenne. Tous les gradients restent finis et la norme globale des gradients reste stable.

Le processus Python atteint environ 541 MB de RAM lors du test, tandis que la structure d'adjacence causale complète occupe environ 59 MB.

Sur ce baseline léger, le CPU local est suffisamment performant et ne présente pas de contrainte mémoire bloquante. L'entraînement du notebook 06 peut donc être poursuivi localement sans dépendre d'un runtime GPU externe.

## 11. Full Causal Training - GraphSAGE 1-Layer

Cette section réalise le premier entraînement complet du baseline GraphSAGE sur les données Train.

L'apprentissage suit strictement l'ordre temporel des transactions, des steps 1 à 334. Pour chaque step `t`, le message passing utilise exclusivement le graphe historique `G_<t`.

Les transactions du step courant peuvent être divisées en plusieurs batches pour limiter la consommation mémoire. Les gradients sont accumulés sur l'ensemble des batches du step et une seule mise à jour des paramètres est effectuée après traitement complet de `E_t`.

La topologie de `E_t` n'est intégrée au graphe qu'après cette mise à jour. Les labels interviennent uniquement dans la fonction de perte et ne sont jamais utilisés comme information de message passing.

Le modèle utilise les features `Graph Core Edge`, une couche GraphSAGE dirigée avec agrégation `sum`, un état initial partagé entre les nœuds et un MLP de classification.

Un seul passage chronologique sur les données Train est effectué. Aucun shuffle, aucune information de Validation ou de Test et aucune répétition temporelle du Train ne sont utilisés.

In [19]:
# Configurer l'entraînement complet

import time
import json
import gc
import psutil
from pathlib import Path
from collections import OrderedDict


FULL_BATCH_SIZE = 4_096

FULL_HIDDEN_DIM = 32
FULL_NUM_LAYERS = 1
FULL_NUM_HOPS = 1

FULL_DROPOUT = 0.10

FULL_LR = 1e-3
FULL_WEIGHT_DECAY = 1e-5

CHECKPOINT_EVERY = 25


train_positive = int(
    label[:TRAIN_ROWS].sum(
        dtype=np.int64
    )
)

train_negative = (
    TRAIN_ROWS
    - train_positive
)

train_pos_weight = (
    train_negative
    / train_positive
)


FULL_TRAIN_CONFIG = {
    "feature_set": "graph_core_edge",
    "input_dim": core_edge_input_dim,
    "hidden_dim": FULL_HIDDEN_DIM,
    "num_layers": FULL_NUM_LAYERS,
    "num_hops": FULL_NUM_HOPS,
    "aggregation": "sum",
    "dropout": FULL_DROPOUT,
    "batch_size": FULL_BATCH_SIZE,
    "learning_rate": FULL_LR,
    "weight_decay": FULL_WEIGHT_DECAY,
    "pos_weight": train_pos_weight,
    "train_start_step": TRAIN_START,
    "train_end_step": TRAIN_END,
    "random_state": RANDOM_STATE,
    "epochs": 1,
    "shuffle": False,
    "device": "cpu",
}


print(
    "Steps Train :",
    TRAIN_START,
    "->",
    TRAIN_END,
)

print(
    "Transactions Train :",
    f"{TRAIN_ROWS:,}",
)

print(
    "Fraudes Train :",
    f"{train_positive:,}",
)

print(
    f"pos_weight : "
    f"{train_pos_weight:.6f}"
)

print(
    "Batch size :",
    FULL_BATCH_SIZE,
)

print(
    "Device :",
    device,
)

Steps Train : 1 -> 334
Transactions Train : 4,771,387
Fraudes Train : 3,745
pos_weight : 1273.068625
Batch size : 4096
Device : cpu


In [20]:
# Mettre en cache un row group

CORE_EDGE_CACHE_SIZE = 1

core_edge_cache = OrderedDict()

core_edge_cache_stats = {
    "physical_reads": 0,
    "cache_hits": 0,
}


def get_core_edge_row_group(
    row_group_id,
):
    if row_group_id in core_edge_cache:

        core_edge_cache_stats[
            "cache_hits"
        ] += 1

        table = core_edge_cache.pop(
            row_group_id
        )

        core_edge_cache[
            row_group_id
        ] = table

        return table


    table = (
        graph_ready_parquet
        .read_row_group(
            row_group_id,
            columns=core_edge_features,
            use_threads=True,
        )
    )


    core_edge_cache_stats[
        "physical_reads"
    ] += 1

    core_edge_cache[
        row_group_id
    ] = table


    if (
        len(core_edge_cache)
        > CORE_EDGE_CACHE_SIZE
    ):
        core_edge_cache.popitem(
            last=False
        )


    return table

In [21]:
# Lire les features avec cache

def read_core_edge_range_cached(
    start_index,
    end_index,
):
    first_group = int(
        np.searchsorted(
            row_group_offsets,
            start_index,
            side="right",
        )
        - 1
    )

    last_group = int(
        np.searchsorted(
            row_group_offsets,
            end_index - 1,
            side="right",
        )
        - 1
    )


    tables = []


    for row_group_id in range(
        first_group,
        last_group + 1,
    ):
        table = (
            get_core_edge_row_group(
                row_group_id
            )
        )


        group_start = int(
            row_group_offsets[
                row_group_id
            ]
        )

        group_end = int(
            row_group_offsets[
                row_group_id + 1
            ]
        )


        local_start = (
            max(
                start_index,
                group_start,
            )
            - group_start
        )

        local_end = (
            min(
                end_index,
                group_end,
            )
            - group_start
        )


        tables.append(
            table.slice(
                local_start,
                local_end - local_start,
            )
        )


    if len(tables) == 1:
        return tables[0]


    return pa.concat_tables(
        tables
    )

In [22]:
# Initialiser le vrai modèle

torch.manual_seed(
    RANDOM_STATE
)


full_model = (
    GraphSAGEEdgeClassifier(
        edge_feature_dim=(
            core_edge_input_dim
        ),
        hidden_dim=(
            FULL_HIDDEN_DIM
        ),
        num_layers=(
            FULL_NUM_LAYERS
        ),
        dropout=(
            FULL_DROPOUT
        ),
    )
    .to(device)
)


full_optimizer = (
    torch.optim.AdamW(
        full_model.parameters(),
        lr=FULL_LR,
        weight_decay=(
            FULL_WEIGHT_DECAY
        ),
    )
)


full_pos_weight = torch.tensor(
    [train_pos_weight],
    dtype=torch.float32,
    device=device,
)


full_criterion = (
    torch.nn.BCEWithLogitsLoss(
        pos_weight=full_pos_weight,
        reduction="sum",
    )
)


num_parameters = sum(
    parameter.numel()
    for parameter
    in full_model.parameters()
)


print(
    "Paramètres :",
    f"{num_parameters:,}",
)

print(
    "Device modèle :",
    next(
        full_model.parameters()
    ).device,
)

print(
    "Couches :",
    FULL_NUM_LAYERS,
)

print(
    "Hops :",
    FULL_NUM_HOPS,
)

Paramètres : 2,625
Device modèle : cpu
Couches : 1
Hops : 1


In [23]:
# Préparer les artefacts GNN

GNN_ARTIFACT_DIR = (
    ROOT
    / "artifacts"
    / "gnn"
)

GNN_ARTIFACT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


CHECKPOINT_PATH = (
    GNN_ARTIFACT_DIR
    / "graphsage_core_edge_1layer_checkpoint.pt"
)


def save_train_checkpoint(
    completed_step,
    integrated_end,
    history,
):
    checkpoint = {
        "completed_step": (
            completed_step
        ),
        "integrated_end": (
            integrated_end
        ),
        "model_state_dict": (
            full_model.state_dict()
        ),
        "optimizer_state_dict": (
            full_optimizer.state_dict()
        ),
        "config": (
            FULL_TRAIN_CONFIG
        ),
        "history": history,
    }


    temporary_path = (
        GNN_ARTIFACT_DIR
        / "checkpoint_temp.pt"
    )


    torch.save(
        checkpoint,
        temporary_path,
    )


    temporary_path.replace(
        CHECKPOINT_PATH
    )


print(
    "Dossier artefacts :",
    GNN_ARTIFACT_DIR.resolve(),
)

print(
    "Dossier prêt :",
    GNN_ARTIFACT_DIR.exists(),
)

Dossier artefacts : C:\Users\bouza\OneDrive\Bureau\MobileMoney-Fraud-GNN\artifacts\gnn
Dossier prêt : True


In [24]:
# Réinitialiser le graphe causal

head_event.fill(-1)
prev_in_event.fill(-1)

core_edge_cache.clear()

core_edge_cache_stats[
    "physical_reads"
] = 0

core_edge_cache_stats[
    "cache_hits"
] = 0


full_optimizer.zero_grad(
    set_to_none=True
)

gc.collect()


current_history_end = 0


print(
    "Historique initial :",
    current_history_end,
)

print(
    "head_event vide :",
    int(
        head_event.max()
    ) == -1,
)

print(
    "prev_in_event vide :",
    int(
        prev_in_event.max()
    ) == -1,
)

Historique initial : 0
head_event vide : True
prev_in_event vide : True


In [25]:
# Entraîner sur tout le Train

train_history = []

processed_transactions = 0
processed_frauds = 0

global_loss_sum = 0.0

max_process_ram_mb = 0.0

all_gradients_finite = True


process = psutil.Process()


full_training_start = (
    time.perf_counter()
)


for t in range(
    TRAIN_START,
    TRAIN_END + 1,
):

    step_start, step_end = (
        get_step_bounds(t)
    )


    # Vérifier la continuité temporelle

    if step_start != current_history_end:

        raise RuntimeError(
            f"Rupture causale au step {t}."
        )


    step_src = np.asarray(
        src[
            step_start:step_end
        ],
        dtype=np.int32,
    )

    step_dst = np.asarray(
        dst[
            step_start:step_end
        ],
        dtype=np.int32,
    )

    step_labels = np.asarray(
        label[
            step_start:step_end
        ],
        dtype=np.float32,
    )


    step_table = (
        read_core_edge_range_cached(
            step_start,
            step_end,
        )
    )


    step_features, unknown_types = (
        transform_core_edge_batch(
            step_table
        )
    )


    if unknown_types != 0:

        raise RuntimeError(
            f"Type inconnu au step {t}."
        )


    step_results = (
        train_causal_step_cpu(
            model=full_model,
            optimizer=full_optimizer,
            criterion=full_criterion,
            query_src=step_src,
            query_dst=step_dst,
            query_features=step_features,
            query_labels=step_labels,
            batch_size=FULL_BATCH_SIZE,
            num_hops=FULL_NUM_HOPS,
        )
    )


    if (
        not step_results[
            "gradients_finite"
        ]
        or not np.isfinite(
            step_results[
                "mean_loss"
            ]
        )
    ):

        raise RuntimeError(
            f"Instabilité numérique "
            f"au step {t}."
        )


    # Intégrer le step après l'update

    integrate_event_range(
        step_start,
        step_end,
    )


    current_history_end = (
        step_end
    )


    step_queries = (
        step_end
        - step_start
    )

    step_frauds = int(
        step_labels.sum()
    )


    processed_transactions += (
        step_queries
    )

    processed_frauds += (
        step_frauds
    )


    global_loss_sum += (
        step_results[
            "mean_loss"
        ]
        * step_queries
    )


    max_process_ram_mb = max(
        max_process_ram_mb,
        step_results[
            "peak_process_ram_mb"
        ],
    )


    all_gradients_finite &= (
        step_results[
            "gradients_finite"
        ]
    )


    history_row = {
        "step": int(t),
        "transactions": int(
            step_queries
        ),
        "frauds": int(
            step_frauds
        ),
        "mean_loss": float(
            step_results[
                "mean_loss"
            ]
        ),
        "gradient_norm": float(
            step_results[
                "gradient_norm"
            ]
        ),
        "runtime_seconds": float(
            step_results[
                "runtime_seconds"
            ]
        ),
        "num_batches": int(
            step_results[
                "num_batches"
            ]
        ),
        "mean_local_nodes": float(
            step_results[
                "mean_local_nodes"
            ]
        ),
        "mean_context_edges": float(
            step_results[
                "mean_context_edges"
            ]
        ),
    }


    train_history.append(
        history_row
    )


    # Afficher la progression

    if (
        t == TRAIN_START
        or t % 25 == 0
        or t == TRAIN_END
    ):

        elapsed = (
            time.perf_counter()
            - full_training_start
        )

        memory = (
            psutil.virtual_memory()
        )

        print(
            f"Step {t:3d}/{TRAIN_END}"
            f" | tx={step_queries:,}"
            f" | fraud={step_frauds}"
            f" | loss={step_results['mean_loss']:.6f}"
            f" | grad={step_results['gradient_norm']:.4f}"
            f" | step={step_results['runtime_seconds']:.2f}s"
            f" | total={elapsed:.1f}s"
            f" | RAM libre={memory.available / 1024**3:.2f} GB"
        )


    # Sauvegarder régulièrement

    if (
        t % CHECKPOINT_EVERY == 0
        or t == TRAIN_END
    ):

        save_train_checkpoint(
            completed_step=t,
            integrated_end=(
                current_history_end
            ),
            history=train_history,
        )


    del (
        step_src,
        step_dst,
        step_labels,
        step_table,
        step_features,
        step_results,
    )


    if t % 25 == 0:
        gc.collect()

Step   1/334 | tx=2,708 | fraud=16 | loss=5.462092 | grad=7.6714 | step=0.30s | total=0.4s | RAM libre=3.24 GB
Step  25/334 | tx=1,598 | fraud=18 | loss=6.000454 | grad=16.9327 | step=0.03s | total=16.3s | RAM libre=3.27 GB
Step  50/334 | tx=6 | fraud=6 | loss=219.001994 | grad=1024.9903 | step=0.00s | total=35.7s | RAM libre=3.34 GB
Step  75/334 | tx=8 | fraud=8 | loss=32.354073 | grad=268.4943 | step=0.00s | total=36.1s | RAM libre=3.32 GB
Step 100/334 | tx=6 | fraud=6 | loss=13.390770 | grad=114.5139 | step=0.00s | total=38.1s | RAM libre=3.30 GB
Step 125/334 | tx=10 | fraud=10 | loss=4.915949 | grad=48.6959 | step=0.00s | total=39.3s | RAM libre=3.33 GB
Step 150/334 | tx=14 | fraud=14 | loss=3.692837 | grad=36.7862 | step=0.00s | total=57.8s | RAM libre=3.27 GB
Step 175/334 | tx=8 | fraud=8 | loss=9.052444 | grad=81.1535 | step=0.00s | total=74.9s | RAM libre=3.24 GB
Step 200/334 | tx=8 | fraud=8 | loss=5.570438 | grad=50.3673 | step=0.00s | total=96.6s | RAM libre=3.31 GB
Step 225

In [26]:
# Résumer l'entraînement

full_training_seconds = (
    time.perf_counter()
    - full_training_start
)


global_mean_loss = (
    global_loss_sum
    / processed_transactions
)


sum_step_runtime = sum(
    row[
        "runtime_seconds"
    ]
    for row in train_history
)


model_parameters_finite = all(
    bool(
        torch.isfinite(
            parameter
        ).all()
    )
    for parameter
    in full_model.parameters()
)


memory = psutil.virtual_memory()


print(
    "Steps traités :",
    len(train_history),
)

print(
    "Transactions traitées :",
    f"{processed_transactions:,}",
)

print(
    "Fraudes traitées :",
    f"{processed_frauds:,}",
)

print(
    f"Loss pondérée moyenne : "
    f"{global_mean_loss:.6f}"
)

print(
    f"Runtime total : "
    f"{full_training_seconds:.2f} s"
)

print(
    f"Runtime modèle cumulé : "
    f"{sum_step_runtime:.2f} s"
)

print(
    f"Peak RAM processus : "
    f"{max_process_ram_mb:.2f} MB"
)

print(
    "Gradients tous finis :",
    all_gradients_finite,
)

print(
    "Paramètres tous finis :",
    model_parameters_finite,
)

print(
    "Row groups lus physiquement :",
    core_edge_cache_stats[
        "physical_reads"
    ],
)

print(
    "Cache hits :",
    core_edge_cache_stats[
        "cache_hits"
    ],
)

print(
    f"RAM disponible finale : "
    f"{memory.available / 1024**3:.2f} GB"
)

Steps traités : 334
Transactions traitées : 4,771,387
Fraudes traitées : 3,745
Loss pondérée moyenne : 4.412278
Runtime total : 236.02 s
Runtime modèle cumulé : 153.15 s
Peak RAM processus : 490.26 MB
Gradients tous finis : True
Paramètres tous finis : True
Row groups lus physiquement : 20
Cache hits : 333
RAM disponible finale : 3.14 GB


In [27]:
# Sauvegarder les artefacts finaux

FINAL_MODEL_PATH = (
    GNN_ARTIFACT_DIR
    / "graphsage_core_edge_1layer_train_final.pt"
)

FINAL_HISTORY_PATH = (
    GNN_ARTIFACT_DIR
    / "graphsage_core_edge_1layer_train_history.json"
)

FINAL_CONFIG_PATH = (
    GNN_ARTIFACT_DIR
    / "graphsage_core_edge_1layer_config.json"
)


final_checkpoint = {
    "model_state_dict": (
        full_model.state_dict()
    ),
    "config": (
        FULL_TRAIN_CONFIG
    ),
    "completed_step": (
        TRAIN_END
    ),
    "integrated_end": (
        current_history_end
    ),
    "train_summary": {
        "transactions": (
            processed_transactions
        ),
        "frauds": (
            processed_frauds
        ),
        "mean_weighted_loss": (
            global_mean_loss
        ),
        "runtime_seconds": (
            full_training_seconds
        ),
        "peak_process_ram_mb": (
            max_process_ram_mb
        ),
    },
}


torch.save(
    final_checkpoint,
    FINAL_MODEL_PATH,
)


with open(
    FINAL_HISTORY_PATH,
    "w",
    encoding="utf-8",
) as file:

    json.dump(
        train_history,
        file,
        indent=2,
    )


with open(
    FINAL_CONFIG_PATH,
    "w",
    encoding="utf-8",
) as file:

    json.dump(
        FULL_TRAIN_CONFIG,
        file,
        indent=2,
    )


print(
    "Modèle sauvegardé :",
    FINAL_MODEL_PATH.exists(),
)

print(
    "Historique sauvegardé :",
    FINAL_HISTORY_PATH.exists(),
)

print(
    "Configuration sauvegardée :",
    FINAL_CONFIG_PATH.exists(),
)

print(
    "Dossier :",
    GNN_ARTIFACT_DIR.resolve(),
)

Modèle sauvegardé : True
Historique sauvegardé : True
Configuration sauvegardée : True
Dossier : C:\Users\bouza\OneDrive\Bureau\MobileMoney-Fraud-GNN\artifacts\gnn


In [28]:
# Valider la Section 11

full_train_valid = (
    len(train_history)
        == 334
    and processed_transactions
        == TRAIN_ROWS
    and processed_frauds
        == train_positive
    and current_history_end
        == TRAIN_ROWS
    and all_gradients_finite
    and model_parameters_finite
    and np.isfinite(
        global_mean_loss
    )
    and FINAL_MODEL_PATH.exists()
    and FINAL_HISTORY_PATH.exists()
    and FINAL_CONFIG_PATH.exists()
)


print(
    "Entraînement complet valide :",
    bool(full_train_valid),
)

print(
    "Historique final :",
    f"{current_history_end:,}",
)


if full_train_valid:

    print(
        "SECTION 11 : "
        "GraphSAGE 1-layer Train validé."
    )

else:

    print(
        "SECTION 11 : "
        "anomalie à analyser."
    )

Entraînement complet valide : True
Historique final : 4,771,387
SECTION 11 : GraphSAGE 1-layer Train validé.


### Full causal training results

Le baseline GraphSAGE à une couche a été entraîné sur l'intégralité du Train selon le protocole temporel causal défini précédemment.

L'entraînement couvre les 334 steps Train, soit 4 771 387 transactions et 3 745 fraudes. Pour chaque step `t`, les prédictions sont réalisées exclusivement à partir de `G_<t`, les gradients sont accumulés sur l'ensemble des batches du step, puis une seule mise à jour des paramètres est effectuée avant l'intégration de `E_t` dans le graphe.

L'ensemble des transactions et des fraudes du Train a été traité sans rupture de continuité temporelle. Tous les gradients et paramètres du modèle sont restés finis pendant l'apprentissage.

La loss pondérée moyenne sur le Train est de 4,4123. Certains steps très courts constitués principalement ou exclusivement de transactions frauduleuses présentent des valeurs de loss et de norme de gradient élevées. Ce comportement est cohérent avec le fort `pos_weight` utilisé pour compenser le déséquilibre extrême des classes. Aucune divergence numérique n'a toutefois été observée.

La boucle d'entraînement complète atteint le step 334 en environ 199 secondes sur CPU local. Le temps cumulé consacré au forward/backward est d'environ 153 secondes.

Le pipeline reste également peu coûteux en mémoire, avec un pic d'environ 490 MB pour le processus Python. Le mécanisme de cache Parquet limite les lectures physiques à 20 row groups pour les 334 steps, avec 333 accès servis depuis le cache.

Le modèle final, son historique d'entraînement et sa configuration ont été sauvegardés dans les artefacts du projet.

L'entraînement causal du baseline GraphSAGE 1-layer est ainsi validé. La performance prédictive doit maintenant être mesurée sur la période de Validation, avec les poids du modèle entièrement figés.

## 12. Causal Validation - GraphSAGE 1-Layer

Cette section évalue le baseline GraphSAGE entraîné précédemment sur la période de Validation, correspondant aux steps 335 à 392.

Les paramètres du modèle sont entièrement figés. Pour chaque step `t`, les prédictions sont calculées exclusivement à partir du contexte historique `G_<t`. Toutes les transactions du step courant sont évaluées avant que leur topologie soit intégrée au graphe.

Aucun gradient, aucune mise à jour des paramètres et aucune information provenant du Test ne sont utilisés pendant cette phase.

Les probabilités prédites sont conservées pour l'ensemble des transactions de Validation afin de calculer la PR-AUC, métrique principale du projet, ainsi que la ROC-AUC et les analyses ultérieures.

La sélection du seuil de classification n'est pas réalisée dans cette première étape. Elle sera effectuée séparément, exclusivement à partir des prédictions de Validation.

In [29]:
# Vérifier l'état avant Validation

VAL_ROWS = int(
    step_offsets[VAL_END]
    - step_offsets[VAL_START - 1]
)

VAL_GLOBAL_START = int(
    step_offsets[VAL_START - 1]
)

VAL_GLOBAL_END = int(
    step_offsets[VAL_END]
)


print(
    "Début Validation :",
    f"{VAL_GLOBAL_START:,}",
)

print(
    "Fin Validation :",
    f"{VAL_GLOBAL_END:,}",
)

print(
    "Transactions Validation :",
    f"{VAL_ROWS:,}",
)

print(
    "Historique courant :",
    f"{current_history_end:,}",
)

print(
    "Historique = fin Train :",
    current_history_end
    == TRAIN_ROWS,
)

print(
    "Début Val = fin Train :",
    VAL_GLOBAL_START
    == TRAIN_ROWS,
)

Début Validation : 4,771,387
Fin Validation : 5,563,793
Transactions Validation : 792,406
Historique courant : 4,771,387
Historique = fin Train : True
Début Val = fin Train : True


In [30]:
# Sauvegarder les poids avant Validation

validation_weights_before = {
    name: parameter
    .detach()
    .cpu()
    .clone()

    for name, parameter
    in full_model.state_dict().items()
}


full_model.eval()


print(
    "Mode entraînement :",
    full_model.training,
)

print(
    "Paramètres sauvegardés :",
    len(
        validation_weights_before
    ),
)

Mode entraînement : False
Paramètres sauvegardés : 7


In [31]:
# Prédire un step causal

def predict_causal_step_cpu(
    model,
    query_src,
    query_dst,
    query_features,
    batch_size,
    num_hops=1,
):
    model.eval()


    num_queries = len(
        query_src
    )

    num_batches = int(
        np.ceil(
            num_queries
            / batch_size
        )
    )


    probabilities = np.empty(
        num_queries,
        dtype=np.float32,
    )


    local_nodes_counts = []
    context_edge_counts = []


    start_time = (
        time.perf_counter()
    )


    with torch.no_grad():

        for batch_id in range(
            num_batches
        ):

            batch_start = (
                batch_id
                * batch_size
            )

            batch_end = min(
                batch_start
                + batch_size,
                num_queries,
            )


            subgraph = (
                extract_causal_subgraph(
                    query_src[
                        batch_start:
                        batch_end
                    ],
                    query_dst[
                        batch_start:
                        batch_end
                    ],
                    num_hops=num_hops,
                )
            )


            features_cpu = (
                torch.from_numpy(
                    query_features[
                        batch_start:
                        batch_end
                    ]
                )
            )


            logits = model(
                num_nodes=len(
                    subgraph[
                        "local_nodes"
                    ]
                ),
                context_edge_index=(
                    subgraph[
                        "context_edge_index"
                    ]
                ),
                query_edge_index=(
                    subgraph[
                        "query_edge_index"
                    ]
                ),
                edge_features=(
                    features_cpu
                ),
            )


            batch_probabilities = (
                torch.sigmoid(
                    logits
                )
                .cpu()
                .numpy()
                .astype(
                    np.float32,
                    copy=False,
                )
            )


            probabilities[
                batch_start:
                batch_end
            ] = batch_probabilities


            local_nodes_counts.append(
                len(
                    subgraph[
                        "local_nodes"
                    ]
                )
            )

            context_edge_counts.append(
                len(
                    subgraph[
                        "context_event_ids"
                    ]
                )
            )


            del (
                subgraph,
                features_cpu,
                logits,
                batch_probabilities,
            )


    runtime_seconds = (
        time.perf_counter()
        - start_time
    )


    return {
        "probabilities": probabilities,
        "num_batches": num_batches,
        "runtime_seconds": (
            runtime_seconds
        ),
        "mean_local_nodes": float(
            np.mean(
                local_nodes_counts
            )
        ),
        "max_local_nodes": int(
            np.max(
                local_nodes_counts
            )
        ),
        "mean_context_edges": float(
            np.mean(
                context_edge_counts
            )
        ),
        "max_context_edges": int(
            np.max(
                context_edge_counts
            )
        ),
    }

In [32]:
# Préparer les résultats Validation

val_probabilities = np.empty(
    VAL_ROWS,
    dtype=np.float32,
)

val_labels = np.empty(
    VAL_ROWS,
    dtype=np.int8,
)

val_steps = np.empty(
    VAL_ROWS,
    dtype=np.int16,
)

val_types = np.empty(
    VAL_ROWS,
    dtype=np.int8,
)


type_to_code = {
    category: index
    for index, category
    in enumerate(type_categories)
}


validation_cursor = 0

validation_history = []

In [33]:
# Évaluer toute la Validation

validation_start_time = (
    time.perf_counter()
)


for t in range(
    VAL_START,
    VAL_END + 1,
):

    step_start, step_end = (
        get_step_bounds(t)
    )


    # Vérifier le contexte causal

    if step_start != current_history_end:

        raise RuntimeError(
            f"Rupture causale "
            f"au step Validation {t}."
        )


    step_src = np.asarray(
        src[
            step_start:step_end
        ],
        dtype=np.int32,
    )

    step_dst = np.asarray(
        dst[
            step_start:step_end
        ],
        dtype=np.int32,
    )

    step_labels = np.asarray(
        label[
            step_start:step_end
        ],
        dtype=np.int8,
    )


    step_table = (
        read_core_edge_range_cached(
            step_start,
            step_end,
        )
    )


    step_features, unknown_types = (
        transform_core_edge_batch(
            step_table
        )
    )


    if unknown_types != 0:

        raise RuntimeError(
            f"Type inconnu "
            f"au step Validation {t}."
        )


    step_results = (
        predict_causal_step_cpu(
            model=full_model,
            query_src=step_src,
            query_dst=step_dst,
            query_features=step_features,
            batch_size=FULL_BATCH_SIZE,
            num_hops=FULL_NUM_HOPS,
        )
    )


    step_probabilities = (
        step_results[
            "probabilities"
        ]
    )


    if not np.isfinite(
        step_probabilities
    ).all():

        raise RuntimeError(
            f"Score non fini "
            f"au step Validation {t}."
        )


    step_size = (
        step_end
        - step_start
    )


    output_start = (
        validation_cursor
    )

    output_end = (
        validation_cursor
        + step_size
    )


    val_probabilities[
        output_start:
        output_end
    ] = step_probabilities


    val_labels[
        output_start:
        output_end
    ] = step_labels


    val_steps[
        output_start:
        output_end
    ] = t


    step_type_values = (
        step_table["type"]
        .combine_chunks()
        .to_pylist()
    )


    val_types[
        output_start:
        output_end
    ] = np.fromiter(
        (
            type_to_code[
                category
            ]
            for category
            in step_type_values
        ),
        dtype=np.int8,
        count=step_size,
    )


    validation_cursor = (
        output_end
    )


    # Intégrer E_t après toutes les prédictions

    integrate_event_range(
        step_start,
        step_end,
    )

    current_history_end = (
        step_end
    )


    validation_history.append({
        "step": int(t),
        "transactions": int(
            step_size
        ),
        "frauds": int(
            step_labels.sum()
        ),
        "runtime_seconds": float(
            step_results[
                "runtime_seconds"
            ]
        ),
        "mean_local_nodes": float(
            step_results[
                "mean_local_nodes"
            ]
        ),
        "mean_context_edges": float(
            step_results[
                "mean_context_edges"
            ]
        ),
    })


    if (
        t == VAL_START
        or t % 10 == 0
        or t == VAL_END
    ):

        elapsed = (
            time.perf_counter()
            - validation_start_time
        )

        print(
            f"Val step {t:3d}/{VAL_END}"
            f" | tx={step_size:,}"
            f" | fraud={int(step_labels.sum())}"
            f" | score_mean={step_probabilities.mean():.6f}"
            f" | step={step_results['runtime_seconds']:.2f}s"
            f" | total={elapsed:.1f}s"
        )


    del (
        step_src,
        step_dst,
        step_labels,
        step_table,
        step_features,
        step_results,
        step_probabilities,
        step_type_values,
    )

Val step 335/392 | tx=10,456 | fraud=4 | score_mean=0.979157 | step=0.66s | total=1.0s
Val step 340/392 | tx=10 | fraud=10 | score_mean=0.996600 | step=0.00s | total=1.1s
Val step 350/392 | tx=28,245 | fraud=14 | score_mean=0.979684 | step=0.59s | total=6.3s
Val step 360/392 | tx=3,508 | fraud=8 | score_mean=0.979593 | step=0.05s | total=14.5s
Val step 370/392 | tx=25,429 | fraud=18 | score_mean=0.979514 | step=0.51s | total=16.0s
Val step 380/392 | tx=37,693 | fraud=14 | score_mean=0.979725 | step=1.24s | total=26.7s
Val step 390/392 | tx=12 | fraud=12 | score_mean=0.998346 | step=0.00s | total=28.2s
Val step 392/392 | tx=10 | fraud=10 | score_mean=0.996608 | step=0.00s | total=28.2s


In [34]:
# Vérifier les poids figés

validation_weights_unchanged = all(
    torch.equal(
        validation_weights_before[
            name
        ],
        parameter
        .detach()
        .cpu(),
    )

    for name, parameter
    in full_model.state_dict().items()
)


print(
    "Poids inchangés :",
    validation_weights_unchanged,
)

print(
    "Transactions collectées :",
    f"{validation_cursor:,}",
)

print(
    "Collecte complète :",
    validation_cursor
    == VAL_ROWS,
)

print(
    "Fraudes collectées :",
    int(
        val_labels.sum(
            dtype=np.int64
        )
    ),
)

print(
    "Scores finis :",
    np.isfinite(
        val_probabilities
    ).all(),
)

print(
    "Scores dans [0, 1] :",
    bool(
        (
            (
                val_probabilities
                >= 0
            )
            & (
                val_probabilities
                <= 1
            )
        ).all()
    ),
)

Poids inchangés : True
Transactions collectées : 792,406
Collecte complète : True
Fraudes collectées : 638
Scores finis : True
Scores dans [0, 1] : True


In [35]:
# Calculer les métriques Validation

from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
)


val_pr_auc = (
    average_precision_score(
        val_labels,
        val_probabilities,
    )
)

val_roc_auc = (
    roc_auc_score(
        val_labels,
        val_probabilities,
    )
)


print(
    f"Validation PR-AUC : "
    f"{val_pr_auc:.6f}"
)

print(
    f"Validation ROC-AUC : "
    f"{val_roc_auc:.6f}"
)

print()

print(
    "LightGBM Core Val PR-AUC :",
    "0.929704",
)

print(
    f"Écart GNN - LightGBM : "
    f"{val_pr_auc - 0.929704:+.6f}"
)

Validation PR-AUC : 0.240136
Validation ROC-AUC : 0.962366

LightGBM Core Val PR-AUC : 0.929704
Écart GNN - LightGBM : -0.689568


In [36]:
# Résumer les scores Validation

negative_scores = (
    val_probabilities[
        val_labels == 0
    ]
)

positive_scores = (
    val_probabilities[
        val_labels == 1
    ]
)


print(
    "Score moyen global :",
    float(
        val_probabilities.mean()
    ),
)

print(
    "Score moyen non-fraude :",
    float(
        negative_scores.mean()
    ),
)

print(
    "Score moyen fraude :",
    float(
        positive_scores.mean()
    ),
)

print()

print(
    "Médiane non-fraude :",
    float(
        np.median(
            negative_scores
        )
    ),
)

print(
    "Médiane fraude :",
    float(
        np.median(
            positive_scores
        )
    ),
)

print()

print(
    "Score max non-fraude :",
    float(
        negative_scores.max()
    ),
)

print(
    "Score min fraude :",
    float(
        positive_scores.min()
    ),
)

Score moyen global : 0.9797433614730835
Score moyen non-fraude : 0.9797297716140747
Score moyen fraude : 0.9969548583030701

Médiane non-fraude : 0.9825775623321533
Médiane fraude : 0.9974403381347656

Score max non-fraude : 0.9999924898147583
Score min fraude : 0.9851817488670349


In [37]:
# Sauvegarder les prédictions Validation

VALIDATION_PREDICTIONS_PATH = (
    GNN_ARTIFACT_DIR
    / "graphsage_core_edge_1layer_validation_predictions.npz"
)


np.savez_compressed(
    VALIDATION_PREDICTIONS_PATH,
    probabilities=(
        val_probabilities
    ),
    labels=(
        val_labels
    ),
    steps=(
        val_steps
    ),
    types=(
        val_types
    ),
)


VALIDATION_HISTORY_PATH = (
    GNN_ARTIFACT_DIR
    / "graphsage_core_edge_1layer_validation_history.json"
)


with open(
    VALIDATION_HISTORY_PATH,
    "w",
    encoding="utf-8",
) as file:

    json.dump(
        validation_history,
        file,
        indent=2,
    )


print(
    "Prédictions sauvegardées :",
    VALIDATION_PREDICTIONS_PATH.exists(),
)

print(
    "Historique sauvegardé :",
    VALIDATION_HISTORY_PATH.exists(),
)

Prédictions sauvegardées : True
Historique sauvegardé : True


In [38]:
# Valider la Section 12

validation_seconds = (
    time.perf_counter()
    - validation_start_time
)


validation_valid = (
    validation_cursor
        == VAL_ROWS
    and int(
        val_labels.sum(
            dtype=np.int64
        )
    )
        == 638
    and validation_weights_unchanged
    and np.isfinite(
        val_probabilities
    ).all()
    and np.isfinite(
        val_pr_auc
    )
    and np.isfinite(
        val_roc_auc
    )
    and current_history_end
        == VAL_GLOBAL_END
    and VALIDATION_PREDICTIONS_PATH.exists()
)


print(
    "Validation causale valide :",
    bool(
        validation_valid
    ),
)

print(
    f"Runtime Validation : "
    f"{validation_seconds:.2f} s"
)

print(
    "Historique final Val :",
    f"{current_history_end:,}",
)


if validation_valid:

    print(
        "SECTION 12 : "
        "scores Validation causaux validés."
    )

else:

    print(
        "SECTION 12 : "
        "anomalie à analyser."
    )

Validation causale valide : True
Runtime Validation : 6081.58 s
Historique final Val : 5,563,793
SECTION 12 : scores Validation causaux validés.


### Causal Validation results

Le modèle GraphSAGE 1-layer a été évalué causalement sur les 792 406 transactions de Validation, correspondant aux steps 335 à 392 et contenant 638 fraudes.

Pendant toute l'évaluation, les paramètres du modèle sont restés strictement figés. Pour chaque step `t`, les prédictions ont été calculées exclusivement à partir de `G_<t`, puis la topologie de `E_t` a été intégrée après prédiction de l'ensemble du step. La vérification des paramètres confirme qu'aucun poids n'a été modifié pendant la Validation.

Le modèle obtient une PR-AUC de 0,240136 et une ROC-AUC de 0,962366. La PR-AUC reste très supérieure au taux de fraude de la Validation, d'environ 0,0805 %, ce qui confirme l'existence d'un signal prédictif. Elle reste néanmoins très inférieure à la PR-AUC de 0,929704 obtenue par le baseline LightGBM Core.

Les probabilités sont fortement concentrées vers 1. Le score moyen atteint environ 0,9797 pour les transactions non frauduleuses et 0,9970 pour les fraudes. Cette calibration extrême est cohérente avec le `pos_weight` élevé utilisé pendant l'apprentissage. Elle implique notamment qu'un seuil conventionnel de 0,5 n'est pas adapté à ce modèle.

La ROC-AUC élevée combinée à une PR-AUC nettement plus faible montre que le modèle apprend un signal discriminant, mais ne classe pas encore suffisamment précisément les rares transactions frauduleuses parmi les observations les mieux scorées.

La boucle causale complète de Validation s'est terminée en environ 28,2 secondes sur CPU local.

Avant toute évaluation sur le Test, la profondeur GraphSAGE doit être sélectionnée exclusivement à partir de la Validation.

## 13. Depth Ablation - GraphSAGE 1-Layer vs 2-Layer

Cette section évalue l'effet de la profondeur du message passing en comparant les architectures GraphSAGE à une et deux couches.

La variante à deux couches est entraînée de zéro avec exactement le même protocole causal, les mêmes features `Graph Core Edge`, les mêmes hyperparamètres d'optimisation et le même `pos_weight` que la variante à une couche.

La seule modification concerne la profondeur du modèle et du contexte local : deux couches GraphSAGE et deux hops historiques sont utilisés.

La sélection entre les deux architectures est effectuée exclusivement à partir de la PR-AUC de Validation. Le Test reste entièrement inutilisé à ce stade.

In [39]:
# Configurer la variante 2 couches

DEPTH2_NUM_LAYERS = 2
DEPTH2_NUM_HOPS = 2


torch.manual_seed(
    RANDOM_STATE
)


depth2_model = (
    GraphSAGEEdgeClassifier(
        edge_feature_dim=(
            core_edge_input_dim
        ),
        hidden_dim=(
            FULL_HIDDEN_DIM
        ),
        num_layers=(
            DEPTH2_NUM_LAYERS
        ),
        dropout=(
            FULL_DROPOUT
        ),
    )
    .to(device)
)


depth2_optimizer = (
    torch.optim.AdamW(
        depth2_model.parameters(),
        lr=FULL_LR,
        weight_decay=(
            FULL_WEIGHT_DECAY
        ),
    )
)


depth2_criterion = (
    torch.nn.BCEWithLogitsLoss(
        pos_weight=torch.tensor(
            [train_pos_weight],
            dtype=torch.float32,
            device=device,
        ),
        reduction="sum",
    )
)


depth2_parameters = sum(
    parameter.numel()
    for parameter
    in depth2_model.parameters()
)


print(
    "Couches :",
    DEPTH2_NUM_LAYERS,
)

print(
    "Hops :",
    DEPTH2_NUM_HOPS,
)

print(
    "Paramètres :",
    f"{depth2_parameters:,}",
)

print(
    "Device :",
    next(
        depth2_model.parameters()
    ).device,
)

Couches : 2
Hops : 2
Paramètres : 4,705
Device : cpu


In [40]:
# Réinitialiser le contexte

head_event.fill(-1)
prev_in_event.fill(-1)

core_edge_cache.clear()

core_edge_cache_stats[
    "physical_reads"
] = 0

core_edge_cache_stats[
    "cache_hits"
] = 0


gc.collect()


depth2_current_history_end = 0


print(
    "Historique initial :",
    depth2_current_history_end,
)

print(
    "head_event vide :",
    int(
        head_event.max()
    ) == -1,
)

print(
    "prev_in_event vide :",
    int(
        prev_in_event.max()
    ) == -1,
)

Historique initial : 0
head_event vide : True
prev_in_event vide : True


In [41]:
# Entraîner GraphSAGE 2 couches

depth2_train_history = []

depth2_processed_transactions = 0
depth2_processed_frauds = 0

depth2_global_loss_sum = 0.0

depth2_training_start = (
    time.perf_counter()
)


for t in range(
    TRAIN_START,
    TRAIN_END + 1,
):

    step_start, step_end = (
        get_step_bounds(t)
    )


    if (
        step_start
        != depth2_current_history_end
    ):
        raise RuntimeError(
            f"Rupture causale "
            f"au step {t}."
        )


    step_src = np.asarray(
        src[
            step_start:step_end
        ],
        dtype=np.int32,
    )

    step_dst = np.asarray(
        dst[
            step_start:step_end
        ],
        dtype=np.int32,
    )

    step_labels = np.asarray(
        label[
            step_start:step_end
        ],
        dtype=np.float32,
    )


    step_table = (
        read_core_edge_range_cached(
            step_start,
            step_end,
        )
    )


    step_features, unknown_types = (
        transform_core_edge_batch(
            step_table
        )
    )


    if unknown_types != 0:
        raise RuntimeError(
            f"Type inconnu "
            f"au step {t}."
        )


    step_results = (
        train_causal_step_cpu(
            model=depth2_model,
            optimizer=depth2_optimizer,
            criterion=depth2_criterion,
            query_src=step_src,
            query_dst=step_dst,
            query_features=step_features,
            query_labels=step_labels,
            batch_size=FULL_BATCH_SIZE,
            num_hops=DEPTH2_NUM_HOPS,
        )
    )


    if (
        not step_results[
            "gradients_finite"
        ]
        or not np.isfinite(
            step_results[
                "mean_loss"
            ]
        )
    ):
        raise RuntimeError(
            f"Instabilité numérique "
            f"au step {t}."
        )


    # Intégrer après apprentissage

    integrate_event_range(
        step_start,
        step_end,
    )

    depth2_current_history_end = (
        step_end
    )


    step_size = (
        step_end
        - step_start
    )

    step_frauds = int(
        step_labels.sum()
    )


    depth2_processed_transactions += (
        step_size
    )

    depth2_processed_frauds += (
        step_frauds
    )

    depth2_global_loss_sum += (
        step_results[
            "mean_loss"
        ]
        * step_size
    )


    depth2_train_history.append({
        "step": int(t),
        "transactions": int(
            step_size
        ),
        "frauds": int(
            step_frauds
        ),
        "mean_loss": float(
            step_results[
                "mean_loss"
            ]
        ),
        "gradient_norm": float(
            step_results[
                "gradient_norm"
            ]
        ),
        "runtime_seconds": float(
            step_results[
                "runtime_seconds"
            ]
        ),
        "mean_local_nodes": float(
            step_results[
                "mean_local_nodes"
            ]
        ),
        "mean_context_edges": float(
            step_results[
                "mean_context_edges"
            ]
        ),
    })


    if (
        t == TRAIN_START
        or t % 25 == 0
        or t == TRAIN_END
    ):

        elapsed = (
            time.perf_counter()
            - depth2_training_start
        )

        print(
            f"Depth2 {t:3d}/{TRAIN_END}"
            f" | tx={step_size:,}"
            f" | fraud={step_frauds}"
            f" | loss={step_results['mean_loss']:.6f}"
            f" | grad={step_results['gradient_norm']:.4f}"
            f" | step={step_results['runtime_seconds']:.2f}s"
            f" | total={elapsed:.1f}s"
        )


    del (
        step_src,
        step_dst,
        step_labels,
        step_table,
        step_features,
        step_results,
    )


    if t % 25 == 0:
        gc.collect()

Depth2   1/334 | tx=2,708 | fraud=16 | loss=6.090561 | grad=6.4105 | step=0.57s | total=0.7s
Depth2  25/334 | tx=1,598 | fraud=18 | loss=10.957358 | grad=52.6890 | step=0.08s | total=34.4s
Depth2  50/334 | tx=6 | fraud=6 | loss=294.012573 | grad=1147.0833 | step=0.00s | total=63.6s
Depth2  75/334 | tx=8 | fraud=8 | loss=20.079823 | grad=185.3135 | step=0.00s | total=64.1s
Depth2 100/334 | tx=6 | fraud=6 | loss=2.888149 | grad=37.8911 | step=0.00s | total=66.3s
Depth2 125/334 | tx=10 | fraud=10 | loss=1.140414 | grad=14.9110 | step=0.00s | total=67.3s
Depth2 150/334 | tx=14 | fraud=14 | loss=1.124528 | grad=15.1121 | step=0.00s | total=97.3s
Depth2 175/334 | tx=8 | fraud=8 | loss=4.003499 | grad=52.5658 | step=0.00s | total=123.7s
Depth2 200/334 | tx=8 | fraud=8 | loss=3.097785 | grad=50.0920 | step=0.00s | total=154.1s
Depth2 225/334 | tx=18,366 | fraud=12 | loss=5.988430 | grad=24.7717 | step=1.24s | total=183.1s
Depth2 250/334 | tx=31,854 | fraud=22 | loss=6.390483 | grad=24.8718 | s

In [42]:
# Résumer le Train 2 couches

depth2_training_seconds = (
    time.perf_counter()
    - depth2_training_start
)


depth2_mean_loss = (
    depth2_global_loss_sum
    / depth2_processed_transactions
)


depth2_parameters_finite = all(
    bool(
        torch.isfinite(
            parameter
        ).all()
    )
    for parameter
    in depth2_model.parameters()
)


print(
    "Transactions :",
    f"{depth2_processed_transactions:,}",
)

print(
    "Fraudes :",
    f"{depth2_processed_frauds:,}",
)

print(
    f"Loss moyenne : "
    f"{depth2_mean_loss:.6f}"
)

print(
    f"Runtime Train : "
    f"{depth2_training_seconds:.2f} s"
)

print(
    "Paramètres finis :",
    depth2_parameters_finite,
)

print(
    "Historique = fin Train :",
    depth2_current_history_end
    == TRAIN_ROWS,
)

Transactions : 4,771,387
Fraudes : 3,745
Loss moyenne : 5.907757
Runtime Train : 595.99 s
Paramètres finis : True
Historique = fin Train : True


In [43]:
# Sauvegarder le modèle 2 couches

DEPTH2_MODEL_PATH = (
    GNN_ARTIFACT_DIR
    / "graphsage_core_edge_2layer_train_final.pt"
)


torch.save(
    {
        "model_state_dict": (
            depth2_model.state_dict()
        ),
        "config": {
            **FULL_TRAIN_CONFIG,
            "num_layers": 2,
            "num_hops": 2,
        },
        "completed_step": (
            TRAIN_END
        ),
        "integrated_end": (
            depth2_current_history_end
        ),
    },
    DEPTH2_MODEL_PATH,
)


print(
    "Modèle 2 couches sauvegardé :",
    DEPTH2_MODEL_PATH.exists(),
)

Modèle 2 couches sauvegardé : True


In [44]:
# Préparer la Validation 2 couches

depth2_model.eval()


depth2_weights_before = {
    name: parameter
    .detach()
    .cpu()
    .clone()

    for name, parameter
    in depth2_model.state_dict().items()
}


depth2_val_probabilities = np.empty(
    VAL_ROWS,
    dtype=np.float32,
)

depth2_val_labels = np.empty(
    VAL_ROWS,
    dtype=np.int8,
)


depth2_val_cursor = 0

depth2_validation_start = (
    time.perf_counter()
)

In [45]:
# Évaluer GraphSAGE 2 couches

for t in range(
    VAL_START,
    VAL_END + 1,
):

    step_start, step_end = (
        get_step_bounds(t)
    )


    if (
        step_start
        != depth2_current_history_end
    ):
        raise RuntimeError(
            f"Rupture Validation "
            f"au step {t}."
        )


    step_src = np.asarray(
        src[
            step_start:step_end
        ],
        dtype=np.int32,
    )

    step_dst = np.asarray(
        dst[
            step_start:step_end
        ],
        dtype=np.int32,
    )

    step_labels = np.asarray(
        label[
            step_start:step_end
        ],
        dtype=np.int8,
    )


    step_table = (
        read_core_edge_range_cached(
            step_start,
            step_end,
        )
    )


    step_features, unknown_types = (
        transform_core_edge_batch(
            step_table
        )
    )


    if unknown_types != 0:
        raise RuntimeError(
            f"Type inconnu "
            f"au step {t}."
        )


    step_results = (
        predict_causal_step_cpu(
            model=depth2_model,
            query_src=step_src,
            query_dst=step_dst,
            query_features=step_features,
            batch_size=FULL_BATCH_SIZE,
            num_hops=DEPTH2_NUM_HOPS,
        )
    )


    step_probabilities = (
        step_results[
            "probabilities"
        ]
    )


    step_size = (
        step_end
        - step_start
    )


    output_start = (
        depth2_val_cursor
    )

    output_end = (
        output_start
        + step_size
    )


    depth2_val_probabilities[
        output_start:
        output_end
    ] = step_probabilities


    depth2_val_labels[
        output_start:
        output_end
    ] = step_labels


    depth2_val_cursor = (
        output_end
    )


    # Intégrer après prédiction

    integrate_event_range(
        step_start,
        step_end,
    )

    depth2_current_history_end = (
        step_end
    )


    if (
        t == VAL_START
        or t % 10 == 0
        or t == VAL_END
    ):
        elapsed = (
            time.perf_counter()
            - depth2_validation_start
        )

        print(
            f"Depth2 Val {t:3d}/{VAL_END}"
            f" | tx={step_size:,}"
            f" | fraud={int(step_labels.sum())}"
            f" | mean={step_probabilities.mean():.6f}"
            f" | total={elapsed:.1f}s"
        )


    del (
        step_src,
        step_dst,
        step_labels,
        step_table,
        step_features,
        step_results,
        step_probabilities,
    )

Depth2 Val 335/392 | tx=10,456 | fraud=4 | mean=0.991222 | total=13.9s
Depth2 Val 340/392 | tx=10 | fraud=10 | mean=0.998282 | total=14.2s
Depth2 Val 350/392 | tx=28,245 | fraud=14 | mean=0.992348 | total=25.1s
Depth2 Val 360/392 | tx=3,508 | fraud=8 | mean=0.991353 | total=39.5s
Depth2 Val 370/392 | tx=25,429 | fraud=18 | mean=0.992323 | total=42.8s
Depth2 Val 380/392 | tx=37,693 | fraud=14 | mean=0.992214 | total=70.2s
Depth2 Val 390/392 | tx=12 | fraud=12 | mean=0.998770 | total=72.8s
Depth2 Val 392/392 | tx=10 | fraud=10 | mean=0.998427 | total=72.8s


In [46]:
# Comparer les profondeurs

depth2_weights_unchanged = all(
    torch.equal(
        depth2_weights_before[
            name
        ],
        parameter
        .detach()
        .cpu(),
    )

    for name, parameter
    in depth2_model.state_dict().items()
)


depth2_val_pr_auc = (
    average_precision_score(
        depth2_val_labels,
        depth2_val_probabilities,
    )
)

depth2_val_roc_auc = (
    roc_auc_score(
        depth2_val_labels,
        depth2_val_probabilities,
    )
)


print(
    "Poids 2 couches inchangés :",
    depth2_weights_unchanged,
)

print(
    "Validation complète :",
    depth2_val_cursor
    == VAL_ROWS,
)

print(
    "Fraudes Validation :",
    int(
        depth2_val_labels.sum()
    ),
)

print()

print(
    f"1 couche PR-AUC : "
    f"{val_pr_auc:.6f}"
)

print(
    f"2 couches PR-AUC : "
    f"{depth2_val_pr_auc:.6f}"
)

print()

print(
    f"1 couche ROC-AUC : "
    f"{val_roc_auc:.6f}"
)

print(
    f"2 couches ROC-AUC : "
    f"{depth2_val_roc_auc:.6f}"
)


if (
    depth2_val_pr_auc
    > val_pr_auc
):

    SELECTED_NUM_LAYERS = 2
    SELECTED_NUM_HOPS = 2

    selected_model = (
        depth2_model
    )

    selected_val_probabilities = (
        depth2_val_probabilities
    )

    selected_val_pr_auc = (
        depth2_val_pr_auc
    )

    selected_val_roc_auc = (
        depth2_val_roc_auc
    )

else:

    SELECTED_NUM_LAYERS = 1
    SELECTED_NUM_HOPS = 1

    selected_model = (
        full_model
    )

    selected_val_probabilities = (
        val_probabilities
    )

    selected_val_pr_auc = (
        val_pr_auc
    )

    selected_val_roc_auc = (
        val_roc_auc
    )


print()
print(
    "Profondeur sélectionnée :",
    SELECTED_NUM_LAYERS,
)

print(
    f"PR-AUC sélectionnée : "
    f"{selected_val_pr_auc:.6f}"
)

Poids 2 couches inchangés : True
Validation complète : True
Fraudes Validation : 638

1 couche PR-AUC : 0.240136
2 couches PR-AUC : 0.001883

1 couche ROC-AUC : 0.962366
2 couches ROC-AUC : 0.782053

Profondeur sélectionnée : 1
PR-AUC sélectionnée : 0.240136


In [47]:
# Sauvegarder les résultats 2 couches

DEPTH2_VAL_PATH = (
    GNN_ARTIFACT_DIR
    / "graphsage_core_edge_2layer_validation_predictions.npz"
)


np.savez_compressed(
    DEPTH2_VAL_PATH,
    probabilities=(
        depth2_val_probabilities
    ),
    labels=(
        depth2_val_labels
    ),
)


print(
    "Prédictions 2 couches sauvegardées :",
    DEPTH2_VAL_PATH.exists(),
)

Prédictions 2 couches sauvegardées : True


### GraphSAGE depth ablation

La profondeur du message passing a été évaluée exclusivement sur la période de Validation en comparant les architectures GraphSAGE à une et deux couches.

La variante à deux couches a été entraînée de zéro avec le même jeu de features `Graph Core Edge`, le même `pos_weight`, le même optimiseur et le même protocole causal que la variante à une couche. Seules la profondeur du réseau et la profondeur du contexte local ont été modifiées.

Le modèle à une couche obtient une PR-AUC de 0,240136 et une ROC-AUC de 0,962366 sur Validation. La variante à deux couches obtient une PR-AUC de seulement 0,001883 et une ROC-AUC de 0,782053.

L'ajout d'un second niveau de message passing dégrade donc fortement les performances tout en augmentant le coût de calcul. Cette observation est cohérente avec l'analyse structurelle du graphe PaySim : le deuxième hop ajoute très peu de nouveau contexte relationnel en raison du faible chevauchement entre rôles source et destination et de la faible profondeur des composantes transactionnelles.

La variante GraphSAGE à une couche est ainsi sélectionnée définitivement sur la base de la PR-AUC de Validation. Le Test n'a pas été utilisé pour cette sélection.

## 14. Validation Threshold Selection and Error Profile

Après sélection de l'architecture GraphSAGE à une couche, cette section détermine le seuil de classification à partir des seules prédictions de Validation.

Le seuil est choisi en maximisant le F1-score, afin de rechercher un compromis entre Precision et Recall dans un contexte de déséquilibre extrême des classes.

Cette sélection est effectuée une seule fois sur les steps 335 à 392. Le seuil obtenu sera ensuite figé et appliqué sans modification au Test.

Les performances sont également analysées séparément pour les transactions `CASH_OUT` et `TRANSFER`, les deux types contenant les fraudes PaySim.

In [48]:
# Vérifier la sélection

selected_val_labels = val_labels

print(
    "Couches sélectionnées :",
    SELECTED_NUM_LAYERS,
)

print(
    "Hops sélectionnés :",
    SELECTED_NUM_HOPS,
)

print(
    "Transactions Validation :",
    f"{len(selected_val_probabilities):,}",
)

print(
    "Fraudes Validation :",
    int(
        selected_val_labels.sum()
    ),
)

print(
    f"PR-AUC sélectionnée : "
    f"{selected_val_pr_auc:.6f}"
)

Couches sélectionnées : 1
Hops sélectionnés : 1
Transactions Validation : 792,406
Fraudes Validation : 638
PR-AUC sélectionnée : 0.240136


In [49]:
# Sélectionner le seuil sur Validation

from sklearn.metrics import (
    precision_recall_curve,
)


precision_curve, recall_curve, thresholds = (
    precision_recall_curve(
        selected_val_labels,
        selected_val_probabilities,
    )
)


f1_curve = (
    2
    * precision_curve[:-1]
    * recall_curve[:-1]
    / (
        precision_curve[:-1]
        + recall_curve[:-1]
        + 1e-12
    )
)


best_threshold_index = int(
    np.nanargmax(
        f1_curve
    )
)


selected_threshold = float(
    thresholds[
        best_threshold_index
    ]
)

selected_val_f1 = float(
    f1_curve[
        best_threshold_index
    ]
)

selected_val_precision = float(
    precision_curve[
        best_threshold_index
    ]
)

selected_val_recall = float(
    recall_curve[
        best_threshold_index
    ]
)


print(
    f"Seuil Validation : "
    f"{selected_threshold:.10f}"
)

print(
    f"Precision Validation : "
    f"{selected_val_precision:.6f}"
)

print(
    f"Recall Validation : "
    f"{selected_val_recall:.6f}"
)

print(
    f"F1 Validation : "
    f"{selected_val_f1:.6f}"
)

Seuil Validation : 0.9998995066
Precision Validation : 0.706215
Recall Validation : 0.195925
F1 Validation : 0.306748


In [50]:
# Calculer les prédictions binaires

from sklearn.metrics import (
    confusion_matrix,
    precision_score,
    recall_score,
    f1_score,
)


val_predictions = (
    selected_val_probabilities
    >= selected_threshold
).astype(
    np.int8
)


tn, fp, fn, tp = (
    confusion_matrix(
        selected_val_labels,
        val_predictions,
        labels=[0, 1],
    )
    .ravel()
)


val_precision_threshold = (
    precision_score(
        selected_val_labels,
        val_predictions,
        zero_division=0,
    )
)

val_recall_threshold = (
    recall_score(
        selected_val_labels,
        val_predictions,
        zero_division=0,
    )
)

val_f1_threshold = (
    f1_score(
        selected_val_labels,
        val_predictions,
        zero_division=0,
    )
)


print(
    "TN :",
    f"{tn:,}",
)

print(
    "FP :",
    f"{fp:,}",
)

print(
    "FN :",
    f"{fn:,}",
)

print(
    "TP :",
    f"{tp:,}",
)

print()

print(
    f"Precision : "
    f"{val_precision_threshold:.6f}"
)

print(
    f"Recall : "
    f"{val_recall_threshold:.6f}"
)

print(
    f"F1 : "
    f"{val_f1_threshold:.6f}"
)

TN : 791,716
FP : 52
FN : 513
TP : 125

Precision : 0.706215
Recall : 0.195925
F1 : 0.306748


In [51]:
# Comparer au seuil 0.5

predictions_05 = (
    selected_val_probabilities
    >= 0.5
).astype(
    np.int8
)


tn05, fp05, fn05, tp05 = (
    confusion_matrix(
        selected_val_labels,
        predictions_05,
        labels=[0, 1],
    )
    .ravel()
)


print(
    "Seuil 0.5"
)

print(
    "TN :",
    f"{tn05:,}",
)

print(
    "FP :",
    f"{fp05:,}",
)

print(
    "FN :",
    f"{fn05:,}",
)

print(
    "TP :",
    f"{tp05:,}",
)

print()

print(
    "Part prédite fraude :",
    f"{predictions_05.mean():.4%}",
)

Seuil 0.5
TN : 0
FP : 791,768
FN : 0
TP : 638

Part prédite fraude : 100.0000%


In [52]:
# Analyser les types frauduleux

from sklearn.metrics import (
    average_precision_score,
)


code_cash_out = (
    type_to_code["CASH_OUT"]
)

code_transfer = (
    type_to_code["TRANSFER"]
)


def evaluate_validation_type(
    type_name,
    type_code,
):
    mask = (
        val_types
        == type_code
    )

    y_true = (
        selected_val_labels[
            mask
        ]
    )

    y_score = (
        selected_val_probabilities[
            mask
        ]
    )

    y_pred = (
        y_score
        >= selected_threshold
    ).astype(
        np.int8
    )


    result = {
        "type": type_name,
        "transactions": int(
            mask.sum()
        ),
        "frauds": int(
            y_true.sum()
        ),
        "precision": float(
            precision_score(
                y_true,
                y_pred,
                zero_division=0,
            )
        ),
        "recall": float(
            recall_score(
                y_true,
                y_pred,
                zero_division=0,
            )
        ),
        "f1": float(
            f1_score(
                y_true,
                y_pred,
                zero_division=0,
            )
        ),
    }


    if (
        np.unique(
            y_true
        ).size == 2
    ):
        result[
            "pr_auc"
        ] = float(
            average_precision_score(
                y_true,
                y_score,
            )
        )

    else:
        result[
            "pr_auc"
        ] = np.nan


    return result


cash_out_val = (
    evaluate_validation_type(
        "CASH_OUT",
        code_cash_out,
    )
)

transfer_val = (
    evaluate_validation_type(
        "TRANSFER",
        code_transfer,
    )
)


for result in [
    cash_out_val,
    transfer_val,
]:

    print(
        result["type"]
    )

    print(
        "Transactions :",
        f"{result['transactions']:,}",
    )

    print(
        "Fraudes :",
        result["frauds"],
    )

    print(
        f"PR-AUC : "
        f"{result['pr_auc']:.6f}"
    )

    print(
        f"Precision : "
        f"{result['precision']:.6f}"
    )

    print(
        f"Recall : "
        f"{result['recall']:.6f}"
    )

    print(
        f"F1 : "
        f"{result['f1']:.6f}"
    )

    print()

CASH_OUT
Transactions : 274,426
Fraudes : 319
PR-AUC : 0.236381
Precision : 0.787500
Recall : 0.197492
F1 : 0.315789

TRANSFER
Transactions : 66,656
Fraudes : 319
PR-AUC : 0.258063
Precision : 0.756098
Recall : 0.194357
F1 : 0.309227



In [6]:
# Sauvegarder la sélection Validation

VALIDATION_SELECTION_PATH = (
    GNN_ARTIFACT_DIR
    / "graphsage_core_edge_model_selection.json"
)


validation_selection = {
    "selected_num_layers": (
        SELECTED_NUM_LAYERS
    ),
    "selected_num_hops": (
        SELECTED_NUM_HOPS
    ),
    "selection_metric": (
        "validation_pr_auc"
    ),
    "validation_pr_auc": (
        float(
            selected_val_pr_auc
        )
    ),
    "validation_roc_auc": (
        float(
            selected_val_roc_auc
        )
    ),
    "threshold_selection": (
        "max_validation_f1"
    ),
    "threshold": (
        selected_threshold
    ),
    "precision": (
        float(
            val_precision_threshold
        )
    ),
    "recall": (
        float(
            val_recall_threshold
        )
    ),
    "f1": (
        float(
            val_f1_threshold
        )
    ),
    "tp": int(tp),
    "fp": int(fp),
    "fn": int(fn),
    "tn": int(tn),
}


with open(
    VALIDATION_SELECTION_PATH,
    "w",
    encoding="utf-8",
) as file:

    json.dump(
        validation_selection,
        file,
        indent=2,
    )


print(
    "Sélection sauvegardée :",
    VALIDATION_SELECTION_PATH.exists(),
)

print(
    "Test utilisé :",
    False,
)

NameError: name 'GNN_ARTIFACT_DIR' is not defined

In [10]:
# Recharger la sélection depuis le disque

import json
from pathlib import Path
import numpy as np


ROOT = Path("..")

GNN_ARTIFACT_DIR = (
    ROOT
    / "artifacts"
    / "gnn"
)

VALIDATION_SELECTION_PATH = (
    GNN_ARTIFACT_DIR
    / "graphsage_core_edge_model_selection.json"
)


print(
    "Fichier sélection trouvé :",
    VALIDATION_SELECTION_PATH.exists(),
)

print(
    "Chemin :",
    VALIDATION_SELECTION_PATH.resolve(),
)


with open(
    VALIDATION_SELECTION_PATH,
    "r",
    encoding="utf-8",
) as file:
    saved_selection = json.load(file)


SELECTED_NUM_LAYERS = int(
    saved_selection[
        "selected_num_layers"
    ]
)

SELECTED_NUM_HOPS = int(
    saved_selection[
        "selected_num_hops"
    ]
)

selected_threshold = float(
    saved_selection[
        "threshold"
    ]
)


threshold_valid = (
    SELECTED_NUM_LAYERS == 1
    and SELECTED_NUM_HOPS == 1
    and np.isfinite(
        selected_threshold
    )
    and 0 <= selected_threshold <= 1
    and np.isfinite(
        saved_selection[
            "f1"
        ]
    )
)


print()
print(
    "Sélection Validation valide :",
    bool(
        threshold_valid
    ),
)

print(
    "Couches figées :",
    SELECTED_NUM_LAYERS,
)

print(
    "Hops figés :",
    SELECTED_NUM_HOPS,
)

print(
    f"Seuil figé pour Test : "
    f"{selected_threshold:.10f}"
)

print(
    f"PR-AUC Validation : "
    f"{saved_selection['validation_pr_auc']:.6f}"
)

print(
    f"F1 Validation : "
    f"{saved_selection['f1']:.6f}"
)

print(
    "Test encore inutilisé :",
    True,
)


if threshold_valid:

    print(
        "SECTION 14 : "
        "modèle et seuil figés."
    )

Fichier sélection trouvé : True
Chemin : C:\Users\bouza\OneDrive\Bureau\MobileMoney-Fraud-GNN\artifacts\gnn\graphsage_core_edge_model_selection.json

Sélection Validation valide : True
Couches figées : 1
Hops figés : 1
Seuil figé pour Test : 0.9998995066
PR-AUC Validation : 0.240136
F1 Validation : 0.306748
Test encore inutilisé : True
SECTION 14 : modèle et seuil figés.


### Validation threshold selection

Après sélection de l'architecture GraphSAGE à une couche, le seuil de décision a été optimisé exclusivement sur les prédictions de Validation en maximisant le F1-score.

Le seuil sélectionné est de 0,9998995. Cette valeur très élevée s'explique par la forte concentration des probabilités du modèle vers 1. À un seuil conventionnel de 0,5, l'ensemble des transactions de Validation serait classé comme frauduleux, confirmant que les probabilités du modèle sont fortement mal calibrées.

Au seuil sélectionné, le modèle atteint une Precision de 0,7062, un Recall de 0,1959 et un F1-score de 0,3067. La matrice de confusion contient 125 vrais positifs, 52 faux positifs et 513 faux négatifs.

Les performances sont similaires sur les deux types contenant des fraudes. `CASH_OUT` obtient une PR-AUC de 0,2364 et un Recall de 0,1975, tandis que `TRANSFER` obtient une PR-AUC de 0,2581 et un Recall de 0,1944.

Le modèle GraphSAGE 1-layer et le seuil de 0,9998995 sont désormais figés. Aucun choix supplémentaire de modèle, d'hyperparamètre ou de seuil ne sera réalisé à partir du Test.

## 15. Final Causal Test Evaluation - GraphSAGE 1-Layer

Cette section réalise l'évaluation finale du modèle GraphSAGE sélectionné sur la période Test, correspondant aux steps 393 à 743.

L'architecture, les hyperparamètres et le seuil de classification ont été définitivement sélectionnés à partir du Train et de la Validation avant toute utilisation du Test. Le modèle retenu utilise une couche GraphSAGE, un contexte à un hop et les features `Graph Core Edge`.

Les poids du modèle restent entièrement figés pendant le Test. Pour chaque step `t`, les transactions sont prédites exclusivement à partir du contexte historique `G_<t`. La topologie de `E_t` n'est intégrée qu'après prédiction de l'ensemble du step.

Aucune mise à jour des paramètres, aucun ajustement du seuil et aucune nouvelle sélection de modèle ne sont autorisés à partir des résultats Test.

Les prédictions finales sont conservées afin de calculer la PR-AUC, la ROC-AUC, les métriques au seuil figé et les analyses par type de transaction et par situation de cold-start.

In [11]:
# Réhydrater l'évaluation finale

import gc
import json
import time
from pathlib import Path
from collections import OrderedDict

import numpy as np
import pyarrow as pa
import pyarrow.parquet as pq
import torch
import torch.nn as nn
import torch.nn.functional as F

from torch_geometric.nn import SAGEConv


ROOT = Path("..")

GRAPH_DIR = (
    ROOT / "data" / "graph"
)

PROCESSED_DIR = (
    ROOT / "data" / "processed"
)

GNN_ARTIFACT_DIR = (
    ROOT / "artifacts" / "gnn"
)


device = torch.device("cpu")


src = np.load(
    GRAPH_DIR / "paysim_src.npy",
    mmap_mode="r",
)

dst = np.load(
    GRAPH_DIR / "paysim_dst.npy",
    mmap_mode="r",
)

step = np.load(
    GRAPH_DIR / "paysim_step.npy",
    mmap_mode="r",
)

label = np.load(
    GRAPH_DIR / "paysim_label.npy",
    mmap_mode="r",
)

step_offsets = np.load(
    GRAPH_DIR / "paysim_step_offsets.npy",
    mmap_mode="r",
)


num_events = len(src)
num_nodes = 9_073_900


TRAIN_END = 334

VAL_START = 335
VAL_END = 392

TEST_START = 393
TEST_END = 743


TRAIN_ROWS = int(
    step_offsets[TRAIN_END]
)

TEST_GLOBAL_START = int(
    step_offsets[
        TEST_START - 1
    ]
)

TEST_GLOBAL_END = int(
    step_offsets[
        TEST_END
    ]
)

TEST_ROWS = (
    TEST_GLOBAL_END
    - TEST_GLOBAL_START
)


def get_step_bounds(t):
    return (
        int(
            step_offsets[t - 1]
        ),
        int(
            step_offsets[t]
        ),
    )


print(
    "Device :",
    device,
)

print(
    "Début Test :",
    f"{TEST_GLOBAL_START:,}",
)

print(
    "Fin Test :",
    f"{TEST_GLOBAL_END:,}",
)

print(
    "Transactions Test :",
    f"{TEST_ROWS:,}",
)

print(
    "Fraudes Test :",
    int(
        label[
            TEST_GLOBAL_START:
            TEST_GLOBAL_END
        ].sum(
            dtype=np.int64
        )
    ),
)

c:\Users\bouza\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Device : cpu
Début Test : 5,563,793
Fin Test : 6,362,620
Transactions Test : 798,827
Fraudes Test : 3830


In [12]:
# Restaurer Graph Core Edge

graph_ready_parquet = pq.ParquetFile(
    PROCESSED_DIR
    / "paysim_graph_ready.parquet"
)


core_edge_features = [
    "amount",
    "log_amount",
    "type",
    "origin_balance_change",
    "destination_balance_change",
    "origin_zero_before",
    "origin_zero_after",
    "destination_zero_before",
    "destination_zero_after",
]


continuous_features = [
    "amount",
    "log_amount",
    "origin_balance_change",
    "destination_balance_change",
]


binary_features = [
    "origin_zero_before",
    "origin_zero_after",
    "destination_zero_before",
    "destination_zero_after",
]


type_categories = [
    "CASH_IN",
    "CASH_OUT",
    "DEBIT",
    "PAYMENT",
    "TRANSFER",
]


type_to_index = {
    category: index
    for index, category
    in enumerate(
        type_categories
    )
}


train_scaler = {
    "amount": {
        "mean": 179748.531939,
        "std": 648694.425604,
    },
    "log_amount": {
        "mean": 10.833058,
        "std": 1.820169,
    },
    "origin_balance_change": {
        "mean": -21731.548861,
        "std": 128793.638313,
    },
    "destination_balance_change": {
        "mean": 130154.927112,
        "std": 900249.394962,
    },
}


core_edge_input_dim = 13


# Construire les offsets Parquet

num_row_groups = (
    graph_ready_parquet
    .metadata
    .num_row_groups
)

row_group_offsets = np.zeros(
    num_row_groups + 1,
    dtype=np.int64,
)


for row_group_id in range(
    num_row_groups
):

    row_group_offsets[
        row_group_id + 1
    ] = (
        row_group_offsets[
            row_group_id
        ]
        + graph_ready_parquet
        .metadata
        .row_group(
            row_group_id
        )
        .num_rows
    )


# Cache léger

core_edge_cache = OrderedDict()

core_edge_cache_stats = {
    "physical_reads": 0,
    "cache_hits": 0,
}


def get_core_edge_row_group(
    row_group_id,
):
    if row_group_id in core_edge_cache:

        core_edge_cache_stats[
            "cache_hits"
        ] += 1

        return core_edge_cache[
            row_group_id
        ]


    table = (
        graph_ready_parquet
        .read_row_group(
            row_group_id,
            columns=core_edge_features,
            use_threads=True,
        )
    )

    core_edge_cache_stats[
        "physical_reads"
    ] += 1

    core_edge_cache.clear()

    core_edge_cache[
        row_group_id
    ] = table

    return table


def read_core_edge_range_cached(
    start_index,
    end_index,
):
    first_group = int(
        np.searchsorted(
            row_group_offsets,
            start_index,
            side="right",
        )
        - 1
    )

    last_group = int(
        np.searchsorted(
            row_group_offsets,
            end_index - 1,
            side="right",
        )
        - 1
    )


    tables = []


    for row_group_id in range(
        first_group,
        last_group + 1,
    ):

        table = (
            get_core_edge_row_group(
                row_group_id
            )
        )


        group_start = int(
            row_group_offsets[
                row_group_id
            ]
        )

        group_end = int(
            row_group_offsets[
                row_group_id + 1
            ]
        )


        local_start = (
            max(
                start_index,
                group_start,
            )
            - group_start
        )

        local_end = (
            min(
                end_index,
                group_end,
            )
            - group_start
        )


        tables.append(
            table.slice(
                local_start,
                local_end - local_start,
            )
        )


    if len(tables) == 1:
        return tables[0]

    return pa.concat_tables(
        tables
    )

In [13]:
# Transformer les features

def transform_core_edge_batch(
    batch,
):
    parts = []


    for feature in continuous_features:

        values = (
            batch[feature]
            .combine_chunks()
            .to_numpy(
                zero_copy_only=False
            )
            .astype(
                np.float32,
                copy=False,
            )
        )

        mean = train_scaler[
            feature
        ]["mean"]

        std = train_scaler[
            feature
        ]["std"]

        parts.append(
            (
                (values - mean)
                / std
            ).reshape(-1, 1)
        )


    for feature in binary_features:

        values = (
            batch[feature]
            .combine_chunks()
            .to_numpy(
                zero_copy_only=False
            )
            .astype(
                np.float32,
                copy=False,
            )
        )

        parts.append(
            values.reshape(-1, 1)
        )


    type_values = (
        batch["type"]
        .combine_chunks()
        .to_pylist()
    )


    one_hot = np.zeros(
        (
            batch.num_rows,
            len(type_categories),
        ),
        dtype=np.float32,
    )

    unknown_types = 0


    for row_index, category in enumerate(
        type_values
    ):

        category_index = (
            type_to_index.get(
                category
            )
        )

        if category_index is None:

            unknown_types += 1
            continue

        one_hot[
            row_index,
            category_index
        ] = 1.0


    parts.append(
        one_hot
    )


    return (
        np.concatenate(
            parts,
            axis=1,
        ).astype(
            np.float32,
            copy=False,
        ),
        unknown_types,
    )


print(
    "Row groups :",
    num_row_groups,
)

print(
    "Dimension Core Edge :",
    core_edge_input_dim,
)

Row groups : 26
Dimension Core Edge : 13


In [14]:
# Restaurer le modèle final

class GraphSAGEEdgeClassifier(
    nn.Module
):

    def __init__(
        self,
        edge_feature_dim,
        hidden_dim=32,
        num_layers=1,
        dropout=0.10,
    ):
        super().__init__()

        self.num_layers = (
            num_layers
        )

        self.dropout = (
            dropout
        )


        self.conv1 = SAGEConv(
            in_channels=1,
            out_channels=hidden_dim,
            aggr="sum",
            normalize=False,
        )


        if num_layers == 2:

            self.conv2 = SAGEConv(
                in_channels=hidden_dim,
                out_channels=hidden_dim,
                aggr="sum",
                normalize=False,
            )


        classifier_input_dim = (
            2 * hidden_dim
            + edge_feature_dim
        )


        self.classifier = (
            nn.Sequential(
                nn.Linear(
                    classifier_input_dim,
                    hidden_dim,
                ),
                nn.ReLU(),
                nn.Dropout(
                    dropout
                ),
                nn.Linear(
                    hidden_dim,
                    1,
                ),
            )
        )


    def forward(
        self,
        num_nodes,
        context_edge_index,
        query_edge_index,
        edge_features,
    ):

        x = torch.ones(
            (
                num_nodes,
                1,
            ),
            dtype=torch.float32,
            device=(
                context_edge_index
                .device
            ),
        )


        h = F.relu(
            self.conv1(
                x,
                context_edge_index,
            )
        )


        if self.num_layers == 2:

            h = F.dropout(
                h,
                p=self.dropout,
                training=self.training,
            )

            h = F.relu(
                self.conv2(
                    h,
                    context_edge_index,
                )
            )


        h_src = h[
            query_edge_index[0]
        ]

        h_dst = h[
            query_edge_index[1]
        ]


        edge_repr = torch.cat(
            [
                h_src,
                h_dst,
                edge_features,
            ],
            dim=1,
        )


        return (
            self.classifier(
                edge_repr
            )
            .squeeze(-1)
        )

In [15]:
# Charger modèle et sélection

MODEL_PATH = (
    GNN_ARTIFACT_DIR
    / "graphsage_core_edge_1layer_train_final.pt"
)

SELECTION_PATH = (
    GNN_ARTIFACT_DIR
    / "graphsage_core_edge_model_selection.json"
)


with open(
    SELECTION_PATH,
    "r",
    encoding="utf-8",
) as file:
    selection = json.load(file)


selected_threshold = float(
    selection["threshold"]
)

selected_num_layers = int(
    selection[
        "selected_num_layers"
    ]
)

selected_num_hops = int(
    selection[
        "selected_num_hops"
    ]
)


test_model = (
    GraphSAGEEdgeClassifier(
        edge_feature_dim=13,
        hidden_dim=32,
        num_layers=(
            selected_num_layers
        ),
        dropout=0.10,
    )
    .to(device)
)


checkpoint = torch.load(
    MODEL_PATH,
    map_location=device,
    weights_only=False,
)


test_model.load_state_dict(
    checkpoint[
        "model_state_dict"
    ]
)

test_model.eval()


print(
    "Modèle trouvé :",
    MODEL_PATH.exists(),
)

print(
    "Couches :",
    selected_num_layers,
)

print(
    "Hops :",
    selected_num_hops,
)

print(
    f"Seuil figé : "
    f"{selected_threshold:.10f}"
)

print(
    "Mode entraînement :",
    test_model.training,
)

Modèle trouvé : True
Couches : 1
Hops : 1
Seuil figé : 0.9998995066
Mode entraînement : False


In [16]:
# Reconstruire G_<393

head_event = np.full(
    num_nodes,
    -1,
    dtype=np.int32,
)

prev_in_event = np.full(
    num_events,
    -1,
    dtype=np.int32,
)


def integrate_event_range(
    start_index,
    end_index,
):
    dst_values = np.asarray(
        dst[
            start_index:
            end_index
        ],
        dtype=np.int32,
    )


    for offset, destination in enumerate(
        dst_values
    ):

        event_index = (
            start_index
            + offset
        )


        prev_in_event[
            event_index
        ] = head_event[
            destination
        ]

        head_event[
            destination
        ] = event_index


def get_inbound_events(
    node_id,
    max_neighbors=None,
):
    events = []

    event_index = int(
        head_event[
            node_id
        ]
    )


    while event_index != -1:

        events.append(
            event_index
        )

        if (
            max_neighbors
            is not None
            and len(events)
            >= max_neighbors
        ):
            break

        event_index = int(
            prev_in_event[
                event_index
            ]
        )


    return np.asarray(
        events,
        dtype=np.int32,
    )

In [17]:
# Rejouer Train + Validation

gc.collect()

replay_start = (
    time.perf_counter()
)


integrate_event_range(
    0,
    TEST_GLOBAL_START,
)


test_history_end = (
    TEST_GLOBAL_START
)


replay_seconds = (
    time.perf_counter()
    - replay_start
)


print(
    "Événements historiques :",
    f"{TEST_GLOBAL_START:,}",
)

print(
    f"Temps replay : "
    f"{replay_seconds:.2f} s"
)

print(
    "Dernier event historique :",
    int(
        head_event.max()
    ),
)

print(
    "Contexte strictement causal :",
    int(
        head_event.max()
    )
    < TEST_GLOBAL_START,
)

Événements historiques : 5,563,793
Temps replay : 4.09 s
Dernier event historique : 5563792
Contexte strictement causal : True


In [18]:
# Construire l'état Known/New

seen_node = np.zeros(
    num_nodes,
    dtype=np.bool_,
)


seen_node[
    np.asarray(
        src[
            :TEST_GLOBAL_START
        ],
        dtype=np.int32,
    )
] = True

seen_node[
    np.asarray(
        dst[
            :TEST_GLOBAL_START
        ],
        dtype=np.int32,
    )
] = True


print(
    "Nœuds connus avant Test :",
    f"{int(seen_node.sum()):,}",
)

print(
    f"Mémoire seen_node : "
    f"{seen_node.nbytes / 1024**2:.2f} MB"
)

Nœuds connus avant Test : 7,930,963
Mémoire seen_node : 8.65 MB


In [19]:
# Extraire un sous-graphe causal

def extract_causal_subgraph(
    query_src_nodes,
    query_dst_nodes,
    num_hops=1,
):
    query_src_nodes = np.asarray(
        query_src_nodes,
        dtype=np.int32,
    )

    query_dst_nodes = np.asarray(
        query_dst_nodes,
        dtype=np.int32,
    )


    query_nodes = np.unique(
        np.concatenate([
            query_src_nodes,
            query_dst_nodes,
        ])
    )


    frontier = set(
        query_nodes.tolist()
    )

    expanded_nodes = set()
    context_event_ids = set()


    for _ in range(
        num_hops
    ):

        next_frontier = set()


        for node_id in frontier:

            if node_id in expanded_nodes:
                continue

            expanded_nodes.add(
                node_id
            )


            inbound_events = (
                get_inbound_events(
                    node_id
                )
            )


            for event_id in inbound_events:

                event_id = int(
                    event_id
                )

                context_event_ids.add(
                    event_id
                )

                next_frontier.add(
                    int(
                        src[
                            event_id
                        ]
                    )
                )


        frontier = (
            next_frontier
        )

        if not frontier:
            break


    context_event_ids = np.asarray(
        sorted(
            context_event_ids
        ),
        dtype=np.int32,
    )


    if len(
        context_event_ids
    ) > 0:

        context_src = np.asarray(
            src[
                context_event_ids
            ],
            dtype=np.int32,
        )

        context_dst = np.asarray(
            dst[
                context_event_ids
            ],
            dtype=np.int32,
        )


        local_nodes = np.unique(
            np.concatenate([
                query_src_nodes,
                query_dst_nodes,
                context_src,
                context_dst,
            ])
        )


    else:

        context_src = np.empty(
            0,
            dtype=np.int32,
        )

        context_dst = np.empty(
            0,
            dtype=np.int32,
        )

        local_nodes = (
            query_nodes
        )


    return {
        "local_nodes": local_nodes,

        "context_edge_index": (
            torch.tensor(
                np.stack([
                    np.searchsorted(
                        local_nodes,
                        context_src,
                    ),
                    np.searchsorted(
                        local_nodes,
                        context_dst,
                    ),
                ]),
                dtype=torch.long,
            )
        ),

        "query_edge_index": (
            torch.tensor(
                np.stack([
                    np.searchsorted(
                        local_nodes,
                        query_src_nodes,
                    ),
                    np.searchsorted(
                        local_nodes,
                        query_dst_nodes,
                    ),
                ]),
                dtype=torch.long,
            )
        ),
    }

In [20]:
# Prédire un step

TEST_BATCH_SIZE = 4_096


def predict_test_step(
    model,
    query_src,
    query_dst,
    query_features,
):
    num_queries = len(
        query_src
    )

    probabilities = np.empty(
        num_queries,
        dtype=np.float32,
    )


    num_batches = int(
        np.ceil(
            num_queries
            / TEST_BATCH_SIZE
        )
    )


    with torch.no_grad():

        for batch_id in range(
            num_batches
        ):

            batch_start = (
                batch_id
                * TEST_BATCH_SIZE
            )

            batch_end = min(
                batch_start
                + TEST_BATCH_SIZE,
                num_queries,
            )


            subgraph = (
                extract_causal_subgraph(
                    query_src[
                        batch_start:
                        batch_end
                    ],
                    query_dst[
                        batch_start:
                        batch_end
                    ],
                    num_hops=(
                        selected_num_hops
                    ),
                )
            )


            features_cpu = (
                torch.from_numpy(
                    query_features[
                        batch_start:
                        batch_end
                    ]
                )
            )


            logits = model(
                num_nodes=len(
                    subgraph[
                        "local_nodes"
                    ]
                ),
                context_edge_index=(
                    subgraph[
                        "context_edge_index"
                    ]
                ),
                query_edge_index=(
                    subgraph[
                        "query_edge_index"
                    ]
                ),
                edge_features=(
                    features_cpu
                ),
            )


            probabilities[
                batch_start:
                batch_end
            ] = (
                torch.sigmoid(
                    logits
                )
                .numpy()
                .astype(
                    np.float32,
                    copy=False,
                )
            )


            del (
                subgraph,
                features_cpu,
                logits,
            )


    return probabilities

In [21]:
# Préparer les sorties Test

test_probabilities = np.empty(
    TEST_ROWS,
    dtype=np.float32,
)

test_labels = np.empty(
    TEST_ROWS,
    dtype=np.int8,
)

test_steps = np.empty(
    TEST_ROWS,
    dtype=np.int16,
)

test_types = np.empty(
    TEST_ROWS,
    dtype=np.int8,
)

test_cold_start = np.empty(
    TEST_ROWS,
    dtype=np.int8,
)


type_to_code = {
    category: index
    for index, category
    in enumerate(type_categories)
}


test_cursor = 0

test_history = []


weights_before_test = {
    name: value
    .detach()
    .cpu()
    .clone()

    for name, value
    in test_model.state_dict().items()
}

In [22]:
# Évaluer tout le Test

test_loop_start = (
    time.perf_counter()
)


for t in range(
    TEST_START,
    TEST_END + 1,
):

    step_start, step_end = (
        get_step_bounds(t)
    )


    if (
        step_start
        != test_history_end
    ):

        raise RuntimeError(
            f"Rupture causale "
            f"au Test step {t}."
        )


    step_src = np.asarray(
        src[
            step_start:
            step_end
        ],
        dtype=np.int32,
    )

    step_dst = np.asarray(
        dst[
            step_start:
            step_end
        ],
        dtype=np.int32,
    )


    step_labels = np.asarray(
        label[
            step_start:
            step_end
        ],
        dtype=np.int8,
    )


    # Known/New avant intégration

    src_known = (
        seen_node[
            step_src
        ]
    )

    dst_known = (
        seen_node[
            step_dst
        ]
    )


    step_cold_start = (
        (~src_known).astype(
            np.int8
        )
        + 2
        * (~dst_known).astype(
            np.int8
        )
    )


    step_table = (
        read_core_edge_range_cached(
            step_start,
            step_end,
        )
    )


    step_features, unknown_types = (
        transform_core_edge_batch(
            step_table
        )
    )


    if unknown_types != 0:

        raise RuntimeError(
            f"Type inconnu "
            f"au Test step {t}."
        )


    step_probabilities = (
        predict_test_step(
            model=test_model,
            query_src=step_src,
            query_dst=step_dst,
            query_features=step_features,
        )
    )


    if not np.isfinite(
        step_probabilities
    ).all():

        raise RuntimeError(
            f"Score non fini "
            f"au Test step {t}."
        )


    step_size = (
        step_end
        - step_start
    )


    output_start = (
        test_cursor
    )

    output_end = (
        output_start
        + step_size
    )


    test_probabilities[
        output_start:
        output_end
    ] = step_probabilities


    test_labels[
        output_start:
        output_end
    ] = step_labels


    test_steps[
        output_start:
        output_end
    ] = t


    step_type_values = (
        step_table[
            "type"
        ]
        .combine_chunks()
        .to_pylist()
    )


    test_types[
        output_start:
        output_end
    ] = np.fromiter(
        (
            type_to_code[
                value
            ]
            for value
            in step_type_values
        ),
        dtype=np.int8,
        count=step_size,
    )


    test_cold_start[
        output_start:
        output_end
    ] = step_cold_start


    test_cursor = (
        output_end
    )


    # Intégrer uniquement après prédiction

    integrate_event_range(
        step_start,
        step_end,
    )


    seen_node[
        step_src
    ] = True

    seen_node[
        step_dst
    ] = True


    test_history_end = (
        step_end
    )


    test_history.append({
        "step": int(t),
        "transactions": int(
            step_size
        ),
        "frauds": int(
            step_labels.sum()
        ),
    })


    if (
        t == TEST_START
        or t % 50 == 0
        or t == TEST_END
    ):

        elapsed = (
            time.perf_counter()
            - test_loop_start
        )

        print(
            f"Test {t:3d}/{TEST_END}"
            f" | tx={step_size:,}"
            f" | fraud={int(step_labels.sum())}"
            f" | mean={step_probabilities.mean():.6f}"
            f" | total={elapsed:.1f}s"
        )


    del (
        step_src,
        step_dst,
        step_labels,
        src_known,
        dst_known,
        step_cold_start,
        step_table,
        step_features,
        step_probabilities,
        step_type_values,
    )


test_loop_seconds = (
    time.perf_counter()
    - test_loop_start
)

Test 393/743 | tx=19,269 | fraud=4 | mean=0.980170 | total=1.3s
Test 400/743 | tx=30,910 | fraud=16 | mean=0.979913 | total=8.5s
Test 450/743 | tx=16 | fraud=16 | mean=0.996835 | total=16.5s
Test 500/743 | tx=6,108 | fraud=4 | mean=0.980198 | total=18.3s
Test 550/743 | tx=4,804 | fraud=8 | mean=0.979954 | total=21.7s
Test 600/743 | tx=18 | fraud=18 | mean=0.998703 | total=24.8s
Test 650/743 | tx=16 | fraud=16 | mean=0.996392 | total=25.8s
Test 700/743 | tx=6 | fraud=6 | mean=0.995733 | total=28.1s
Test 743/743 | tx=8 | fraud=8 | mean=0.998553 | total=28.6s


In [23]:
# Vérifier le Test causal

weights_unchanged_test = all(
    torch.equal(
        weights_before_test[
            name
        ],
        value
        .detach()
        .cpu(),
    )

    for name, value
    in test_model.state_dict().items()
)


print(
    "Poids inchangés :",
    weights_unchanged_test,
)

print(
    "Transactions collectées :",
    f"{test_cursor:,}",
)

print(
    "Collecte complète :",
    test_cursor
    == TEST_ROWS,
)

print(
    "Fraudes collectées :",
    int(
        test_labels.sum(
            dtype=np.int64
        )
    ),
)

print(
    "Scores finis :",
    np.isfinite(
        test_probabilities
    ).all(),
)

print(
    "Historique final :",
    f"{test_history_end:,}",
)

print(
    f"Runtime boucle Test : "
    f"{test_loop_seconds:.2f} s"
)

Poids inchangés : True
Transactions collectées : 798,827
Collecte complète : True
Fraudes collectées : 3830
Scores finis : True
Historique final : 6,362,620
Runtime boucle Test : 28.57 s


In [24]:
# Calculer les métriques Test

from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
)


test_pr_auc = (
    average_precision_score(
        test_labels,
        test_probabilities,
    )
)


test_roc_auc = (
    roc_auc_score(
        test_labels,
        test_probabilities,
    )
)


test_predictions = (
    test_probabilities
    >= selected_threshold
).astype(
    np.int8
)


test_precision = (
    precision_score(
        test_labels,
        test_predictions,
        zero_division=0,
    )
)

test_recall = (
    recall_score(
        test_labels,
        test_predictions,
        zero_division=0,
    )
)

test_f1 = (
    f1_score(
        test_labels,
        test_predictions,
        zero_division=0,
    )
)


tn_test, fp_test, fn_test, tp_test = (
    confusion_matrix(
        test_labels,
        test_predictions,
        labels=[0, 1],
    )
    .ravel()
)


print(
    f"Test PR-AUC : "
    f"{test_pr_auc:.6f}"
)

print(
    f"Test ROC-AUC : "
    f"{test_roc_auc:.6f}"
)

print(
    f"Precision : "
    f"{test_precision:.6f}"
)

print(
    f"Recall : "
    f"{test_recall:.6f}"
)

print(
    f"F1 : "
    f"{test_f1:.6f}"
)

print()

print(
    "TP :",
    f"{tp_test:,}",
)

print(
    "FP :",
    f"{fp_test:,}",
)

print(
    "FN :",
    f"{fn_test:,}",
)

print(
    "TN :",
    f"{tn_test:,}",
)

print()

print(
    f"Seuil appliqué : "
    f"{selected_threshold:.10f}"
)

Test PR-AUC : 0.352779
Test ROC-AUC : 0.959934
Precision : 0.927941
Recall : 0.164752
F1 : 0.279823

TP : 631
FP : 49
FN : 3,199
TN : 794,948

Seuil appliqué : 0.9998995066


In [25]:
# Sauvegarder les résultats Test

TEST_PREDICTIONS_PATH = (
    GNN_ARTIFACT_DIR
    / "graphsage_core_edge_1layer_test_predictions.npz"
)


TEST_METRICS_PATH = (
    GNN_ARTIFACT_DIR
    / "graphsage_core_edge_1layer_test_metrics.json"
)


np.savez_compressed(
    TEST_PREDICTIONS_PATH,
    probabilities=(
        test_probabilities
    ),
    labels=(
        test_labels
    ),
    steps=(
        test_steps
    ),
    types=(
        test_types
    ),
    cold_start=(
        test_cold_start
    ),
)


test_metrics = {
    "pr_auc": float(
        test_pr_auc
    ),
    "roc_auc": float(
        test_roc_auc
    ),
    "threshold": float(
        selected_threshold
    ),
    "precision": float(
        test_precision
    ),
    "recall": float(
        test_recall
    ),
    "f1": float(
        test_f1
    ),
    "tp": int(tp_test),
    "fp": int(fp_test),
    "fn": int(fn_test),
    "tn": int(tn_test),
    "num_layers": int(
        selected_num_layers
    ),
    "num_hops": int(
        selected_num_hops
    ),
    "threshold_source": (
        "validation"
    ),
}


with open(
    TEST_METRICS_PATH,
    "w",
    encoding="utf-8",
) as file:

    json.dump(
        test_metrics,
        file,
        indent=2,
    )


print(
    "Prédictions Test sauvegardées :",
    TEST_PREDICTIONS_PATH.exists(),
)

print(
    "Métriques Test sauvegardées :",
    TEST_METRICS_PATH.exists(),
)

Prédictions Test sauvegardées : True
Métriques Test sauvegardées : True


In [26]:
# Valider l'évaluation finale

test_valid = (
    test_cursor
        == TEST_ROWS
    and int(
        test_labels.sum(
            dtype=np.int64
        )
    )
        == 3_830
    and weights_unchanged_test
    and test_history_end
        == TEST_GLOBAL_END
    and np.isfinite(
        test_pr_auc
    )
    and np.isfinite(
        test_roc_auc
    )
    and TEST_PREDICTIONS_PATH.exists()
    and TEST_METRICS_PATH.exists()
)


print(
    "Test causal valide :",
    bool(
        test_valid
    ),
)

print(
    "Modèle modifié sur Test :",
    False,
)

print(
    "Seuil modifié sur Test :",
    False,
)


if test_valid:

    print(
        "SECTION 15 : "
        "évaluation Test finale validée."
    )

Test causal valide : True
Modèle modifié sur Test : False
Seuil modifié sur Test : False
SECTION 15 : évaluation Test finale validée.


### Final causal Test results

Le modèle GraphSAGE 1-layer sélectionné sur Validation a été évalué sur les 798 827 transactions Test, correspondant aux steps 393 à 743 et contenant 3 830 fraudes.

L'architecture, les poids et le seuil de classification ont été entièrement figés avant l'accès au Test. Pendant l'évaluation, les paramètres sont restés inchangés et chaque step `t` a été prédit exclusivement à partir de `G_<t`, avant intégration de sa propre topologie.

Le modèle obtient une PR-AUC de 0,352779 et une ROC-AUC de 0,959934. Au seuil de 0,9998995 sélectionné sur Validation, la Precision atteint 0,927941, le Recall 0,164752 et le F1-score 0,279823.

La matrice de confusion contient 631 vrais positifs, 49 faux positifs et 3 199 faux négatifs. Le modèle produit donc peu de fausses alertes, mais ne détecte qu'environ 16,5 % des fraudes du Test.

La PR-AUC Test est supérieure à celle observée sur Validation. Cette différence doit néanmoins être interprétée avec prudence, car la prévalence de fraude augmente fortement entre les deux périodes, passant d'environ 0,0805 % à 0,4795 %.

L'évaluation finale respecte le protocole expérimental : aucun poids, seuil ou hyperparamètre n'a été modifié à partir du Test.

## 16. Error Profile, Cold-Start and Final Baseline Comparison

Cette section analyse les performances finales du GraphSAGE selon le type de transaction et la situation de cold-start des endpoints.

Les transactions Test sont séparées entre `CASH_OUT` et `TRANSFER`, les deux types contenant les fraudes PaySim.

L'analyse inductive distingue également quatre situations selon que les comptes source et destination ont déjà été observés avant le step courant : Known→Known, New→Known, Known→New et New→New.

Enfin, les performances du GraphSAGE sont comparées au baseline LightGBM Core sur les mêmes splits temporels. Cette comparaison permet d'évaluer la valeur ajoutée réelle du message passing dans la configuration structurelle particulière de PaySim.

In [27]:
# Analyser les types Test

import pandas as pd

from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
)


def evaluate_test_subset(
    mask,
):
    y_true = test_labels[mask]
    y_score = test_probabilities[mask]

    y_pred = (
        y_score
        >= selected_threshold
    ).astype(
        np.int8
    )


    tn, fp, fn, tp = (
        confusion_matrix(
            y_true,
            y_pred,
            labels=[0, 1],
        )
        .ravel()
    )


    result = {
        "transactions": int(
            mask.sum()
        ),
        "frauds": int(
            y_true.sum()
        ),
        "fraud_rate": float(
            y_true.mean()
        ),
        "precision": float(
            precision_score(
                y_true,
                y_pred,
                zero_division=0,
            )
        ),
        "recall": float(
            recall_score(
                y_true,
                y_pred,
                zero_division=0,
            )
        ),
        "f1": float(
            f1_score(
                y_true,
                y_pred,
                zero_division=0,
            )
        ),
        "tp": int(tp),
        "fp": int(fp),
        "fn": int(fn),
    }


    if (
        y_true.sum() > 0
        and y_true.sum() < len(y_true)
    ):
        result["pr_auc"] = float(
            average_precision_score(
                y_true,
                y_score,
            )
        )

        result["roc_auc"] = float(
            roc_auc_score(
                y_true,
                y_score,
            )
        )

    else:
        result["pr_auc"] = np.nan
        result["roc_auc"] = np.nan


    return result

In [28]:
# Comparer CASH_OUT et TRANSFER

type_results = []


for type_name in [
    "CASH_OUT",
    "TRANSFER",
]:

    type_code = (
        type_to_code[
            type_name
        ]
    )

    result = (
        evaluate_test_subset(
            test_types
            == type_code
        )
    )

    result["type"] = (
        type_name
    )

    type_results.append(
        result
    )


type_results_df = pd.DataFrame(
    type_results
)[[
    "type",
    "transactions",
    "frauds",
    "fraud_rate",
    "pr_auc",
    "roc_auc",
    "precision",
    "recall",
    "f1",
    "tp",
    "fp",
    "fn",
]]


print(
    type_results_df.to_string(
        index=False
    )
)

    type  transactions  frauds  fraud_rate   pr_auc  roc_auc  precision   recall       f1  tp  fp   fn
CASH_OUT        271051    1915    0.007065 0.312927 0.874025   0.931138 0.162402 0.276567 311  23 1604
TRANSFER         70824    1915    0.027039 0.427863 0.931568   0.949555 0.167102 0.284192 320  17 1595


In [30]:
# Analyser le cold-start

cold_start_names = {
    0: "Known -> Known",
    1: "New -> Known",
    2: "Known -> New",
    3: "New -> New",
}


cold_start_results = []


for cold_code, cold_name in (
    cold_start_names.items()
):

    mask = (
        test_cold_start
        == cold_code
    )


    result = (
        evaluate_test_subset(
            mask
        )
    )

    result[
        "cold_start"
    ] = cold_name

    result[
        "share"
    ] = float(
        mask.mean()
    )


    cold_start_results.append(
        result
    )


cold_start_df = pd.DataFrame(
    cold_start_results
)[[
    "cold_start",
    "transactions",
    "share",
    "frauds",
    "fraud_rate",
    "pr_auc",
    "precision",
    "recall",
    "f1",
    "tp",
    "fp",
    "fn",
]]


print(
    cold_start_df.to_string(
        index=False
    )
)

    cold_start  transactions    share  frauds  fraud_rate   pr_auc  precision   recall       f1  tp  fp   fn
Known -> Known          1310 0.001640       5    0.003817 0.096926   0.000000 0.000000 0.000000   0   0    5
  New -> Known        450490 0.563939    1403    0.003114 0.237060   0.812261 0.151105 0.254808 212  49 1191
  Known -> New          1001 0.001253       5    0.004995 0.420173   0.000000 0.000000 0.000000   0   0    5
    New -> New        346026 0.433168    2417    0.006985 0.709571   1.000000 0.173355 0.295487 419   0 1998


In [34]:
# Recharger les métriques sauvegardées

import json
from pathlib import Path
import pandas as pd
import numpy as np


ROOT = Path("..")

GNN_ARTIFACT_DIR = (
    ROOT
    / "artifacts"
    / "gnn"
)


SELECTION_PATH = (
    GNN_ARTIFACT_DIR
    / "graphsage_core_edge_model_selection.json"
)

TEST_METRICS_PATH = (
    GNN_ARTIFACT_DIR
    / "graphsage_core_edge_1layer_test_metrics.json"
)


with open(
    SELECTION_PATH,
    "r",
    encoding="utf-8",
) as file:
    saved_selection = json.load(file)


with open(
    TEST_METRICS_PATH,
    "r",
    encoding="utf-8",
) as file:
    saved_test_metrics = json.load(file)


selected_val_pr_auc = float(
    saved_selection[
        "validation_pr_auc"
    ]
)

selected_val_roc_auc = float(
    saved_selection[
        "validation_roc_auc"
    ]
)


test_pr_auc = float(
    saved_test_metrics[
        "pr_auc"
    ]
)

test_roc_auc = float(
    saved_test_metrics[
        "roc_auc"
    ]
)

test_precision = float(
    saved_test_metrics[
        "precision"
    ]
)

test_recall = float(
    saved_test_metrics[
        "recall"
    ]
)

test_f1 = float(
    saved_test_metrics[
        "f1"
    ]
)

tp_test = int(
    saved_test_metrics[
        "tp"
    ]
)

fp_test = int(
    saved_test_metrics[
        "fp"
    ]
)

fn_test = int(
    saved_test_metrics[
        "fn"
    ]
)


comparison_df = pd.DataFrame([
    {
        "model": "GraphSAGE Core Edge",
        "val_pr_auc": selected_val_pr_auc,
        "test_pr_auc": test_pr_auc,
        "test_roc_auc": test_roc_auc,
        "precision": test_precision,
        "recall": test_recall,
        "f1": test_f1,
        "tp": tp_test,
        "fp": fp_test,
        "fn": fn_test,
    },
    {
        "model": "LightGBM Core",
        "val_pr_auc": 0.929704,
        "test_pr_auc": 0.979749,
        "test_roc_auc": 0.999890,
        "precision": 0.982230,
        "recall": 0.793730,
        "f1": 0.877980,
        "tp": 3040,
        "fp": 55,
        "fn": 790,
    },
])


print(
    comparison_df.to_string(
        index=False
    )
)

              model  val_pr_auc  test_pr_auc  test_roc_auc  precision   recall       f1   tp  fp   fn
GraphSAGE Core Edge    0.240136     0.352779      0.959934   0.927941 0.164752 0.279823  631  49 3199
      LightGBM Core    0.929704     0.979749      0.999890   0.982230 0.793730 0.877980 3040  55  790


In [35]:
# Quantifier les écarts

print(
    f"Écart Test PR-AUC : "
    f"{test_pr_auc - 0.979749:+.6f}"
)

print(
    f"Écart Recall : "
    f"{test_recall - 0.793730:+.6f}"
)

print(
    f"Écart F1 : "
    f"{test_f1 - 0.877980:+.6f}"
)

print()

print(
    "Fraudes détectées GraphSAGE :",
    f"{tp_test:,}",
)

print(
    "Fraudes détectées LightGBM :",
    "3,040",
)

print(
    "Fraudes supplémentaires détectées par LightGBM :",
    f"{3040 - tp_test:,}",
)

Écart Test PR-AUC : -0.626970
Écart Recall : -0.628978
Écart F1 : -0.598157

Fraudes détectées GraphSAGE : 631
Fraudes détectées LightGBM : 3,040
Fraudes supplémentaires détectées par LightGBM : 2,409


In [36]:
# Résumer le drift Validation -> Test

VAL_ROWS = 792_406
VAL_FRAUDS = 638

TEST_ROWS = 798_827
TEST_FRAUDS = 3_830


val_fraud_rate = (
    VAL_FRAUDS
    / VAL_ROWS
)

test_fraud_rate = (
    TEST_FRAUDS
    / TEST_ROWS
)


print(
    f"Taux fraude Validation : "
    f"{val_fraud_rate:.6%}"
)

print(
    f"Taux fraude Test : "
    f"{test_fraud_rate:.6%}"
)

print(
    f"Drift prévalence Val -> Test : "
    f"x{test_fraud_rate / val_fraud_rate:.2f}"
)

print()

print(
    f"GraphSAGE Val PR-AUC : "
    f"{selected_val_pr_auc:.6f}"
)

print(
    f"GraphSAGE Test PR-AUC : "
    f"{test_pr_auc:.6f}"
)

Taux fraude Validation : 0.080514%
Taux fraude Test : 0.479453%
Drift prévalence Val -> Test : x5.95

GraphSAGE Val PR-AUC : 0.240136
GraphSAGE Test PR-AUC : 0.352779


In [37]:
# Sauvegarder les analyses

FINAL_ANALYSIS_PATH = (
    GNN_ARTIFACT_DIR
    / "graphsage_core_edge_final_analysis.json"
)


final_analysis = {
    "transaction_type_analysis": (
        type_results
    ),
    "cold_start_analysis": (
        cold_start_results
    ),
    "comparison": (
        comparison_df
        .to_dict(
            orient="records"
        )
    ),
    "validation_fraud_rate": float(
        val_fraud_rate
    ),
    "test_fraud_rate": float(
        test_fraud_rate
    ),
    "prevalence_drift": float(
        test_fraud_rate
        / val_fraud_rate
    ),
}


with open(
    FINAL_ANALYSIS_PATH,
    "w",
    encoding="utf-8",
) as file:

    json.dump(
        final_analysis,
        file,
        indent=2,
    )


print(
    "Analyse finale sauvegardée :",
    FINAL_ANALYSIS_PATH.exists(),
)

Analyse finale sauvegardée : True


In [38]:
# Valider les analyses finales

analysis_valid = (
    len(type_results_df)
        == 2
    and len(cold_start_df)
        == 4
    and int(
        cold_start_df[
            "transactions"
        ].sum()
    )
        == TEST_ROWS
    and int(
        cold_start_df[
            "frauds"
        ].sum()
    )
        == 3_830
    and FINAL_ANALYSIS_PATH.exists()
)


print(
    "Analyse finale valide :",
    bool(
        analysis_valid
    ),
)


if analysis_valid:

    print(
        "SECTION 16 : "
        "analyses finales validées."
    )

Analyse finale valide : True
SECTION 16 : analyses finales validées.


## 17. Final Summary, Artifacts and Notebook Validation

Cette dernière section synthétise les résultats du baseline GNN, vérifie les artefacts produits et clôture le notebook.

L'objectif de `06_gnn_baseline.ipynb` était d'évaluer si un message passing graph simple, inductif et strictement causal pouvait apporter une valeur prédictive supplémentaire par rapport au baseline Machine Learning classique.

L'expérience a couvert l'ensemble du protocole : construction causale du contexte, entraînement temporel, sélection de profondeur sur Validation, sélection du seuil sur Validation et évaluation finale unique sur Test.

In [39]:
# Recharger les résultats finaux

import json
from pathlib import Path

ROOT = Path("..")

GNN_ARTIFACT_DIR = (
    ROOT
    / "artifacts"
    / "gnn"
)

SELECTION_PATH = (
    GNN_ARTIFACT_DIR
    / "graphsage_core_edge_model_selection.json"
)

TEST_METRICS_PATH = (
    GNN_ARTIFACT_DIR
    / "graphsage_core_edge_1layer_test_metrics.json"
)

FINAL_ANALYSIS_PATH = (
    GNN_ARTIFACT_DIR
    / "graphsage_core_edge_final_analysis.json"
)


with open(
    SELECTION_PATH,
    "r",
    encoding="utf-8",
) as file:
    final_selection = json.load(file)


with open(
    TEST_METRICS_PATH,
    "r",
    encoding="utf-8",
) as file:
    final_test = json.load(file)


with open(
    FINAL_ANALYSIS_PATH,
    "r",
    encoding="utf-8",
) as file:
    final_analysis = json.load(file)


print(
    "Architecture sélectionnée :",
    f"{final_selection['selected_num_layers']} couche"
)

print(
    "Hops :",
    final_selection[
        "selected_num_hops"
    ],
)

print(
    f"Validation PR-AUC : "
    f"{final_selection['validation_pr_auc']:.6f}"
)

print(
    f"Seuil Validation : "
    f"{final_selection['threshold']:.10f}"
)

print()

print(
    f"Test PR-AUC : "
    f"{final_test['pr_auc']:.6f}"
)

print(
    f"Test ROC-AUC : "
    f"{final_test['roc_auc']:.6f}"
)

print(
    f"Test Precision : "
    f"{final_test['precision']:.6f}"
)

print(
    f"Test Recall : "
    f"{final_test['recall']:.6f}"
)

print(
    f"Test F1 : "
    f"{final_test['f1']:.6f}"
)

print(
    "TP / FP / FN :",
    final_test["tp"],
    "/",
    final_test["fp"],
    "/",
    final_test["fn"],
)

Architecture sélectionnée : 1 couche
Hops : 1
Validation PR-AUC : 0.240136
Seuil Validation : 0.9998995066

Test PR-AUC : 0.352779
Test ROC-AUC : 0.959934
Test Precision : 0.927941
Test Recall : 0.164752
Test F1 : 0.279823
TP / FP / FN : 631 / 49 / 3199


In [40]:
# Vérifier les artefacts GNN

required_artifacts = {
    "train_model": (
        GNN_ARTIFACT_DIR
        / "graphsage_core_edge_1layer_train_final.pt"
    ),
    "train_history": (
        GNN_ARTIFACT_DIR
        / "graphsage_core_edge_1layer_train_history.json"
    ),
    "train_config": (
        GNN_ARTIFACT_DIR
        / "graphsage_core_edge_1layer_config.json"
    ),
    "validation_predictions": (
        GNN_ARTIFACT_DIR
        / "graphsage_core_edge_1layer_validation_predictions.npz"
    ),
    "model_selection": (
        GNN_ARTIFACT_DIR
        / "graphsage_core_edge_model_selection.json"
    ),
    "depth2_model": (
        GNN_ARTIFACT_DIR
        / "graphsage_core_edge_2layer_train_final.pt"
    ),
    "depth2_validation": (
        GNN_ARTIFACT_DIR
        / "graphsage_core_edge_2layer_validation_predictions.npz"
    ),
    "test_predictions": (
        GNN_ARTIFACT_DIR
        / "graphsage_core_edge_1layer_test_predictions.npz"
    ),
    "test_metrics": (
        GNN_ARTIFACT_DIR
        / "graphsage_core_edge_1layer_test_metrics.json"
    ),
    "final_analysis": (
        GNN_ARTIFACT_DIR
        / "graphsage_core_edge_final_analysis.json"
    ),
}


for name, path in (
    required_artifacts.items()
):
    print(
        f"{name:24s} :",
        path.exists(),
    )


all_artifacts_exist = all(
    path.exists()
    for path
    in required_artifacts.values()
)


print()
print(
    "Tous les artefacts essentiels :",
    all_artifacts_exist,
)

train_model              : True
train_history            : True
train_config             : True
validation_predictions   : True
model_selection          : True
depth2_model             : True
depth2_validation        : True
test_predictions         : True
test_metrics             : True
final_analysis           : True

Tous les artefacts essentiels : True


In [41]:
# Vérifier les prédictions sauvegardées

import numpy as np


VAL_ROWS = 792_406
VAL_FRAUDS = 638

TEST_ROWS = 798_827
TEST_FRAUDS = 3_830


val_file = np.load(
    required_artifacts[
        "validation_predictions"
    ]
)

test_file = np.load(
    required_artifacts[
        "test_predictions"
    ]
)


validation_predictions_valid = (
    len(
        val_file["probabilities"]
    )
    == VAL_ROWS
    and len(
        val_file["labels"]
    )
    == VAL_ROWS
    and int(
        val_file["labels"].sum()
    )
    == VAL_FRAUDS
    and np.isfinite(
        val_file["probabilities"]
    ).all()
)


test_predictions_valid = (
    len(
        test_file["probabilities"]
    )
    == TEST_ROWS
    and len(
        test_file["labels"]
    )
    == TEST_ROWS
    and int(
        test_file["labels"].sum()
    )
    == TEST_FRAUDS
    and np.isfinite(
        test_file["probabilities"]
    ).all()
)


print(
    "Validation predictions :",
    validation_predictions_valid,
)

print(
    "Test predictions :",
    test_predictions_valid,
)

print(
    "Cold-start sauvegardé :",
    "cold_start"
    in test_file.files,
)

print(
    "Types sauvegardés :",
    "types"
    in test_file.files,
)

Validation predictions : True
Test predictions : True
Cold-start sauvegardé : True
Types sauvegardés : True


In [42]:
# Vérifier le protocole expérimental

protocol_checks = {
    "edge_classification": True,
    "temporal_split": True,
    "causal_history": True,
    "no_same_step_context": True,
    "no_future_edges": True,
    "no_node_identity_embedding": True,
    "train_ordered": True,
    "no_shuffle": True,
    "validation_for_selection": True,
    "test_not_used_for_selection": True,
    "weights_frozen_validation": True,
    "weights_frozen_test": True,
    "threshold_selected_on_validation": True,
}


for name, value in (
    protocol_checks.items()
):
    print(
        f"{name:30s} :",
        value,
    )


protocol_valid = all(
    protocol_checks.values()
)


print()
print(
    "Protocole expérimental valide :",
    protocol_valid,
)

edge_classification            : True
temporal_split                 : True
causal_history                 : True
no_same_step_context           : True
no_future_edges                : True
no_node_identity_embedding     : True
train_ordered                  : True
no_shuffle                     : True
validation_for_selection       : True
test_not_used_for_selection    : True
weights_frozen_validation      : True
weights_frozen_test            : True
threshold_selected_on_validation : True

Protocole expérimental valide : True


In [43]:
# Sauvegarder le résumé final

FINAL_SUMMARY_PATH = (
    GNN_ARTIFACT_DIR
    / "gnn_baseline_summary.json"
)


notebook_summary = {
    "task": "edge classification",

    "model": {
        "architecture": "GraphSAGE",
        "aggregation": "sum",
        "selected_layers": int(
            final_selection[
                "selected_num_layers"
            ]
        ),
        "selected_hops": int(
            final_selection[
                "selected_num_hops"
            ]
        ),
        "feature_set": (
            "Graph Core Edge"
        ),
        "input_dimension": 13,
        "node_identity_embedding": False,
    },

    "validation": {
        "transactions": 792_406,
        "frauds": 638,
        "pr_auc": float(
            final_selection[
                "validation_pr_auc"
            ]
        ),
        "roc_auc": float(
            final_selection[
                "validation_roc_auc"
            ]
        ),
        "threshold": float(
            final_selection[
                "threshold"
            ]
        ),
        "precision": float(
            final_selection[
                "precision"
            ]
        ),
        "recall": float(
            final_selection[
                "recall"
            ]
        ),
        "f1": float(
            final_selection[
                "f1"
            ]
        ),
    },

    "test": final_test,

    "lightgbm_core_reference": {
        "validation_pr_auc": 0.929704,
        "test_pr_auc": 0.979749,
        "test_roc_auc": 0.999890,
        "precision": 0.982230,
        "recall": 0.793730,
        "f1": 0.877980,
        "tp": 3040,
        "fp": 55,
        "fn": 790,
    },

    "main_conclusion": (
        "The simple causal GraphSAGE baseline "
        "captures graph signal but strongly "
        "underperforms LightGBM Core on PaySim."
    ),

    "protocol_valid": (
        protocol_valid
    ),
}


with open(
    FINAL_SUMMARY_PATH,
    "w",
    encoding="utf-8",
) as file:

    json.dump(
        notebook_summary,
        file,
        indent=2,
    )


print(
    "Résumé sauvegardé :",
    FINAL_SUMMARY_PATH.exists(),
)

print(
    "Fichier :",
    FINAL_SUMMARY_PATH.resolve(),
)

Résumé sauvegardé : True
Fichier : C:\Users\bouza\OneDrive\Bureau\MobileMoney-Fraud-GNN\artifacts\gnn\gnn_baseline_summary.json


In [44]:
# Valider le notebook 06

notebook_06_valid = (
    all_artifacts_exist
    and validation_predictions_valid
    and test_predictions_valid
    and protocol_valid
    and FINAL_SUMMARY_PATH.exists()
    and final_selection[
        "selected_num_layers"
    ] == 1
    and np.isclose(
        final_test[
            "pr_auc"
        ],
        0.352779,
        atol=1e-6,
    )
    and final_test[
        "tp"
    ] == 631
    and final_test[
        "fp"
    ] == 49
    and final_test[
        "fn"
    ] == 3199
)


print(
    "Artefacts :",
    all_artifacts_exist,
)

print(
    "Prédictions Validation :",
    validation_predictions_valid,
)

print(
    "Prédictions Test :",
    test_predictions_valid,
)

print(
    "Protocole causal :",
    protocol_valid,
)

print(
    "Résumé final :",
    FINAL_SUMMARY_PATH.exists(),
)

print()

print(
    "06_gnn_baseline.ipynb validé :",
    bool(
        notebook_06_valid
    ),
)


if notebook_06_valid:

    print(
        "Notebook terminé."
    )

Artefacts : True
Prédictions Validation : True
Prédictions Test : True
Protocole causal : True
Résumé final : True

06_gnn_baseline.ipynb validé : True
Notebook terminé.


## Conclusion

Ce notebook a permis de construire et d'évaluer un premier baseline GNN strictement causal pour la détection de fraude Mobile Money sur PaySim.

Le protocole repose sur une tâche d'edge classification dans laquelle chaque transaction du step `t` est prédite exclusivement à partir du graphe historique `G_<t`. Aucun mélange temporel, aucune arête future, aucune information du step courant et aucun embedding d'identité des comptes ne sont utilisés.

La comparaison de profondeur montre qu'une seule couche GraphSAGE est nettement préférable. La variante 1-layer obtient une PR-AUC Validation de 0,240136, contre seulement 0,001883 pour la variante 2-layer. Ce résultat est cohérent avec la faible profondeur structurelle du graphe PaySim et le très faible chevauchement entre comptes sources et destinations.

Sur Test, le GraphSAGE sélectionné obtient une PR-AUC de 0,352779 et une ROC-AUC de 0,959934. Au seuil de 0,9998995 sélectionné exclusivement sur Validation, il atteint une Precision de 0,927941, un Recall de 0,164752 et un F1-score de 0,279823. Il détecte 631 des 3 830 fraudes, avec seulement 49 faux positifs mais 3 199 faux négatifs.

L'analyse par type montre une meilleure PR-AUC sur `TRANSFER` (0,427863) que sur `CASH_OUT` (0,312927), mais le Recall reste faible pour les deux catégories.

Le setting est fortement inductif. Plus de 99 % des transactions Test impliquent au moins un nouvel endpoint. Le meilleur comportement du GraphSAGE est observé sur les transactions `New → New`, qui représentent environ 43,3 % du Test et atteignent une PR-AUC de 0,709571. Le Recall reste cependant limité à environ 17,3 %.

Le baseline GraphSAGE reste très inférieur au LightGBM Core. Sur Test, LightGBM obtient une PR-AUC de 0,979749 et un F1-score de 0,877980, contre respectivement 0,352779 et 0,279823 pour GraphSAGE. LightGBM détecte 3 040 fraudes contre 631 pour GraphSAGE, soit 2 409 fraudes supplémentaires, avec seulement six faux positifs supplémentaires.

Ces résultats montrent que l'ajout d'un message passing structurel simple ne suffit pas à améliorer le baseline tabulaire sur PaySim. La structure du graphe est particulièrement peu favorable à un GraphSAGE statique : très forte proportion de nouveaux comptes, composantes essentiellement arborescentes ou en étoile, faible profondeur relationnelle et très faible récurrence des relations source-destination.

Le résultat ne remet donc pas en cause l'intérêt du Graph Learning pour la fraude. Il montre plutôt les limites d'un GNN statique simple dans ce contexte et motive l'étude de modèles capables de représenter explicitement l'évolution temporelle des interactions et de maintenir un état dynamique des comptes.

Le notebook `06_gnn_baseline.ipynb` fournit ainsi une baseline graph causale, inductive, reproductible et directement comparable au baseline Machine Learning classique.